In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODULE 1 v6 — Dialogue Transformer Encoder                              ║
# ║    SuperEmotion Dataset:                                                 ║
# ║      Already single-label — no conversion needed                         ║
# ║      7 classes map cleanly to our 7-class system                         ║
# ║      439k train samples (2.3× more than GoEmotions)                      ║
# ║      Includes MELD as a source → better domain transfer                  ║
# ║      Psychologically grounded Shaver taxonomy                            ║
# ║                                                                          ║
# ║  CLASS MAPPING (SuperEmotion → our 7 classes):                           ║
# ║      neutral  → neutral   (exact)                                       ║
# ║      joy      → joyful    (exact)                                       ║
# ║      sadness  → sad       (exact)                                       ║
# ║      anger    → mad       (exact)                                       ║
# ║      fear     → scared    (exact)                                       ║
# ║      love     → peaceful  (calm/positive valence)                       ║
# ║      surprise → powerful  (high arousal/dramatic)                       ║
# ║                                                                                                                   
# ║      ISEAR = 72% of SuperEmotion, 0% neutral                           ║
# ║      Cap each source at MAX_SAMPLES_PER_SOURCE=50k                     ║
# ║      Result: balanced multi-source distribution                         ║
# ║                                                                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import os, json, pickle, time, re, glob, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import (Dataset, DataLoader,
                               WeightedRandomSampler)
from sklearn.metrics import (f1_score, accuracy_score,
                              classification_report,
                              confusion_matrix)
from sklearn.model_selection import train_test_split
from datasets import load_dataset as hf_load_dataset   # HuggingFace datasets
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PLOT_DIR = "/kaggle/working/plots"
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
print(f"  Plots  → {PLOT_DIR}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

class CFG1:
    # ── Paths ───────────────────────────────────────────────────────────────
    # All data comes from SuperEmotion — no local CSV paths needed.
    # SuperEmotion includes MELD, GoEmotions, ISEAR, TwitterEmotion,
    # CrowdFlower, SemEval — already split into train/validation/test.
    SUPEREMO_HF = "cirimus/super-emotion"
    # Cap per source to prevent ISEAR (72% of raw data) from dominating.
    # ISEAR has 0 neutral — without cap, neutral collapses to ~6%.
    MAX_SAMPLES_PER_SOURCE = 50_000
    CKPT_DIR   = "/kaggle/working/checkpoints"

    # ── Vocabulary ──────────────────────────────────────────────────────────
    VOCAB_SIZE = 20000
    MAX_LEN    = 50

    # ── Transformer Architecture ────────────────────────────────────────────
    EMBED_DIM   = 128
    D_MODEL     = 256
    NUM_HEADS   = 4
    NUM_LAYERS  = 4
    D_FF        = 512
    SPEAKER_DIM = 64
    TURN_DIM    = 32
    MAX_TURNS   = 60
    NUM_PREV_UTT= 5

    # ── Shared ──────────────────────────────────────────────────────────────
    NUM_EMOTIONS    = 7
    NUM_SCENE_TYPES = 6
    # No per-utterance speaker info in SuperEmotion — speaker embedding
    # is set to 1 (just UNKNOWN) and kept for architectural compatibility.
    MAX_SPEAKERS    = 2
    DROPOUT         = 0.2

    # ── Phase 1: SuperEmotion train split pre-training ──────────────────────
    GOEMO_EPOCHS = 12
    GOEMO_LR     = 2e-4
    GOEMO_BATCH  = 128

    # ── Phase 2: SuperEmotion val/test split fine-tuning ────────────────────
    # Using SuperEmotion's built-in val split for fine-tuning and
    # test split for final evaluation — no separate dataset needed.
    MELD_EPOCHS  = 15   # reduced: val split is cleaner, less overfitting risk
    MELD_LR      = 1e-4
    MELD_BATCH   = 64
    EARLY_STOP   = 5

Path(CFG1.CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("  Config ready ✓")
print(f"  Transformer: {CFG1.NUM_LAYERS}L × {CFG1.NUM_HEADS}H × {CFG1.D_MODEL}d")
print(f"  MAX_SPEAKERS: {CFG1.MAX_SPEAKERS}  (speaker embedding kept for architecture compatibility)")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Emotion Labels + Mappings                                       │
# └──────────────────────────────────────────────────────────────────────────┘

EMOTION_CLASSES  = ['neutral', 'joyful', 'sad', 'mad',
                    'scared', 'peaceful', 'powerful']
SCENE_TYPE_NAMES = ['dialogue', 'action', 'investigation',
                    'suspense', 'dramatic_reveal', 'transition']

# ── SuperEmotion → our 7-class mapping ──────────────────────────────────────
# SuperEmotion uses Shaver's taxonomy: neutral, joy, sadness, anger,
# fear, love, surprise.  Maps cleanly onto our 7 classes.
SUPEREMO_MERGE = {
    'neutral':  'neutral',
    'joy':      'joyful',
    'sadness':  'sad',
    'anger':    'mad',
    'fear':     'scared',
    'love':     'peaceful',   # calm/positive valence — closest match
    'surprise': 'powerful',   # high arousal / dramatic — closest match
}

ACTION_KEYWORDS = [
    'explodes', 'crashes', 'jumps', 'attacks', 'fights', 'shoots',
    'runs', 'falls', 'screams', 'strikes', 'kicks', 'punches',
    'slams', 'grabs', 'throws', 'dodges', 'rushes', 'flees',
]


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Vocabulary                                                      │
# └──────────────────────────────────────────────────────────────────────────┘

def tokenize(text: str) -> list:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\'\-]", ' ', text)
    return text.split()

def build_vocab(texts: list, max_size: int) -> dict:
    counter = Counter()
    for t in texts:
        counter.update(tokenize(str(t)))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

def encode(text: str, vocab: dict, max_len: int) -> list:
    tokens = tokenize(str(text))[:max_len]
    ids    = [vocab.get(t, 1) for t in tokens]
    ids   += [0] * (max_len - len(ids))
    return ids


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — SuperEmotion Loader                                             │
# │                                                                          │
# │  SuperEmotion (cirimus/super-emotion) — 439k train samples              │
# │  Already single-label with Shaver's 7-class taxonomy.                   │
# │  No multi-label conversion needed — clean training signal.              │
# │                                                                          │
# │  The returned DataFrame has the same columns as the old GoEmotions one: │
# │    'text', 'emotion_label' (int 0-6), 'emotion_name' (str)             │
# │  so all downstream code (Dataset, DataLoader, plots) works unchanged.  │
# └──────────────────────────────────────────────────────────────────────────┘

def _map_split(hf_split, split_name: str,
               text_col: str, label_col: str,
               source_col, max_per_source: int) -> pd.DataFrame:
    """
    Convert one HF split to our mapped DataFrame.

    SuperEmotion actual columns:
      'labels'      — list of int class indices  e.g. [3]
      'labels_str'  — list of string names       e.g. ['anger']
      'labels_source' — list of source names
      'source'      — source dataset name (str)

    We use 'labels_str' when available (cleaner), falling back to
    'labels' with ClassLabel int→name resolution.
    For multi-label rows we take the first (dominant) label only.
    """
    df = hf_split.to_pandas()
    df[text_col] = df[text_col].astype(str).str.strip()
    df = df[df[text_col].notna() & (df[text_col] != '') &
            (df[text_col] != 'nan')].reset_index(drop=True)

    # Prefer labels_str — already human-readable strings
    str_col = 'labels_str' if 'labels_str' in df.columns else None

    # Build int→name map from HF ClassLabel features if needed
    _int_map = {}
    try:
        feat = hf_split.features[label_col]
        if hasattr(feat, 'feature') and hasattr(feat.feature, 'names'):
            _int_map = {i: n.lower() for i, n in enumerate(feat.feature.names)}
        elif hasattr(feat, 'names'):
            _int_map = {i: n.lower() for i, n in enumerate(feat.names)}
    except Exception:
        pass
    if _int_map:
        print(f"  [{split_name}] ClassLabel map: {_int_map}")

    # Cap per source — train only
    if source_col and max_per_source > 0 and split_name == 'train':
        print(f"\n  [{split_name}] Source counts before cap ({max_per_source:,}/source):")
        for src, cnt in df[source_col].value_counts().items():
            print(f"    {src:<22} {cnt:>7,}")
        capped = []
        for src, grp in df.groupby(source_col):
            if len(grp) > max_per_source:
                grp = grp.sample(max_per_source, random_state=SEED)
            capped.append(grp)
        df = pd.concat(capped, ignore_index=True)
        df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        print(f"  [{split_name}] After cap: {len(df):,} samples")

    rows, skipped = [], 0
    for _, row in df.iterrows():
        text = str(row[text_col]).strip()
        if not text or text == 'nan':
            skipped += 1
            continue

        # ── Extract label string ─────────────────────────────────────────
        if str_col:
            # labels_str is a list like ['anger'] or ['joy', 'love']
            raw = row[str_col]
            if isinstance(raw, list) and len(raw) > 0:
                label_str = str(raw[0]).lower().strip()
            else:
                # Sometimes stored as string repr "['anger']"
                s = str(raw).strip("[]'\" ")
                label_str = s.split(',')[0].strip().strip("'\" ").lower()
        else:
            # Use labels (int list) + ClassLabel map
            raw = row[label_col]
            if isinstance(raw, list) and len(raw) > 0:
                idx = int(raw[0])
            else:
                try:
                    idx = int(str(raw).strip("[]").split(',')[0].strip())
                except Exception:
                    skipped += 1
                    continue
            label_str = _int_map.get(idx, str(idx))

        mapped = SUPEREMO_MERGE.get(label_str)
        if mapped is None:
            skipped += 1
            continue

        rows.append({
            'text':          text,
            'emotion_label': EMOTION_CLASSES.index(mapped),
            'emotion_name':  mapped,
        })

    if not rows:
        raise RuntimeError(
            f"[{split_name}] 0 rows mapped — all {skipped} skipped.\n"
            f"  label_col='{label_col}', str_col='{str_col}'\n"
            f"  Sample raw labels: {df[str_col if str_col else label_col].head(5).tolist()}\n"
            f"  SUPEREMO_MERGE keys: {list(SUPEREMO_MERGE.keys())}")

    result = pd.DataFrame(rows)

    # Distribution summary
    dist  = Counter(result['emotion_name'].tolist())
    total = len(result)
    max_c = max(dist.values()) if dist else 1
    print(f"\n  [{split_name}] {total:,} samples  (skipped {skipped:,})")
    for emo in EMOTION_CLASSES:
        c   = dist.get(emo, 0)
        pct = c / total * 100
        bar = '█' * int(c / max_c * 20)
        print(f"    {emo:<10} {c:>6,}  {pct:5.1f}%  {bar}")

    neutral_pct = dist.get('neutral', 0) / total * 100
    if neutral_pct < 5.0:
        print(f"  ⚠️  neutral={neutral_pct:.1f}% low — reduce MAX_SAMPLES_PER_SOURCE")
    else:
        print(f"  ✓ neutral={neutral_pct:.1f}%")

    return result


def load_superemotion(hf_name: str,
                      max_per_source: int = CFG1.MAX_SAMPLES_PER_SOURCE
                      ) -> tuple:
    """
    Load all 3 SuperEmotion splits from HuggingFace.

    Returns: (train_df, val_df, test_df)
      Each DataFrame has columns: text, emotion_label (int), emotion_name (str)

    Phase 1 pre-trains on train_df.
    Phase 2 fine-tunes on val_df  (smaller, focused — acts as adaptation set).
    Final evaluation on test_df.

    Why use val for Phase 2?
      SuperEmotion val = 54,835 samples — large enough for fine-tuning.
      Keeps train/val/test roles clean with no data leakage.
      Source cap applied to train only; val and test used as-is.
    """
    print(f"\n  Loading SuperEmotion: {hf_name}")
    print(f"  (first run downloads ~200 MB → cached to ~/.cache/huggingface)")

    hf_ds      = hf_load_dataset(hf_name)
    text_col   = 'text'
    label_col  = 'label'
    # Detect source column name (may vary by dataset version)
    sample_cols = hf_ds['train'].column_names
    print(f"  Columns: {sample_cols}")

    # Auto-detect label column — prefer labels_str (human-readable), then labels
    for candidate in ['labels_str', 'labels', 'label', 'emotion', 'emotions',
                       'sentiment', 'category']:
        if candidate in sample_cols:
            label_col = candidate
            break
    else:
        label_col = next(
            c for c in sample_cols if c not in ('text', 'source', 'dataset',
                                                 'origin', 'id', 'labels_source'))

    # Auto-detect text column
    for candidate in ['text', 'sentence', 'utterance', 'content']:
        if candidate in sample_cols:
            text_col = candidate
            break

    source_col = next((c for c in ['source', 'dataset', 'origin']
                       if c in sample_cols), None)
    print(f"  text_col='{text_col}'  label_col='{label_col}'  source_col='{source_col}'")

    train_df = _map_split(hf_ds['train'],      'train',
                          text_col, label_col, source_col, max_per_source)
    val_df   = _map_split(hf_ds['validation'], 'validation',
                          text_col, label_col, source_col, max_per_source)
    test_df  = _map_split(hf_ds['test'],       'test',
                          text_col, label_col, source_col, max_per_source)

    return train_df, val_df, test_df


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — Dataset class                                                   │
# │  SuperEmotionDataset handles ALL splits (train / val / test).            │
# │  No MELD-specific code needed — SuperEmotion already includes MELD.     │
# └──────────────────────────────────────────────────────────────────────────┘

class SuperEmotionDataset(Dataset):
    """
    Wraps the SuperEmotion DataFrame (or any pre-mapped DataFrame with
    columns: text, emotion_label) into a PyTorch Dataset.
    Kept interface-identical to the old GoEmotionsDataset so all
    DataLoader / training code below works without any changes.
    """
    def __init__(self, df, vocab, max_len=CFG1.MAX_LEN):
        self.samples = []
        zero_hist = torch.zeros(CFG1.NUM_PREV_UTT, max_len, dtype=torch.long)
        for _, row in df.iterrows():
            ids = encode(str(row['text']), vocab, max_len)
            self.samples.append({
                'token_ids':   torch.tensor(ids, dtype=torch.long),
                'speaker_id':  torch.tensor(0, dtype=torch.long),
                'turn_pos':    torch.tensor(0, dtype=torch.long),
                'utt_history': zero_hist.clone(),
                'emotion':     torch.tensor(int(row['emotion_label']),
                                            dtype=torch.long),
            })
        print(f"  SuperEmotion dataset: {len(self.samples):,}")

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_utt(batch):
    return {
        'token_ids':   torch.stack([b['token_ids']   for b in batch]),
        'speaker_id':  torch.stack([b['speaker_id']  for b in batch]),
        'turn_pos':    torch.stack([b['turn_pos']    for b in batch]),
        'utt_history': torch.stack([b['utt_history'] for b in batch]),
        'emotion':     torch.stack([b['emotion']     for b in batch]),
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Transformer Components                                          │
# └──────────────────────────────────────────────────────────────────────────┘

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads

        self.W_q  = nn.Linear(d_model, d_model, bias=False)
        self.W_k  = nn.Linear(d_model, d_model, bias=False)
        self.W_v  = nn.Linear(d_model, d_model, bias=False)
        self.W_o  = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

        for w in [self.W_q, self.W_k, self.W_v, self.W_o]:
            nn.init.xavier_uniform_(w.weight)

    def _split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self._split_heads(self.W_q(x))
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_w = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
        attn_w = self.drop(attn_w)

        context = torch.matmul(attn_w, V)
        context = context.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(context), attn_w


class FeedForward(nn.Module):
    def __init__(self, d_model=256, d_ff=512, dropout=0.1):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.fc2  = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        return self.fc2(self.drop(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    """Pre-LN Transformer block for stable training."""
    def __init__(self, d_model=256, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_w = self.attn(self.ln1(x), mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x, attn_w


class DialogueTurnEmbedding(nn.Module):
    def __init__(self, max_turns=60, embed_dim=32):
        super().__init__()
        self.embed = nn.Embedding(max_turns, embed_dim)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, turn_positions):
        return self.embed(turn_positions)


class EmotionQueryToken(nn.Module):
    def __init__(self, d_model=256, num_emotions=7):
        super().__init__()
        self.prototypes = nn.Parameter(
            torch.randn(num_emotions, d_model) * 0.02)
        self.gate = nn.Linear(d_model, num_emotions)
        nn.init.xavier_uniform_(self.gate.weight)

    def get_query(self, x):
        mean_x  = x.mean(dim=1)
        weights = F.softmax(self.gate(mean_x), dim=-1)
        query   = torch.matmul(weights, self.prototypes)
        return query.unsqueeze(1)


class DialogueContextGRU(nn.Module):
    """
    Encodes conversation history.
    FIX 6: explicit .long() cast on history tensor at entry point.
    """
    def __init__(self, vocab_size, embed_dim=128, d_model=256,
                 context_dim=128, num_prev=5, max_len=50, dropout=0.2):
        super().__init__()
        self.num_prev = num_prev
        self.max_len  = max_len

        self.tok_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.utt_proj  = nn.Sequential(
            nn.Linear(embed_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
        )
        self.gru = nn.GRU(input_size=d_model, hidden_size=context_dim,
                          num_layers=1, batch_first=True)
        self.proj = nn.Sequential(
            nn.Linear(context_dim, context_dim),
            nn.LayerNorm(context_dim),
            nn.Tanh(),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, utt_history):
        """
        utt_history: (B, num_prev, max_len)
        FIX 6: .long() ensures no FloatTensor crash in Embedding lookup.
        """
        B, P, T = utt_history.shape
        flat    = utt_history.reshape(B * P, T).long()  
        # Clamp to valid vocab range (safety net)
        flat    = flat.clamp(0, self.tok_embed.num_embeddings - 1)

        emb     = self.tok_embed(flat)
        mask    = (flat != 0).float().unsqueeze(-1)
        pooled  = (emb * mask).sum(1) / (mask.sum(1) + 1e-6)
        pooled  = self.utt_proj(pooled).view(B, P, -1)

        _, h_n  = self.gru(pooled)
        context = self.proj(h_n.squeeze(0))
        return self.drop(context)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Dialogue Transformer Encoder                                    │
# └──────────────────────────────────────────────────────────────────────────┘

class DialogueTransformerEncoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 embed_dim    = CFG1.EMBED_DIM,
                 d_model      = CFG1.D_MODEL,
                 num_heads    = CFG1.NUM_HEADS,
                 num_layers   = CFG1.NUM_LAYERS,
                 d_ff         = CFG1.D_FF,
                 max_len      = CFG1.MAX_LEN,
                 max_turns    = CFG1.MAX_TURNS,
                 speaker_dim  = CFG1.SPEAKER_DIM,
                 turn_dim     = CFG1.TURN_DIM,
                 num_speakers = CFG1.MAX_SPEAKERS,
                 num_emotions = CFG1.NUM_EMOTIONS,
                 num_prev     = CFG1.NUM_PREV_UTT,
                 dropout      = CFG1.DROPOUT):
        super().__init__()
        self.d_model = d_model

        self.tok_embed  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed  = nn.Embedding(max_len + 1, embed_dim)
        self.spk_embed  = nn.Embedding(num_speakers, speaker_dim)
        self.turn_embed = DialogueTurnEmbedding(max_turns, turn_dim)

        nn.init.normal_(self.tok_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        nn.init.normal_(self.spk_embed.weight, std=0.02)

        # 128+128+64+32 = 352 → 256
        input_concat_dim = embed_dim + embed_dim + speaker_dim + turn_dim
        self.input_proj = nn.Sequential(
            nn.Linear(input_concat_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )

        self.emo_query  = EmotionQueryToken(d_model, num_emotions)

        self.layers   = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.ctx_gru  = DialogueContextGRU(
            vocab_size  = vocab_size,
            embed_dim   = embed_dim,
            d_model     = d_model,
            context_dim = 128,
            num_prev    = num_prev,
            max_len     = max_len,
            dropout     = dropout,
        )

        # 256 + 128 → 256
        self.fusion = nn.Sequential(
            nn.Linear(d_model + 128, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.emo_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_emotions),
        )

    def _build_pad_mask(self, token_ids, extra_token=True):
        B, T   = token_ids.shape
        pad    = (token_ids != 0).float()
        if extra_token:
            ones = torch.ones(B, 1, device=token_ids.device)
            pad  = torch.cat([ones, pad], dim=1)
        return pad.unsqueeze(1).unsqueeze(2)

    def encode(self, token_ids, speaker_ids, turn_positions):
        B, T  = token_ids.shape
        pos   = torch.arange(T, device=token_ids.device).unsqueeze(0).expand(B, -1)

        tok_e = self.tok_embed(token_ids)
        pos_e = self.pos_embed(pos)
        spk_e = self.spk_embed(speaker_ids).unsqueeze(1).expand(-1, T, -1)
        trn_e = self.turn_embed(turn_positions).unsqueeze(1).expand(-1, T, -1)

        x     = self.input_proj(torch.cat([tok_e, pos_e, spk_e, trn_e], dim=-1))
        query = self.emo_query.get_query(x)
        x     = torch.cat([query, x], dim=1)
        mask  = self._build_pad_mask(token_ids)

        all_attn = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            all_attn.append(attn_w)

        x = self.final_ln(x)
        return x[:, 0, :], all_attn

    def forward(self, token_ids, speaker_ids, turn_positions,
                utt_history, return_vector=False):
        cls_out, all_attn = self.encode(token_ids, speaker_ids, turn_positions)
        ctx_out           = self.ctx_gru(utt_history)
        fused             = self.fusion(torch.cat([cls_out, ctx_out], dim=-1))
        logits            = self.emo_head(fused)

        if return_vector:
            return logits, fused, all_attn
        return logits


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Stage Direction Encoder                                        │
# └──────────────────────────────────────────────────────────────────────────┘

class StageDirectionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128,
                 num_scene_types=6, dropout=0.3):
        super().__init__()
        self.tok_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=2,
                            batch_first=True, bidirectional=True,
                            dropout=dropout)
        self.norm            = nn.LayerNorm(hidden_dim * 2)
        self.drop            = nn.Dropout(dropout)
        self.scene_type_head = nn.Sequential(
            nn.Linear(hidden_dim*2, 64), nn.ReLU(),
            nn.Linear(64, num_scene_types))
        self.action_head     = nn.Sequential(
            nn.Linear(hidden_dim*2, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, 256), nn.LayerNorm(256), nn.ReLU())

    def forward(self, token_ids):
        x      = self.tok_embed(token_ids)
        out, _ = self.lstm(x)
        mask   = (token_ids != 0).unsqueeze(-1).float()
        pooled = self.drop(self.norm(
            (out * mask).sum(1) / (mask.sum(1) + 1e-6)))
        return {
            'stage_vector': self.proj(pooled),
            'scene_logits': self.scene_type_head(pooled),
            'action_score': self.action_head(pooled).squeeze(-1),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — Loss Functions                                                 │
# │                                                                          │
# └──────────────────────────────────────────────────────────────────────────┘

class Phase1Loss(nn.Module):
    """
    FIX 3: Plain CrossEntropy for Phase 1.
    No neutral penalty — GoEmotions has natural ~27% neutral.
    Penalising neutral during pre-training distorts representations.
    FIX 4: label_smoothing=0.05 (reduced from 0.1).
    """
    def __init__(self, weights=None, label_smoothing=0.05):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(
            weight          = weights,
            label_smoothing = label_smoothing,
        )

    def utterance_loss(self, logits, targets):
        return self.ce(logits, targets)


class NeutralPenaltyLoss(nn.Module):
    """
    Phase 2 only: CrossEntropy + neutral penalty.
    Prevents model from lazily predicting neutral for MELD's minority classes.
    """
    def __init__(self, weights=None, neutral_idx=0,
                 neutral_penalty=1.8, label_smoothing=0.1):
        super().__init__()
        self.neutral_idx     = neutral_idx
        self.neutral_penalty = neutral_penalty
        self.ce = nn.CrossEntropyLoss(
            weight          = weights,
            label_smoothing = label_smoothing,
            reduction       = 'none',
        )

    def forward(self, logits, targets):
        loss_per     = self.ce(logits, targets)
        preds        = logits.argmax(dim=-1)
        lazy_neutral = ((preds == self.neutral_idx) & (targets != self.neutral_idx))
        multiplier   = torch.where(
            lazy_neutral,
            torch.full_like(loss_per, self.neutral_penalty),
            torch.ones_like(loss_per))
        return (loss_per * multiplier).mean()


class Phase2Loss(nn.Module):
    """Phase 2 loss wrapper."""
    def __init__(self, emotion_weights=None,
                 neutral_penalty=1.8, label_smoothing=0.1):
        super().__init__()
        self.loss_fn = NeutralPenaltyLoss(
            weights         = emotion_weights,
            neutral_idx     = EMOTION_CLASSES.index('neutral'),
            neutral_penalty = neutral_penalty,
            label_smoothing = label_smoothing,
        )

    def utterance_loss(self, logits, targets):
        return self.loss_fn(logits, targets)


def compute_class_weights(labels, num_classes,
                          max_weight=15.0,
                          extra_boost=None) -> torch.Tensor:
    """Inverse-frequency weights, clamped to avoid explosion."""
    counts  = np.bincount(labels, minlength=num_classes).astype(float)
    counts  = np.where(counts == 0, 1, counts)
    weights = len(labels) / (num_classes * counts)
    if extra_boost:
        for cls_name, mult in extra_boost.items():
            if cls_name in EMOTION_CLASSES:
                weights[EMOTION_CLASSES.index(cls_name)] *= mult
    weights = np.clip(weights, 0.1, max_weight)
    print(f"\n  Class weights (clamped ≤{max_weight}):")
    for i, emo in enumerate(EMOTION_CLASSES):
        print(f"    {emo:<10}  count={int(counts[i]):>6,}  "
              f"weight={weights[i]:.3f}")
    return torch.tensor(weights, dtype=torch.float32)


def make_warmup_cosine_scheduler(optimizer, num_warmup_steps: int,
                                  num_total_steps: int):
    """
    Linear warmup for first `num_warmup_steps` steps,
    then cosine decay to 0 over the remaining steps.

    Why warmup matters for scratch transformers:
      At init, weight matrices are random → gradients are large and noisy.
      A high LR in epoch 1 pushes weights into bad basins they never escape.
      Warmup ramps LR from ~0 → target over the first few percent of steps,
      letting the model stabilise before full-speed training begins.

    Recommended warmup = 5% of total steps (this is the standard used in
    BERT, GPT-2, and most modern NLP transformers).

    Usage:
        steps_per_epoch = len(loader)
        total_steps     = num_epochs * steps_per_epoch
        warmup_steps    = int(0.05 * total_steps)
        scheduler = make_warmup_cosine_scheduler(
            optimizer, warmup_steps, total_steps)
        # call scheduler.step() AFTER every optimizer.step()
    """
    def lr_lambda(current_step: int):
        if current_step < num_warmup_steps:
            # Linear ramp: 0 → 1
            return float(current_step) / max(1, num_warmup_steps)
        # Cosine decay: 1 → 0
        progress = float(current_step - num_warmup_steps) / max(
            1, num_total_steps - num_warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def make_balanced_sampler(dataset) -> WeightedRandomSampler:
    """Oversample minority classes for balanced batches."""
    labels   = [s['emotion'].item() for s in dataset.samples]
    counts   = np.bincount(labels, minlength=CFG1.NUM_EMOTIONS).astype(float)
    counts   = np.where(counts == 0, 1, counts)
    class_w  = 1.0 / counts
    sample_w = torch.tensor([class_w[l] for l in labels], dtype=torch.float32)
    sampler  = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    print(f"\n  Balanced sampler: {len(sampler):,} samples/epoch")
    for i, emo in enumerate(EMOTION_CLASSES):
        print(f"    {emo:<10}  count={int(counts[i]):>5}  "
              f"rel_weight=×{class_w[i]/class_w[0]:.1f}")
    return sampler


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — Threshold Calibration + Prediction                            │
# └──────────────────────────────────────────────────────────────────────────┘

def calibrate_thresholds(model, val_loader,
                          num_classes=7) -> np.ndarray:
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['token_ids'].to(DEVICE)
            spk  = batch['speaker_id'].to(DEVICE)
            trn  = batch['turn_pos'].to(DEVICE)
            hist = batch['utt_history'].to(DEVICE)
            probs = torch.softmax(
                model(ids, spk, trn, hist), dim=-1)
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(batch['emotion'].numpy().tolist())

    probs_arr  = np.array(all_probs)
    labels_arr = np.array(all_labels)
    thresholds = np.full(num_classes, 0.5)

    for c in range(num_classes):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.05, 0.90, 0.05):
            preds = np.where(probs_arr[:, c] >= t,
                             c, probs_arr.argmax(axis=1))
            mask  = (labels_arr == c) | (preds == c)
            if mask.sum() == 0: continue
            f1    = f1_score(labels_arr[mask], preds[mask],
                             labels=[c], average='macro', zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t

    print(f"\n  Calibrated thresholds:")
    for i, emo in enumerate(EMOTION_CLASSES):
        d = '↓' if thresholds[i] < 0.5 else '↑'
        print(f"    {emo:<10} {thresholds[i]:.2f} {d}")
    return thresholds


def predict_with_thresholds(model, loader, thresholds):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids   = batch['token_ids'].to(DEVICE)
            spk   = batch['speaker_id'].to(DEVICE)
            trn   = batch['turn_pos'].to(DEVICE)
            hist  = batch['utt_history'].to(DEVICE)
            probs = torch.softmax(
                model(ids, spk, trn, hist), dim=-1).cpu().numpy()
            for row in probs:
                adjusted = row / (thresholds + 1e-8)
                all_preds.append(int(np.argmax(adjusted)))
            all_labels.extend(batch['emotion'].numpy().tolist())
    return all_labels, all_preds


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Train / Eval Functions                                         │
# └──────────────────────────────────────────────────────────────────────────┘

def train_one_epoch(model, loader, loss_fn, optimizer,
                    epoch, total, phase_name='',
                    batch_scheduler=None):
    """
    batch_scheduler: if provided, stepped after every optimizer.step().
    Used for warmup+cosine (which operates per-step, not per-epoch).
    """
    model.train()
    total_loss = 0.0
    all_t, all_p = [], []
    t0 = time.time()

    pbar = tqdm(loader,
                desc=f"{phase_name} Ep {epoch+1:02d}/{total} ▶ Train",
                leave=True, bar_format="{l_bar}{bar:25}{r_bar}")

    for step, batch in enumerate(pbar):
        ids  = batch['token_ids'].to(DEVICE)
        spk  = batch['speaker_id'].to(DEVICE)
        trn  = batch['turn_pos'].to(DEVICE)
        hist = batch['utt_history'].to(DEVICE)
        emo  = batch['emotion'].to(DEVICE)

        logits = model(ids, spk, trn, hist)
        loss   = loss_fn.utterance_loss(logits, emo)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if batch_scheduler is not None:
            batch_scheduler.step()

        total_loss += loss.item()
        all_t.extend(emo.cpu().numpy().tolist())
        all_p.extend(logits.argmax(-1).cpu().numpy().tolist())

        if (step + 1) % 20 == 0:
            f1 = f1_score(all_t, all_p, average='macro',
                          zero_division=0) * 100
            pbar.set_postfix(loss=f"{total_loss/(step+1):.3f}",
                             f1=f"{f1:.1f}%")

    elapsed  = time.time() - t0
    f1_final = f1_score(all_t, all_p, average='macro', zero_division=0)
    avg_loss = total_loss / len(loader)
    m, s     = divmod(int(elapsed), 60)
    print(f"    ↳ TRAIN  loss={avg_loss:.4f}  "
          f"macro_f1={f1_final*100:.2f}%  ({m}m{s:02d}s)")
    return avg_loss, f1_final


def eval_model(model, loader, loss_fn, split='val'):
    model.eval()
    total_loss = 0.0
    all_t, all_p = [], []

    with torch.no_grad():
        for batch in loader:
            ids  = batch['token_ids'].to(DEVICE)
            spk  = batch['speaker_id'].to(DEVICE)
            trn  = batch['turn_pos'].to(DEVICE)
            hist = batch['utt_history'].to(DEVICE)
            emo  = batch['emotion'].to(DEVICE)

            logits = model(ids, spk, trn, hist)
            loss   = loss_fn.utterance_loss(logits, emo)

            total_loss += loss.item()
            all_t.extend(emo.cpu().numpy().tolist())
            all_p.extend(logits.argmax(-1).cpu().numpy().tolist())

    f1   = f1_score(all_t, all_p, average='macro', zero_division=0)
    acc  = accuracy_score(all_t, all_p)
    loss = total_loss / len(loader)
    print(f"    ↳ {split.upper():12s}  loss={loss:.4f}  "
          f"macro_f1={f1*100:.2f}%  acc={acc*100:.2f}%")
    return loss, f1, all_t, all_p


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 14 — Plotting Utilities (unchanged from v4)                        │
# └──────────────────────────────────────────────────────────────────────────┘

DARK_BG  = '#0f0f1a'
PANEL_BG = '#1a1a2e'
ACCENT   = ['#2ecc71','#e74c3c','#f39c12',
            '#3498db','#9b59b6','#1abc9c','#e67e22']

def _style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors='#cccccc', labelsize=8)
    ax.set_title(title, color='white', fontsize=10,
                 fontweight='bold', pad=6)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#333355')

def plot_training_curves(goemo_hist, meld_hist):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 1 v5 — Training Curves',
                 color='white', fontsize=12, fontweight='bold')

    for hist, lbl, col in [
            (goemo_hist, 'GoEmotions Pre-training', 0),
            (meld_hist,  'MELD Fine-tuning', 1)]:
        ep_tr = [h['epoch'] for h in hist['train']]
        ep_va = [h['epoch'] for h in hist['val']]

        ax = axes[0][col]
        ax.plot(ep_tr, [h['loss'] for h in hist['train']],
                color='#3498db', lw=2, marker='o', ms=3, label='Train')
        ax.plot(ep_va, [h['loss'] for h in hist['val']],
                color='#e74c3c', lw=2, marker='s', ms=3,
                linestyle='--', label='Val')
        ax.legend(fontsize=7, facecolor=PANEL_BG, labelcolor='white')
        _style_ax(ax, f'{lbl} — Loss', 'Epoch', 'Loss')

        ax = axes[1][col]
        ax.plot(ep_tr, [h['f1']*100 for h in hist['train']],
                color='#2ecc71', lw=2, marker='o', ms=3, label='Train')
        ax.plot(ep_va, [h['f1']*100 for h in hist['val']],
                color='#f39c12', lw=2, marker='s', ms=3,
                linestyle='--', label='Val')
        ax.legend(fontsize=7, facecolor=PANEL_BG, labelcolor='white')
        ax.set_ylim(0, 100)
        _style_ax(ax, f'{lbl} — Macro F1', 'Epoch', 'Macro F1 (%)')

    plt.tight_layout()
    p = f"{PLOT_DIR}/01_training_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ 01_training_curves.png")
    return p

def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm     = confusion_matrix(y_true, y_pred, labels=list(range(7)))
    cm_pct = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-6)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(title, color='white', fontsize=11, fontweight='bold')

    for ax, data, fmt, t in zip(
            axes, [cm, cm_pct], ['d', '.2f'], ['Count', 'Row-normalised']):
        sns.heatmap(data, annot=True, fmt=fmt,
                    xticklabels=EMOTION_CLASSES, yticklabels=EMOTION_CLASSES,
                    cmap='Blues', ax=ax, linewidths=0.5, linecolor='#333355')
        ax.set_facecolor(PANEL_BG)
        ax.set_title(t, color='white', fontsize=9, fontweight='bold')
        ax.set_xlabel('Predicted', color='#aaaaaa')
        ax.set_ylabel('True',      color='#aaaaaa')
        ax.tick_params(colors='#cccccc')

    plt.tight_layout()
    p = f"{PLOT_DIR}/{filename}"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ {filename}")
    return p

def plot_per_class_f1(y_true, y_pred, title, filename):
    f1s   = f1_score(y_true, y_pred, average=None, zero_division=0,
                     labels=list(range(7)))
    macro = np.mean(f1s)

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK_BG)
    bars = ax.bar(EMOTION_CLASSES, f1s * 100, color=ACCENT, alpha=0.85,
                  edgecolor='white', linewidth=0.5)
    ax.axhline(macro*100, color='white', linestyle='--', linewidth=1.2,
               label=f'Macro={macro*100:.2f}%')
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{v*100:.1f}%', ha='center', va='bottom',
                color='white', fontsize=8, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9, facecolor=PANEL_BG, labelcolor='white')
    _style_ax(ax, title, 'Emotion', 'F1 (%)')
    plt.tight_layout()
    p = f"{PLOT_DIR}/{filename}"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ {filename}")
    return p

def plot_dataset_distributions(train_df, val_df):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('SuperEmotion Class Distributions (v6)',
                 color='white', fontsize=12, fontweight='bold')

    g_cnt = Counter(train_df['emotion_name'].tolist())
    g_val = [g_cnt.get(e, 0) for e in EMOTION_CLASSES]
    axes[0].bar(EMOTION_CLASSES, g_val, color=ACCENT, alpha=0.85,
                edgecolor='white', linewidth=0.5)
    _style_ax(axes[0], 'SuperEmotion Train', 'Emotion', 'Count')
    axes[0].tick_params(axis='x', rotation=30)

    m_cnt = Counter(val_df['emotion_name'].tolist())
    m_val = [m_cnt.get(e, 0) for e in EMOTION_CLASSES]
    axes[1].bar(EMOTION_CLASSES, m_val, color=ACCENT, alpha=0.85,
                edgecolor='white', linewidth=0.5)
    _style_ax(axes[1], 'SuperEmotion Val', 'Emotion', 'Count')
    axes[1].tick_params(axis='x', rotation=30)

    x  = np.arange(len(EMOTION_CLASSES))
    w  = 0.35
    gp = np.array(g_val) / max(sum(g_val), 1) * 100
    mp = np.array(m_val) / max(sum(m_val), 1) * 100
    axes[2].bar(x-w/2, gp, w, label='Train', color='#3498db',
                alpha=0.85, edgecolor='white')
    axes[2].bar(x+w/2, mp, w, label='Val', color='#e74c3c',
                alpha=0.85, edgecolor='white')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(EMOTION_CLASSES, rotation=30,
                             color='#cccccc', fontsize=8)
    axes[2].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    _style_ax(axes[2], 'Train vs Val (%)', 'Emotion', '%')

    plt.tight_layout()
    p = f"{PLOT_DIR}/04_dataset_distributions.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ 04_dataset_distributions.png")
    return p

def plot_phase_comparison(goemo_f1, pre_meld_f1, final_f1,
                          pc_before, pc_after):
    """Plot 5: Pre-training impact comparison — macro F1 + per-class delta."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Impact of GoEmotions Pre-training (v5)',
                 color='white', fontsize=12, fontweight='bold')

    systems = ['GoEmotions\nPre-train', 'MELD\n(pre-ft)', 'Final\n(Ours)']
    f1s     = [goemo_f1*100, pre_meld_f1*100, final_f1*100]
    colors  = ['#3498db', '#f39c12', '#2ecc71']
    bars    = axes[0].bar(systems, f1s, color=colors,
                          alpha=0.85, edgecolor='white', width=0.5)
    for bar, v in zip(bars, f1s):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.3,
                     f'{v:.2f}%', ha='center', va='bottom',
                     color='white', fontsize=10, fontweight='bold')
    axes[0].set_ylim(0, 100)
    _style_ax(axes[0], 'Macro F1 Across Phases', 'Phase', 'Macro F1 (%)')

    x = np.arange(len(EMOTION_CLASSES))
    w = 0.35
    axes[1].bar(x-w/2, [v*100 for v in pc_before], w,
                label='Before ft', color='#e74c3c', alpha=0.85,
                edgecolor='white')
    axes[1].bar(x+w/2, [v*100 for v in pc_after], w,
                label='After ft',  color='#2ecc71', alpha=0.85,
                edgecolor='white')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(EMOTION_CLASSES, rotation=30,
                             color='#cccccc', fontsize=8)
    axes[1].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    axes[1].set_ylim(0, 105)
    _style_ax(axes[1], 'Per-Class F1: Before vs After Fine-tuning',
              'Emotion', 'F1 (%)')

    plt.tight_layout()
    p = f"{PLOT_DIR}/05_pretraining_impact.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ 05_pretraining_impact.png")
    return p


def plot_attention_weights(model, sample_texts, vocab):
    model.eval()
    fig, axes = plt.subplots(len(sample_texts), 1,
                              figsize=(14, 2.5 * len(sample_texts)))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Attention Weights — Dialogue Transformer v5',
                 color='white', fontsize=11, fontweight='bold')
    if len(sample_texts) == 1:
        axes = [axes]

    zero_hist = torch.zeros(1, CFG1.NUM_PREV_UTT, CFG1.MAX_LEN,
                             dtype=torch.long).to(DEVICE)

    for ax, (text, true_emo) in zip(axes, sample_texts):
        ids    = encode(text, vocab, CFG1.MAX_LEN)
        tokens = tokenize(text)[:CFG1.MAX_LEN]
        n_tok  = len(tokens)

        id_t   = torch.tensor([ids], dtype=torch.long).to(DEVICE)
        spk_t  = torch.tensor([0],  dtype=torch.long).to(DEVICE)
        trn_t  = torch.tensor([0],  dtype=torch.long).to(DEVICE)

        with torch.no_grad():
            logits, _, all_attn = model(id_t, spk_t, trn_t,
                                        zero_hist, return_vector=True)
            pred_id = logits.argmax(-1).item()
            pred    = EMOTION_CLASSES[pred_id]

        avg_attn = torch.stack([a.mean(dim=1) for a in all_attn]).mean(dim=0)
        attn_row = avg_attn[0, 0, 1:n_tok+1].cpu().numpy()
        if attn_row.max() > 0:
            attn_row = attn_row / attn_row.max()

        ax.imshow(attn_row.reshape(1, -1), aspect='auto',
                  cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_xticks(range(n_tok))
        ax.set_xticklabels(tokens, rotation=45, ha='right',
                            fontsize=8, color='white')
        ax.set_yticks([])
        color = '#2ecc71' if pred == true_emo else '#e74c3c'
        ax.set_title(f'"{text[:55]}..."  True: {true_emo}  Pred: {pred}',
                     color=color, fontsize=8, pad=4)
        ax.set_facecolor(PANEL_BG)

    plt.tight_layout()
    p = f"{PLOT_DIR}/06_attention_weights.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ 06_attention_weights.png")
    return p


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 15 — Load Data                                                      │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "="*60)
print("  LOADING DATASETS  (SuperEmotion only)")
print("="*60)

# Load all 3 splits from HuggingFace in one call.
# train_df → Phase 1 pre-training  (~300k after ISEAR cap)
# val_df   → Phase 2 fine-tuning   (~54k, used as-is)
# test_df  → Final evaluation      (~58k, never touched during training)
superemo_train_df, superemo_val_df, superemo_test_df = load_superemotion(
    CFG1.SUPEREMO_HF, CFG1.MAX_SAMPLES_PER_SOURCE)

print(f"\n  Split sizes:")
print(f"    train : {len(superemo_train_df):,}")
print(f"    val   : {len(superemo_val_df):,}")
print(f"    test  : {len(superemo_test_df):,}")

# Vocabulary built from all texts
all_texts = (superemo_train_df['text'].tolist() +
             superemo_val_df['text'].tolist() +
             superemo_test_df['text'].tolist())
vocab = build_vocab(all_texts, CFG1.VOCAB_SIZE)
print(f"\n  Vocab size: {len(vocab):,}")

plot_dataset_distributions(superemo_train_df, superemo_val_df)

# Datasets — all use SuperEmotionDataset (same interface for all splits)
goemo_train_ds = SuperEmotionDataset(superemo_train_df, vocab)  # Phase 1 train
goemo_val_ds   = SuperEmotionDataset(superemo_val_df,   vocab)  # Phase 1 val
meld_train_ds  = SuperEmotionDataset(superemo_val_df,   vocab)  # Phase 2 train
meld_val_ds    = SuperEmotionDataset(superemo_val_df,   vocab)  # Phase 2 val monitor
meld_test_ds   = SuperEmotionDataset(superemo_test_df,  vocab)  # Final eval

_kw = dict(num_workers=2, pin_memory=True, collate_fn=collate_utt)
goemo_tr_loader = DataLoader(goemo_train_ds, CFG1.GOEMO_BATCH, shuffle=True,  **_kw)
goemo_va_loader = DataLoader(goemo_val_ds,   CFG1.GOEMO_BATCH, shuffle=False, **_kw)
meld_va_loader  = DataLoader(meld_val_ds,    CFG1.MELD_BATCH,  shuffle=False, **_kw)
meld_te_loader  = DataLoader(meld_test_ds,   CFG1.MELD_BATCH,  shuffle=False, **_kw)
print("  DataLoaders ready ✓")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 16 — Build Models                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

utt_encoder = DialogueTransformerEncoder(
    vocab_size = len(vocab),
).to(DEVICE)

stage_encoder = StageDirectionEncoder(
    vocab_size      = len(vocab),
    embed_dim       = CFG1.EMBED_DIM,
    hidden_dim      = CFG1.D_MODEL // 2,
    num_scene_types = CFG1.NUM_SCENE_TYPES,
    dropout         = CFG1.DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in utt_encoder.parameters())
print(f"\n  DialogueTransformerEncoder: {total_params:,} params")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 17 — Phase 1: SuperEmotion Pre-training                             │
# │                                                                          │
# │  SuperEmotion replaces GoEmotions                                   │
# │  LR = 2e-4  |  No neutral penalty  |  label_smoothing = 0.05           │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  PHASE 1 — SuperEmotion PRE-TRAINING")
print(f"  {len(goemo_train_ds):,} samples  "
      f"{CFG1.GOEMO_EPOCHS} epochs  LR={CFG1.GOEMO_LR}")
print("  Single-label clean taxonomy | LR=2e-4 | smooth=0.05")
print("═"*60)

g_labels  = [s['emotion'].item() for s in goemo_train_ds.samples]
g_weights = compute_class_weights(
    g_labels, CFG1.NUM_EMOTIONS, max_weight=10.0).to(DEVICE)

# FIX 3 + FIX 4: Plain CrossEntropy, no penalty, smooth=0.05
loss_fn_g  = Phase1Loss(weights=g_weights, label_smoothing=0.05)
optimizer1 = optim.Adam(utt_encoder.parameters(),
                         lr=CFG1.GOEMO_LR, weight_decay=1e-5)
# Warmup + cosine scheduler (per-step)
# 5% of total steps for warmup — standard NLP practice
_p1_total_steps  = CFG1.GOEMO_EPOCHS * len(goemo_tr_loader)
_p1_warmup_steps = max(1, int(0.05 * _p1_total_steps))
scheduler1 = make_warmup_cosine_scheduler(
    optimizer1, _p1_warmup_steps, _p1_total_steps)
print(f"  Warmup: {_p1_warmup_steps} steps ({_p1_warmup_steps/len(goemo_tr_loader):.1f} epochs) "
      f"→ cosine over {_p1_total_steps} total steps")

goemo_history = {'train': [], 'val': []}
best_goemo_f1 = 0.0

for epoch in range(CFG1.GOEMO_EPOCHS):
    tl, tf = train_one_epoch(
        utt_encoder, goemo_tr_loader, loss_fn_g,
        optimizer1, epoch, CFG1.GOEMO_EPOCHS, '[SuperEmo]',
        batch_scheduler=scheduler1)
    vl, vf, _, _ = eval_model(
        utt_encoder, goemo_va_loader, loss_fn_g, 'SuperEmo-Val')
    # No epoch-level scheduler.step() — warmup+cosine steps per batch

    goemo_history['train'].append({'epoch': epoch+1, 'loss': tl, 'f1': tf})
    goemo_history['val'].append(  {'epoch': epoch+1, 'loss': vl, 'f1': vf})

    if vf > best_goemo_f1:
        best_goemo_f1 = vf
        torch.save(utt_encoder.state_dict(),
                   f"{CFG1.CKPT_DIR}/phase1_pretrain.pt")
        print(f"    ✓ Phase 1 saved (f1={best_goemo_f1*100:.2f}%)")

print(f"\n  Phase 1 complete. Best F1: {best_goemo_f1*100:.2f}%")
print(f"  (Target: 30-40%  |  Previous best: 13.56%)")

# MELD val BEFORE fine-tuning (now uses Phase1Loss to avoid NeutralPenalty crash)
print("\n  MELD val before fine-tuning ...")
_, pre_finetune_f1, yt_pre, yp_pre = eval_model(
    utt_encoder, meld_va_loader, loss_fn_g, 'MELD-Val (pre-ft)')
pc_before = f1_score(yt_pre, yp_pre, average=None, zero_division=0,
                     labels=list(range(7)))


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 18 — Phase 2: MELD Fine-tuning                                      │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  PHASE 2 — MELD FINE-TUNING")
print(f"  {len(meld_train_ds):,} samples  "
      f"{CFG1.MELD_EPOCHS} epochs  LR={CFG1.MELD_LR}")
print("  NeutralPenaltyLoss now ACTIVE (Phase 2 only)")
print("═"*60)

# Load best Phase 1 weights
utt_encoder.load_state_dict(
    torch.load(f"{CFG1.CKPT_DIR}/phase1_pretrain.pt", map_location=DEVICE))

m_labels  = [s['emotion'].item() for s in meld_train_ds.samples]
m_weights = compute_class_weights(
    m_labels, CFG1.NUM_EMOTIONS, max_weight=15.0,
    extra_boost={'scared': 1.5, 'powerful': 1.5}).to(DEVICE)

loss_fn_m  = Phase2Loss(
    emotion_weights = m_weights,
    neutral_penalty = 1.8,
    label_smoothing = 0.1,
)

meld_sampler   = make_balanced_sampler(meld_train_ds)
meld_tr_loader = DataLoader(
    meld_train_ds, batch_size=CFG1.MELD_BATCH,
    sampler=meld_sampler, collate_fn=collate_utt,
    num_workers=2, pin_memory=True)
print("  ✓ Balanced sampler active")

optimizer2 = optim.Adam(
    list(utt_encoder.parameters()) + list(stage_encoder.parameters()),
    lr=CFG1.MELD_LR, weight_decay=1e-5)

_p2_total_steps  = CFG1.MELD_EPOCHS * len(meld_tr_loader)
_p2_warmup_steps = max(1, int(0.05 * _p2_total_steps))
scheduler2 = make_warmup_cosine_scheduler(
    optimizer2, _p2_warmup_steps, _p2_total_steps)
print(f"  Warmup: {_p2_warmup_steps} steps ({_p2_warmup_steps/len(meld_tr_loader):.1f} epochs) "
      f"→ cosine over {_p2_total_steps} total steps")

meld_history = {'train': [], 'val': []}
best_meld_f1 = 0.0
patience_c   = 0

for epoch in range(CFG1.MELD_EPOCHS):
    tl, tf = train_one_epoch(
        utt_encoder, meld_tr_loader, loss_fn_m,
        optimizer2, epoch, CFG1.MELD_EPOCHS, '[MELD]',
        batch_scheduler=scheduler2)
    vl, vf, _, _ = eval_model(
        utt_encoder, meld_va_loader, loss_fn_m, 'MELD-Val')
    # No epoch-level scheduler.step() — warmup+cosine steps per batch

    meld_history['train'].append({'epoch': epoch+1, 'loss': tl, 'f1': tf})
    meld_history['val'].append(  {'epoch': epoch+1, 'loss': vl, 'f1': vf})

    if vf > best_meld_f1:
        best_meld_f1 = vf
        patience_c   = 0
        torch.save({
            'utt_encoder':   utt_encoder.state_dict(),
            'stage_encoder': stage_encoder.state_dict(),
            'macro_f1':      best_meld_f1,
            'vocab':         vocab,
            'config': {
                'vocab_size':    len(vocab),
                'd_model':       CFG1.D_MODEL,
                'num_heads':     CFG1.NUM_HEADS,
                'num_layers':    CFG1.NUM_LAYERS,
                'd_ff':          CFG1.D_FF,
                'embed_dim':     CFG1.EMBED_DIM,
                'speaker_dim':   CFG1.SPEAKER_DIM,
                'turn_dim':      CFG1.TURN_DIM,
                'max_turns':     CFG1.MAX_TURNS,
                'num_speakers':  CFG1.MAX_SPEAKERS,
                'num_emotions':  CFG1.NUM_EMOTIONS,
                'num_prev':      CFG1.NUM_PREV_UTT,
                'dropout':       CFG1.DROPOUT,
            },
        }, f"{CFG1.CKPT_DIR}/module1_best.pt")
        print(f"    ✓ Saved (f1={best_meld_f1*100:.2f}%)")
    else:
        patience_c += 1
        if patience_c >= CFG1.EARLY_STOP:
            print(f"\n  ⏹  Early stop at epoch {epoch+1}")
            break


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 19 — Final Evaluation + Plots                                       │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  FINAL EVALUATION + REPORT PLOTS")
print("═"*60)

ckpt = torch.load(f"{CFG1.CKPT_DIR}/module1_best.pt", map_location=DEVICE)
utt_encoder.load_state_dict(ckpt['utt_encoder'])
utt_encoder.eval()

_, test_f1, yt_test, yp_test = eval_model(
    utt_encoder, meld_te_loader, loss_fn_m, 'TEST (argmax)')

print("\n  Calibrating thresholds on val set ...")
thresholds = calibrate_thresholds(utt_encoder, meld_va_loader)

yt_cal, yp_cal = predict_with_thresholds(utt_encoder, meld_te_loader, thresholds)
cal_f1 = f1_score(yt_cal, yp_cal, average='macro', zero_division=0)

print(f"\n  F1 comparison:")
print(f"    Standard argmax      : {test_f1*100:.2f}%")
print(f"    Calibrated threshold : {cal_f1*100:.2f}%")

if cal_f1 > test_f1:
    print(f"  → Using calibrated (+{(cal_f1-test_f1)*100:.2f}%)")
    yt_test, yp_test = yt_cal, yp_cal
    test_f1          = cal_f1
    np.save(f"{CFG1.CKPT_DIR}/decision_thresholds.npy", thresholds)

pc_after = f1_score(yt_test, yp_test, average=None, zero_division=0,
                    labels=list(range(7)))

print("\n  Classification Report (Test Set):")
print(classification_report(yt_test, yp_test,
                             labels=list(range(7)),
                             target_names=EMOTION_CLASSES,
                             zero_division=0))

# Generate plots
print("\n  Generating report plots ...")
plot_training_curves(goemo_history, meld_history)
plot_confusion_matrix(yt_test, yp_test,
                      'Confusion Matrix — Test Set (DTE v5)',
                      '02_confusion_matrix_test.png')
plot_confusion_matrix(yt_pre, yp_pre,
                      'Confusion Matrix — Before Fine-tuning',
                      '02b_confusion_matrix_pre_finetune.png')
plot_per_class_f1(yt_test, yp_test,
                  f'Per-Class F1 — Test Set (Macro={test_f1*100:.2f}%)',
                  '03_per_class_f1.png')

plot_phase_comparison(
    goemo_f1    = best_goemo_f1,
    pre_meld_f1 = pre_finetune_f1,
    final_f1    = test_f1,
    pc_before   = pc_before,
    pc_after    = pc_after)

# Attention visualisation
sample_texts = [
    ("I can't believe you would do that to me",      'mad'),
    ("Oh my god this is the best day ever!",          'joyful'),
    ("I don't know I just feel so empty inside",      'sad'),
    ("Wait did you hear that something is out there", 'scared'),
    ("Everything is fine I'm totally calm",           'peaceful'),
]
plot_attention_weights(utt_encoder, sample_texts, vocab)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 20 — Extract Scene Emotion Matrices → Module 2                     │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  EXTRACTING SCENE EMOTION MATRICES → MODULE 2")
print("═"*60)

# SuperEmotion has no dialogue/scene structure — each sample is independent.
# We group test samples into pseudo-scenes of fixed window size (10 utterances)
# so Module 2 receives the same matrix format it expects.
SCENE_WINDOW = 10
all_test_samples = meld_test_ds.samples

scene_data = []
utt_encoder.eval()
for i in tqdm(range(0, len(all_test_samples), SCENE_WINDOW),
              desc="  Extracting"):
    utts   = all_test_samples[i : i + SCENE_WINDOW]
    if not utts:
        continue
    ids_b  = torch.stack([u['token_ids']   for u in utts]).to(DEVICE)
    spk_b  = torch.stack([u['speaker_id']  for u in utts]).to(DEVICE)
    trn_b  = torch.stack([u['turn_pos']    for u in utts]).to(DEVICE)
    hist_b = torch.stack([u['utt_history'] for u in utts]).to(DEVICE)

    with torch.no_grad():
        _, vectors, _ = utt_encoder(ids_b, spk_b, trn_b, hist_b,
                                     return_vector=True)
    scene_data.append({
        'scene_id':       (0, 0, i // SCENE_WINDOW),
        'emotion_matrix': vectors.cpu(),
        'labels':         [u['emotion'].item() for u in utts],
    })

import pickle as pkl
os.makedirs(f"{CFG1.CKPT_DIR}/final_emotion_vectors", exist_ok=True)
torch.save(scene_data,
           f"{CFG1.CKPT_DIR}/final_emotion_vectors/scene_emotion_matrix.pt")

label_enc = {
    'LABEL2IDX':        {e: i for i, e in enumerate(EMOTION_CLASSES)},
    'IDX2LABEL':        dict(enumerate(EMOTION_CLASSES)),
    'CLASS_NAMES':      EMOTION_CLASSES,
    'SCENE_TYPE_NAMES': SCENE_TYPE_NAMES,
}
with open(f"{CFG1.CKPT_DIR}/label_encoder.pkl", 'wb') as f:
    pkl.dump(label_enc, f)
with open(f"{CFG1.CKPT_DIR}/m1_history.json", 'w') as f:
    json.dump({'goemo_history': goemo_history,
               'meld_history':  meld_history}, f, indent=2)

print(f"\n  ✓ {len(scene_data):,} pseudo-scenes extracted")
print(f"  ✓ All artifacts saved to {CFG1.CKPT_DIR}/")

print(f"\n  {'═'*55}")
print(f"  FINAL RESULTS SUMMARY — Module 1 v6")
print(f"  {'═'*55}")
print(f"  Phase 1 SuperEmotion best F1 : {best_goemo_f1*100:.2f}%")
print(f"  Val  (pre-finetune)          : {pre_finetune_f1*100:.2f}%")
print(f"  Test (post-finetune)         : {test_f1*100:.2f}%")
print(f"\n  Per-class F1 (test set):")
for i, emo in enumerate(EMOTION_CLASSES):
    delta = (pc_after[i] - pc_before[i]) * 100
    sign  = '+' if delta >= 0 else ''
    print(f"    {emo:<10}  {pc_after[i]*100:5.1f}%  "
          f"({sign}{delta:.1f}% vs pre-ft)")
print(f"\n  Plots → {PLOT_DIR}/")
print(f"  {'═'*55}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 21 — Zip & Download all outputs from Kaggle                        │
# │                                                                          │
# │  Zips checkpoints + plots into a single file, then triggers browser     │
# │  download using Kaggle's FileLink utility.                               │
# │                                                                          │
# │  Contents of module1_outputs.zip:                                       │
# │    checkpoints/                                                          │
# │      phase1_pretrain.pt          ← GoEmotions pre-trained weights       │
# │      module1_best.pt             ← best MELD checkpoint (full state)    │
# │      decision_thresholds.npy     ← calibrated per-class thresholds      │
# │      label_encoder.pkl           ← LABEL2IDX / IDX2LABEL / speaker2id  │
# │      m1_history.json             ← full train/val loss+F1 history       │
# │      final_emotion_vectors/                                              │
# │        scene_emotion_matrix.pt   ← scene vectors for Module 2           │
# │    plots/                                                                │
# │      01_training_curves.png                                              │
# │      02_confusion_matrix_test.png                                        │
# │      02b_confusion_matrix_pre_finetune.png                               │
# │      03_per_class_f1.png                                                 │
# │      04_dataset_distributions.png                                        │
# │      05_pretraining_impact.png                                           │
# │      06_attention_weights.png                                            │
# └──────────────────────────────────────────────────────────────────────────┘

import zipfile
from kaggle_secrets import UserSecretsClient  # noqa: F401 — confirms Kaggle env
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/module1_outputs.zip"

print("\n" + "═"*60)
print("  ZIPPING OUTPUTS FOR DOWNLOAD")
print("="*60)

# Collect all files to zip
_dirs_to_zip = [
    ("/kaggle/working/checkpoints", "checkpoints"),
    ("/kaggle/working/plots",       "plots"),
]

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc_prefix in _dirs_to_zip:
        src_path = Path(src_dir)
        if not src_path.exists():
            print(f"  ⚠️  Skipping missing dir: {src_dir}")
            continue
        files = list(src_path.rglob('*'))
        for fpath in files:
            if fpath.is_file():
                arcname = arc_prefix + '/' + str(fpath.relative_to(src_path))
                zf.write(str(fpath), arcname)
                print(f"  + {arcname}  ({fpath.stat().st_size / 1024:.1f} KB)")

zip_size_mb = Path(ZIP_PATH).stat().st_size / (1024 * 1024)
print(f"\n  ✓ module1_outputs.zip  ({zip_size_mb:.1f} MB)")
print(f"  Path: {ZIP_PATH}")

# Render a clickable download link in the Kaggle notebook output
print("\n  Click the link below to download:")
display(FileLink(ZIP_PATH, result_html_prefix="⬇️  Download: "))

  Device : cuda
  GPU    : Tesla T4
  Plots  → /kaggle/working/plots
  Config ready ✓
  Transformer: 4L × 4H × 256d
  MAX_SPEAKERS: 2  (speaker embedding kept for architecture compatibility)

  LOADING DATASETS  (SuperEmotion only)

  Loading SuperEmotion: cirimus/super-emotion
  (first run downloads ~200 MB → cached to ~/.cache/huggingface)
  Columns: ['text', 'labels', 'labels_str', 'labels_source', 'source']
  text_col='text'  label_col='labels_str'  source_col='source'

  [train] Source counts before cap (50,000/source):
    ISEAR                  333,447
    GoEmotions              42,021
    Crowdflower             31,336
    TwitterEmotion          16,000
    MELD                     9,989
    SemEval                  6,568
  [train] After cap: 155,914 samples

  [train] 147,924 samples  (skipped 7,990)
    neutral    24,688   16.7%  █████████████
    joyful     36,811   24.9%  ████████████████████
    sad        27,138   18.3%  ██████████████
    mad        17,532   11.9%  ████

[SuperEmo] Ep 01/12 ▶ Train:   0%|                         | 0/1156 [00:00<?, ?it/s]

    ↳ TRAIN  loss=1.7844  macro_f1=25.97%  (0m54s)
    ↳ SUPEREMO-VAL  loss=0.9417  macro_f1=71.12%  acc=76.97%
    ✓ Phase 1 saved (f1=71.12%)


[SuperEmo] Ep 02/12 ▶ Train:   0%|                         | 0/1156 [00:00<?, ?it/s]

    ↳ TRAIN  loss=1.1726  macro_f1=61.87%  (0m55s)
    ↳ SUPEREMO-VAL  loss=0.7514  macro_f1=74.98%  acc=81.27%
    ✓ Phase 1 saved (f1=74.98%)


...

[MELD] Ep 15/15 ▶ Train:   0%|                         | 0/841 [00:00<?, ?it/s]

    ↳ TRAIN  loss=0.8949  macro_f1=94.97%  (0m24s)
    ↳ MELD-VAL      loss=0.8425  macro_f1=92.41%  acc=92.99%
    ✓ Saved (f1=92.41%)

════════════════════════════════════════════════════════════
  FINAL EVALUATION + REPORT PLOTS
════════════════════════════════════════════════════════════
    ↳ TEST (ARGMAX)  loss=1.7001  macro_f1=71.27%  acc=78.16%

  Calibrating thresholds on val set ...

  Calibrated thresholds:
    neutral    0.40 ↓
    joyful     0.10 ↓
    sad        0.10 ↓
    mad        0.35 ↓
    scared     0.45 ↓
    peaceful   0.45 ↓
    powerful   0.45 ↓

  F1 comparison:
    Standard argmax      : 71.27%
    Calibrated threshold : 71.50%
  → Using calibrated (+0.23%)

  Classification Report (Test Set):
              precision    recall  f1-score   support

     neutral       0.47      0.42      0.44      3766
      joyful       0.86      0.82      0.84     17254
         sad       0.91      0.88      0.90     13900
         mad       0.81      0.78      0.80      7710


  Extracting:   0%|          | 0/5634 [00:00<?, ?it/s]


  ✓ 5,634 pseudo-scenes extracted
  ✓ All artifacts saved to /kaggle/working/checkpoints/

  ═══════════════════════════════════════════════════════
  FINAL RESULTS SUMMARY — Module 1 v6
  ═══════════════════════════════════════════════════════
  Phase 1 SuperEmotion best F1 : 75.58%
  Val  (pre-finetune)          : 74.19%
  Test (post-finetune)         : 71.50%

  Per-class F1 (test set):
    neutral      44.1%  (-3.1% vs pre-ft)
    joyful       84.0%  (-4.1% vs pre-ft)
    sad          89.6%  (-1.0% vs pre-ft)
    mad          79.6%  (-3.5% vs pre-ft)
    scared       75.2%  (-0.5% vs pre-ft)
    peaceful     70.8%  (-3.7% vs pre-ft)
    powerful     57.2%  (-3.1% vs pre-ft)

  Plots → /kaggle/working/plots/
  ═══════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  ZIPPING OUTPUTS FOR DOWNLOAD
  + checkpoints/phase1_pretrain.pt  (29953.0 KB)
  + checkpoints/module1_best.pt  (43281.1 KB)
  + checkpoints/decision_thresho

/kaggle/working/module1_outputs.zip

In [ ]:
print("simple emotion detection on superemotion")

simple emotion detection on superemotion


In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODULE 1 — v7  Phase 2: MELD Fine-tuning                              ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  PURPOSE:                                                               ║
# ║    Load Phase 1 checkpoint (trained on SuperEmotion, 75.58% F1)        ║
# ║    Fine-tune on real MELD dialogue data                                 ║
# ║    Extract properly grouped scene matrices → Module 2                  ║
# ║                                                                         ║
# ║  WHAT THIS ADDS OVER PHASE 1:                                           ║
# ║    turn_pos    — real utterance index within dialogue (0,1,2…)          ║
# ║    utt_history — real previous 5 utterances as token sequences          ║
# ║    speaker_id  — real speaker identity (Monica=3, Ross=7, etc.)         ║
# ║    Scene grouping by (Season, Episode, Dialogue_ID) — real arcs         ║
# ║                                                                         ║
# ║  CHECKPOINT NAMING (easy to identify):                                  ║
# ║    INPUT  : module1_phase1_pretrain.pt   ← from Kaggle dataset         ║
# ║    OUTPUT : module1_phase2_best.pt       ← best MELD val F1            ║
# ║    EXTRA  : decision_thresholds.npy, label_encoder.pkl                  ║
# ║             scene_emotion_matrix.pt                                     ║
# ║                                                                         ║
# ║  ARCHITECTURE: Identical to v6 — no changes                            ║
# ║    DialogueTransformerEncoder (7.66M params)                            ║
# ║    Loaded from phase1_pretrain.pt, weights preserved                   ║
# ║                                                                         ║
# ║  EXPECTED MELD TEST F1: 48–58%                                          ║
# ║    Lower than SuperEmotion 71% — MELD is harder:                       ║
# ║      only 9,989 train samples vs 147k                                   ║
# ║      47% neutral imbalance                                              ║
# ║      Friends TV → different domain from Reddit/Twitter                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import os, json, pickle, time, re, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import (Dataset, DataLoader,
                               WeightedRandomSampler)
from sklearn.metrics import (f1_score, accuracy_score,
                              classification_report,
                              confusion_matrix)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available()
                         else "cpu")
PLOT_DIR = "/kaggle/working/plots"
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

class CFG1:
    # ── Input: Phase 1 checkpoint (from v6 SuperEmotion training) ────────
    # Upload your module1_outputs.zip to Kaggle as a dataset, then:
    # phase1_pretrain.pt  — weights saved at best Phase 1 val F1
    # module1_phase1_best.pt — same but with full config + vocab dict
    # We need module1_phase1_best.pt for the vocab; fall back to phase1_pretrain.pt
    # for weights only if that is all that is available.
    PHASE1_CKPT = ("/kaggle/input/datasets/suyashnpande/module1-phase1"
                   "/checkpoints/phase1_pretrain.pt")
    PHASE1_BEST = ("/kaggle/input/datasets/suyashnpande/module1-phase1"
                   "/checkpoints/module1_best.pt")

    # ── MELD paths ────────────────────────────────────────────────────────
    MELD_TRAIN  = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw/train/train_sent_emo.csv"
    MELD_DEV    = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw/dev_sent_emo.csv"
    MELD_TEST   = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw/test_sent_emo.csv"

    # ── Output: clearly named for easy identification ─────────────────────
    CKPT_DIR    = "/kaggle/working/checkpoints"
    # Phase 2 best checkpoint
    PHASE2_BEST = "/kaggle/working/checkpoints/module1_phase2_best.pt"
    # Thresholds calibrated on MELD dev set
    THRESHOLDS  = "/kaggle/working/checkpoints/decision_thresholds.npy"
    # Scene matrices for Module 2
    SCENE_OUT   = ("/kaggle/working/checkpoints"
                   "/final_emotion_vectors/scene_emotion_matrix.pt")
    LABEL_ENC   = "/kaggle/working/checkpoints/label_encoder.pkl"

    # ── Architecture (MUST match v6 exactly) ─────────────────────────────
    VOCAB_SIZE   = 20000
    MAX_LEN      = 50
    EMBED_DIM    = 128
    D_MODEL      = 256
    NUM_HEADS    = 4
    NUM_LAYERS   = 4
    D_FF         = 512
    SPEAKER_DIM  = 64
    TURN_DIM     = 32
    MAX_TURNS    = 60
    NUM_PREV_UTT = 5
    NUM_EMOTIONS = 7
    NUM_SCENE_TYPES = 6
    MAX_SPEAKERS = 500    # enough for all MELD characters
    DROPOUT      = 0.2

    # ── Phase 2 training ─────────────────────────────────────────────────
    EPOCHS      = 20
    LR          = 1e-4    # lower than Phase 1 — preserve pretrained weights
    BATCH_SIZE  = 64
    EARLY_STOP  = 5

Path(CFG1.CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG1.SCENE_OUT).parent.mkdir(parents=True, exist_ok=True)
print("  Config ready ✓")
print(f"  Phase 1 checkpoint : {CFG1.PHASE1_CKPT}")
print(f"  Phase 2 output     : {CFG1.PHASE2_BEST}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Emotion Labels + MELD Mapping                                   │
# └──────────────────────────────────────────────────────────────────────────┘

# Same 7-class system used throughout the project
EMOTION_CLASSES  = ['neutral', 'joyful', 'sad', 'mad',
                    'scared', 'peaceful', 'powerful']
SCENE_TYPE_NAMES = ['dialogue', 'action', 'investigation',
                    'suspense', 'dramatic_reveal', 'transition']

# MELD original 7 emotions → our 7 classes
# MELD classes: neutral, joy, sadness, anger, disgust, fear, surprise
# MELD original labels: neutral, joy, sadness, anger, disgust, fear, surprise
#
# KEY DECISION — surprise → powerful (NOT joyful):
#   Consistent with Phase 1 SuperEmotion mapping (surprise → powerful).
#   MELD surprise = "Oh my god!", "I can't believe this!", "What?!"
#   Film music for these moments = dramatic stinger / reveal chord = powerful.
#   surprise → joyful would give peaceful and powerful ZERO training samples.
#   surprise → powerful gives powerful 1,205 samples (12.1%) — properly learned.
#
# MELD has no 'peaceful' examples at all.
# The model relies entirely on Phase 1 pretrained weights for peaceful.
# This is acceptable — peaceful (calm background music) is rare in
# Friends TV dialogue which is the MELD source.
MELD_MERGE = {
    'neutral':     'neutral',   # 47.2% of MELD — heaviest class
    'joy':         'joyful',    # 17.5% — bright uplifting music
    'surprise':    'powerful',  # 12.1% — dramatic reveal / stinger
                                #   consistent with Phase 1 mapping
    'anger':       'mad',       # 11.1% — aggressive driving music
    'sadness':     'sad',       #  6.8% — slow minor strings
    'disgust':     'mad',       #  2.7% — merged with anger (same music)
    'fear':        'scared',    #  2.7% — tremolo, sparse, tense
    # additional labels found in some MELD versions
    'happiness':   'joyful',
    'excitement':  'joyful',
    'frustration': 'mad',
    # pass-through (already our 7 classes)
    'joyful':      'joyful',
    'sad':         'sad',
    'mad':         'mad',
    'scared':      'scared',
    'peaceful':    'peaceful',
    'powerful':    'powerful',
}

# After mapping, MELD train distribution:
#   neutral   4710  47.2%  ← WeightedRandomSampler handles this
#   joyful    1743  17.5%
#   powerful  1205  12.1%  ← gets samples now (was 0 with surprise→joyful)
#   mad       1377  13.8%
#   sad        683   6.8%
#   scared     268   2.7%
#   peaceful     0   0.0%  ← relies on Phase 1 pretrained weights

print(f"  Emotion classes: {EMOTION_CLASSES}")
print(f"  MELD mapping: {len(MELD_MERGE)} entries")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Vocabulary (rebuilt from MELD + loaded from Phase 1)            │
# │                                                                          │
# │  IMPORTANT: We rebuild vocab from all MELD text PLUS the Phase 1        │
# │  vocab. This ensures MELD-specific words (character names, show-         │
# │  specific phrases) are in the vocabulary.                                │
# │                                                                          │
# │  The Phase 1 checkpoint stores the vocab inside module1_best.pt.        │
# │  We extend it with any new MELD tokens, keeping all original IDs.       │
# └──────────────────────────────────────────────────────────────────────────┘

def tokenize(text: str) -> list:
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s\'\-]", ' ', text)
    return text.split()

def build_vocab(texts: list, max_size: int) -> dict:
    counter = Counter()
    for t in texts:
        counter.update(tokenize(str(t)))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

def encode(text: str, vocab: dict, max_len: int) -> list:
    tokens = tokenize(str(text))[:max_len]
    ids    = [vocab.get(t, 1) for t in tokens]
    ids   += [0] * (max_len - len(ids))
    return ids

def build_speaker_encoder(df_list: list) -> dict:
    all_speakers = set()
    for df in df_list:
        if 'Speaker' in df.columns:
            all_speakers.update(df['Speaker'].dropna().unique())
    speaker2id = {'UNKNOWN': 0}
    for sp in sorted(all_speakers):
        speaker2id[sp] = len(speaker2id)
    return speaker2id


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — MELD Dataset with Full Dialogue Context                         │
# │                                                                          │
# │  This is the key upgrade over Phase 1:                                  │
# │                                                                          │
# │  Phase 1 (SuperEmotion):                                                │
# │    turn_pos    = 0  (all utterances treated as first turn)              │
# │    utt_history = zeros  (no conversational context)                     │
# │    speaker_id  = 0  (all treated as UNKNOWN)                            │
# │                                                                          │
# │  Phase 2 (MELD) — NOW ACTIVE:                                           │
# │    turn_pos    = real index (0 = first line, 14 = 15th line)            │
# │    utt_history = previous 5 actual utterances as token IDs              │
# │    speaker_id  = real speaker (Monica=3, Ross=7, Chandler=2…)           │
# │                                                                          │
# │  This teaches the model:                                                │
# │    "Fine." at turn 0 vs turn 14 → different music                      │
# │    After 3 angry turns → next is likely angry or scared                 │
# │    Monica's neutral speech sounds different from Joey's neutral         │
# └──────────────────────────────────────────────────────────────────────────┘

class MELDDataset(Dataset):
    def __init__(self, df, vocab, speaker2id,
                 max_len=CFG1.MAX_LEN,
                 num_prev=CFG1.NUM_PREV_UTT):
        self.samples = []
        zero_ids     = [0] * max_len

        groups = df.groupby(
            ['Season', 'Episode', 'Dialogue_ID'])

        for (s, e, d), grp in groups:
            grp      = grp.sort_values('Utterance_ID')
            rows     = list(grp.itertuples())
            N        = len(rows)
            all_ids  = [encode(str(r.Utterance),
                                vocab, max_len)
                         for r in rows]

            for i, row in enumerate(rows):
                emo   = MELD_MERGE.get(
                    str(row.Emotion).lower(), 'neutral')
                label = EMOTION_CLASSES.index(emo)
                sp_id = min(
                    speaker2id.get(str(row.Speaker), 0),
                    CFG1.MAX_SPEAKERS - 1)

                # Build history: last num_prev utterances
                # zero-padded for early turns
                hist = []
                for k in range(num_prev, 0, -1):
                    hist.append(
                        all_ids[i - k]
                        if i - k >= 0 else zero_ids)

                self.samples.append({
                    'token_ids':   torch.tensor(
                        all_ids[i], dtype=torch.long),
                    'speaker_id':  torch.tensor(
                        sp_id, dtype=torch.long),
                    'turn_pos':    torch.tensor(
                        min(i, CFG1.MAX_TURNS - 1),
                        dtype=torch.long),
                    'utt_history': torch.tensor(
                        hist, dtype=torch.long),
                    'emotion':     torch.tensor(
                        label, dtype=torch.long),
                    # metadata for scene extraction
                    'scene_id':    (int(s), int(e), int(d)),
                    'position':    float(i / max(N-1, 1)),
                    'text':        str(row.Utterance),
                    'speaker':     str(row.Speaker),
                })

        print(f"  MELD dataset: {len(self.samples):,} "
              f"utterances from "
              f"{len(groups):,} dialogues")

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_utt(batch):
    return {
        'token_ids':   torch.stack(
            [b['token_ids']   for b in batch]),
        'speaker_id':  torch.stack(
            [b['speaker_id']  for b in batch]),
        'turn_pos':    torch.stack(
            [b['turn_pos']    for b in batch]),
        'utt_history': torch.stack(
            [b['utt_history'] for b in batch]),
        'emotion':     torch.stack(
            [b['emotion']     for b in batch]),
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — Transformer Architecture (copy-exact from v6)                   │
# │                                                                          │
# │  MUST be byte-for-byte identical to v6 so state_dict loads correctly.   │
# │  Any change → shape mismatch → load fails.                              │
# └──────────────────────────────────────────────────────────────────────────┘

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4,
                 dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads
        self.W_q  = nn.Linear(d_model, d_model, bias=False)
        self.W_k  = nn.Linear(d_model, d_model, bias=False)
        self.W_v  = nn.Linear(d_model, d_model, bias=False)
        self.W_o  = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        for w in [self.W_q, self.W_k, self.W_v, self.W_o]:
            nn.init.xavier_uniform_(w.weight)

    def _split_heads(self, x):
        B, T, _ = x.shape
        return x.view(
            B, T, self.num_heads, self.d_k
        ).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self._split_heads(self.W_q(x))
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))
        scores = (torch.matmul(Q, K.transpose(-2, -1))
                  / math.sqrt(self.d_k))
        if mask is not None:
            scores = scores.masked_fill(
                mask == 0, float('-inf'))
        attn_w = torch.nan_to_num(
            F.softmax(scores, dim=-1), nan=0.0)
        attn_w = self.drop(attn_w)
        context= torch.matmul(attn_w, V)
        context= (context.transpose(1, 2)
                  .contiguous()
                  .view(B, T, self.d_model))
        return self.W_o(context), attn_w


class FeedForward(nn.Module):
    def __init__(self, d_model=256, d_ff=512,
                 dropout=0.1):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff)
        self.fc2  = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        return self.fc2(
            self.drop(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, num_heads=4,
                 d_ff=512, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(
            d_model, num_heads, dropout)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_w = self.attn(
            self.ln1(x), mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x, attn_w


class DialogueTurnEmbedding(nn.Module):
    def __init__(self, max_turns=60, embed_dim=32):
        super().__init__()
        self.embed = nn.Embedding(max_turns, embed_dim)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, turn_positions):
        return self.embed(turn_positions)


class EmotionQueryToken(nn.Module):
    def __init__(self, d_model=256, num_emotions=7):
        super().__init__()
        self.prototypes = nn.Parameter(
            torch.randn(num_emotions, d_model) * 0.02)
        self.gate = nn.Linear(d_model, num_emotions)
        nn.init.xavier_uniform_(self.gate.weight)

    def get_query(self, x):
        mean_x  = x.mean(dim=1)
        weights = F.softmax(
            self.gate(mean_x), dim=-1)
        query   = torch.matmul(weights, self.prototypes)
        return query.unsqueeze(1)


class DialogueContextGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128,
                 d_model=256, context_dim=128,
                 num_prev=5, max_len=50, dropout=0.2):
        super().__init__()
        self.num_prev  = num_prev
        self.max_len   = max_len
        self.tok_embed = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0)
        self.utt_proj  = nn.Sequential(
            nn.Linear(embed_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
        )
        self.gru = nn.GRU(
            input_size  = d_model,
            hidden_size = context_dim,
            num_layers  = 1,
            batch_first = True)
        self.proj = nn.Sequential(
            nn.Linear(context_dim, context_dim),
            nn.LayerNorm(context_dim),
            nn.Tanh(),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, utt_history):
        B, P, T  = utt_history.shape
        flat     = utt_history.reshape(
            B * P, T).long()
        flat     = flat.clamp(
            0, self.tok_embed.num_embeddings - 1)
        emb      = self.tok_embed(flat)
        mask     = (flat != 0).float().unsqueeze(-1)
        pooled   = ((emb * mask).sum(1)
                    / (mask.sum(1) + 1e-6))
        pooled   = self.utt_proj(pooled).view(B, P, -1)
        _, h_n   = self.gru(pooled)
        context  = self.proj(h_n.squeeze(0))
        return self.drop(context)


class DialogueTransformerEncoder(nn.Module):
    def __init__(self,
                 vocab_size,
                 embed_dim    = CFG1.EMBED_DIM,
                 d_model      = CFG1.D_MODEL,
                 num_heads    = CFG1.NUM_HEADS,
                 num_layers   = CFG1.NUM_LAYERS,
                 d_ff         = CFG1.D_FF,
                 max_len      = CFG1.MAX_LEN,
                 max_turns    = CFG1.MAX_TURNS,
                 speaker_dim  = CFG1.SPEAKER_DIM,
                 turn_dim     = CFG1.TURN_DIM,
                 num_speakers = CFG1.MAX_SPEAKERS,
                 num_emotions = CFG1.NUM_EMOTIONS,
                 num_prev     = CFG1.NUM_PREV_UTT,
                 dropout      = CFG1.DROPOUT):
        super().__init__()
        self.d_model = d_model

        self.tok_embed  = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0)
        self.pos_embed  = nn.Embedding(
            max_len + 1, embed_dim)
        self.spk_embed  = nn.Embedding(
            num_speakers, speaker_dim)
        self.turn_embed = DialogueTurnEmbedding(
            max_turns, turn_dim)

        nn.init.normal_(self.tok_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        nn.init.normal_(self.spk_embed.weight, std=0.02)

        input_concat_dim = (embed_dim + embed_dim
                            + speaker_dim + turn_dim)
        self.input_proj = nn.Sequential(
            nn.Linear(input_concat_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
        )
        self.emo_query = EmotionQueryToken(
            d_model, num_emotions)
        self.layers    = nn.ModuleList([
            TransformerBlock(d_model, num_heads,
                              d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.final_ln  = nn.LayerNorm(d_model)
        self.ctx_gru   = DialogueContextGRU(
            vocab_size  = vocab_size,
            embed_dim   = embed_dim,
            d_model     = d_model,
            context_dim = 128,
            num_prev    = num_prev,
            max_len     = max_len,
            dropout     = dropout,
        )
        self.fusion = nn.Sequential(
            nn.Linear(d_model + 128, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.emo_head = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_emotions),
        )

    def _build_pad_mask(self, token_ids,
                         extra_token=True):
        B, T  = token_ids.shape
        pad   = (token_ids != 0).float()
        if extra_token:
            ones = torch.ones(
                B, 1, device=token_ids.device)
            pad  = torch.cat([ones, pad], dim=1)
        return pad.unsqueeze(1).unsqueeze(2)

    def encode(self, token_ids, speaker_ids,
               turn_positions):
        B, T  = token_ids.shape
        pos   = torch.arange(
            T, device=token_ids.device
        ).unsqueeze(0).expand(B, -1)
        tok_e = self.tok_embed(token_ids)
        pos_e = self.pos_embed(pos)
        spk_e = (self.spk_embed(speaker_ids)
                 .unsqueeze(1).expand(-1, T, -1))
        trn_e = (self.turn_embed(turn_positions)
                 .unsqueeze(1).expand(-1, T, -1))
        x     = self.input_proj(
            torch.cat([tok_e, pos_e,
                        spk_e, trn_e], dim=-1))
        query = self.emo_query.get_query(x)
        x     = torch.cat([query, x], dim=1)
        mask  = self._build_pad_mask(token_ids)
        all_attn = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            all_attn.append(attn_w)
        x = self.final_ln(x)
        return x[:, 0, :], all_attn

    def forward(self, token_ids, speaker_ids,
                turn_positions, utt_history,
                return_vector=False):
        cls_out, all_attn = self.encode(
            token_ids, speaker_ids, turn_positions)
        ctx_out = self.ctx_gru(utt_history)
        fused   = self.fusion(
            torch.cat([cls_out, ctx_out], dim=-1))
        logits  = self.emo_head(fused)
        if return_vector:
            return logits, fused, all_attn
        return logits


class StageDirectionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=128,
                 hidden_dim=128, num_scene_types=6,
                 dropout=0.3):
        super().__init__()
        self.tok_embed = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=2,
            batch_first=True, bidirectional=True,
            dropout=dropout)
        self.norm            = nn.LayerNorm(
            hidden_dim * 2)
        self.drop            = nn.Dropout(dropout)
        self.scene_type_head = nn.Sequential(
            nn.Linear(hidden_dim*2, 64), nn.ReLU(),
            nn.Linear(64, num_scene_types))
        self.action_head     = nn.Sequential(
            nn.Linear(hidden_dim*2, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.proj = nn.Sequential(
            nn.Linear(hidden_dim*2, 256),
            nn.LayerNorm(256), nn.ReLU())

    def forward(self, token_ids):
        x      = self.tok_embed(token_ids)
        out, _ = self.lstm(x)
        mask   = (token_ids != 0).unsqueeze(-1).float()
        pooled = self.drop(self.norm(
            (out * mask).sum(1)
            / (mask.sum(1) + 1e-6)))
        return {
            'stage_vector': self.proj(pooled),
            'scene_logits': self.scene_type_head(pooled),
            'action_score': self.action_head(
                pooled).squeeze(-1),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — Loss Functions                                                  │
# │                                                                          │
# │  Phase 2 uses NeutralPenaltyLoss:                                       │
# │    MELD is 47% neutral → model lazily predicts neutral                  │
# │    Extra ×1.8 loss when predicting neutral for non-neutral sample       │
# └──────────────────────────────────────────────────────────────────────────┘

class NeutralPenaltyLoss(nn.Module):
    def __init__(self, weights=None, neutral_idx=0,
                 neutral_penalty=1.8,
                 label_smoothing=0.1):
        super().__init__()
        self.neutral_idx     = neutral_idx
        self.neutral_penalty = neutral_penalty
        self.ce = nn.CrossEntropyLoss(
            weight          = weights,
            label_smoothing = label_smoothing,
            reduction       = 'none')

    def forward(self, logits, targets):
        loss_per     = self.ce(logits, targets)
        preds        = logits.argmax(dim=-1)
        lazy_neutral = ((preds   == self.neutral_idx) &
                        (targets != self.neutral_idx))
        multiplier   = torch.where(
            lazy_neutral,
            torch.full_like(loss_per,
                             self.neutral_penalty),
            torch.ones_like(loss_per))
        return (loss_per * multiplier).mean()


class Phase2Loss(nn.Module):
    def __init__(self, emotion_weights=None,
                 neutral_penalty=1.8,
                 label_smoothing=0.1):
        super().__init__()
        self.loss_fn = NeutralPenaltyLoss(
            weights         = emotion_weights,
            neutral_idx     = EMOTION_CLASSES.index(
                'neutral'),
            neutral_penalty = neutral_penalty,
            label_smoothing = label_smoothing,
        )

    def utterance_loss(self, logits, targets):
        return self.loss_fn(logits, targets)


def compute_class_weights(labels, num_classes,
                           max_weight=15.0,
                           extra_boost=None):
    counts  = np.bincount(
        labels, minlength=num_classes).astype(float)
    counts  = np.where(counts == 0, 1, counts)
    weights = len(labels) / (num_classes * counts)
    if extra_boost:
        for cls_name, mult in extra_boost.items():
            if cls_name in EMOTION_CLASSES:
                idx = EMOTION_CLASSES.index(cls_name)
                weights[idx] *= mult
    weights = np.clip(weights, 0.1, max_weight)
    print(f"\n  MELD class weights (clamped ≤{max_weight}):")
    for i, emo in enumerate(EMOTION_CLASSES):
        print(f"    {emo:<10}  count={int(counts[i]):>5}  "
              f"weight={weights[i]:.3f}")
    return torch.tensor(weights, dtype=torch.float32)


def make_balanced_sampler(dataset):
    labels   = [s['emotion'].item()
                 for s in dataset.samples]
    counts   = np.bincount(
        labels,
        minlength=CFG1.NUM_EMOTIONS).astype(float)

    # FIX: Cap weight for classes with count < MIN_SAMPLES.
    # If peaceful=1, weight=×4710 — that one sample floods every batch.
    # Training on 1 repeated example = pure noise, not learning.
    # Instead: treat count<5 as zero → rely on Phase 1 weights only.
    MIN_SAMPLES = 5
    adjusted = counts.copy()
    for i in range(CFG1.NUM_EMOTIONS):
        if 0 < adjusted[i] < MIN_SAMPLES:
            print(f"  ⚠ '{EMOTION_CLASSES[i]}' has only "
                  f"{int(adjusted[i])} samples — "
                  f"excluded from sampler (Phase 1 weights preserved)")
            adjusted[i] = 0  # exclude from sampling

    adjusted = np.where(adjusted == 0, np.inf, adjusted)
    class_w  = 1.0 / adjusted
    class_w  = np.where(np.isinf(class_w), 0.0, class_w)

    sample_w = torch.tensor(
        [class_w[l] for l in labels],
        dtype=torch.float32)
    sampler  = WeightedRandomSampler(
        sample_w, len(sample_w), replacement=True)

    print(f"\n  Balanced sampler: "
          f"{len(sampler):,} samples/epoch")
    base = class_w[class_w > 0].min() if (class_w > 0).any() else 1.0
    for i, emo in enumerate(EMOTION_CLASSES):
        if class_w[i] == 0:
            print(f"    {emo:<10}  count={int(counts[i]):>4}  "
                  f"EXCLUDED (Phase 1 weights only)")
        else:
            print(f"    {emo:<10}  count={int(counts[i]):>4}  "
                  f"×{class_w[i]/base:.1f}")
    return sampler


def make_warmup_cosine_scheduler(optimizer,
                                   num_warmup_steps,
                                   num_total_steps):
    def lr_lambda(step):
        if step < num_warmup_steps:
            return float(step) / max(1, num_warmup_steps)
        progress = float(step - num_warmup_steps) / max(
            1, num_total_steps - num_warmup_steps)
        return max(0.0,
                   0.5 * (1.0 + math.cos(
                       math.pi * progress)))
    return optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Train / Eval Functions                                          │
# └──────────────────────────────────────────────────────────────────────────┘

def train_one_epoch(model, loader, loss_fn,
                    optimizer, epoch, total,
                    phase_name='',
                    batch_scheduler=None):
    model.train()
    total_loss = 0.0
    all_t, all_p = [], []
    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=(f"{phase_name} Ep {epoch+1:02d}/{total}"
              f" ▶ Train"),
        leave=True,
        bar_format="{l_bar}{bar:25}{r_bar}")

    for step, batch in enumerate(pbar):
        ids  = batch['token_ids'].to(DEVICE)
        spk  = batch['speaker_id'].to(DEVICE)
        trn  = batch['turn_pos'].to(DEVICE)
        hist = batch['utt_history'].to(DEVICE)
        emo  = batch['emotion'].to(DEVICE)

        logits = model(ids, spk, trn, hist)
        loss   = loss_fn.utterance_loss(logits, emo)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0)
        optimizer.step()
        if batch_scheduler is not None:
            batch_scheduler.step()

        total_loss += loss.item()
        all_t.extend(emo.cpu().numpy().tolist())
        all_p.extend(
            logits.argmax(-1).cpu().numpy().tolist())

        if (step + 1) % 20 == 0:
            f1 = f1_score(all_t, all_p,
                           average='macro',
                           zero_division=0) * 100
            pbar.set_postfix(
                loss=f"{total_loss/(step+1):.3f}",
                f1=f"{f1:.1f}%")

    elapsed  = time.time() - t0
    f1_final = f1_score(all_t, all_p,
                         average='macro',
                         zero_division=0)
    avg_loss = total_loss / len(loader)
    m, s     = divmod(int(elapsed), 60)
    print(f"    ↳ TRAIN  loss={avg_loss:.4f}  "
          f"macro_f1={f1_final*100:.2f}%  "
          f"({m}m{s:02d}s)")
    return avg_loss, f1_final


def eval_model(model, loader, loss_fn, split='val'):
    model.eval()
    total_loss = 0.0
    all_t, all_p = [], []

    with torch.no_grad():
        for batch in loader:
            ids  = batch['token_ids'].to(DEVICE)
            spk  = batch['speaker_id'].to(DEVICE)
            trn  = batch['turn_pos'].to(DEVICE)
            hist = batch['utt_history'].to(DEVICE)
            emo  = batch['emotion'].to(DEVICE)
            logits = model(ids, spk, trn, hist)
            loss   = loss_fn.utterance_loss(logits, emo)
            total_loss += loss.item()
            all_t.extend(emo.cpu().numpy().tolist())
            all_p.extend(
                logits.argmax(-1).cpu().numpy().tolist())

    f1   = f1_score(all_t, all_p,
                     average='macro', zero_division=0)
    acc  = accuracy_score(all_t, all_p)
    loss = total_loss / len(loader)
    print(f"    ↳ {split.upper():12s}  "
          f"loss={loss:.4f}  "
          f"macro_f1={f1*100:.2f}%  "
          f"acc={acc*100:.2f}%")
    return loss, f1, all_t, all_p


def calibrate_thresholds(model, val_loader,
                          num_classes=7):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['token_ids'].to(DEVICE)
            spk  = batch['speaker_id'].to(DEVICE)
            trn  = batch['turn_pos'].to(DEVICE)
            hist = batch['utt_history'].to(DEVICE)
            probs= torch.softmax(
                model(ids, spk, trn, hist),
                dim=-1)
            all_probs.extend(
                probs.cpu().numpy().tolist())
            all_labels.extend(
                batch['emotion'].numpy().tolist())

    probs_arr  = np.array(all_probs)
    labels_arr = np.array(all_labels)
    thresholds = np.full(num_classes, 0.5)

    for c in range(num_classes):
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.05, 0.90, 0.05):
            preds = np.where(
                probs_arr[:, c] >= t,
                c, probs_arr.argmax(axis=1))
            mask  = ((labels_arr == c) |
                      (preds == c))
            if mask.sum() == 0:
                continue
            f1 = f1_score(
                labels_arr[mask], preds[mask],
                labels=[c], average='macro',
                zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t

    print(f"\n  Calibrated thresholds (MELD dev):")
    for i, emo in enumerate(EMOTION_CLASSES):
        d = '↓' if thresholds[i] < 0.5 else '↑'
        print(f"    {emo:<10}  {thresholds[i]:.2f} {d}")
    return thresholds


def predict_with_thresholds(model, loader,
                              thresholds):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['token_ids'].to(DEVICE)
            spk  = batch['speaker_id'].to(DEVICE)
            trn  = batch['turn_pos'].to(DEVICE)
            hist = batch['utt_history'].to(DEVICE)
            probs= torch.softmax(
                model(ids, spk, trn, hist),
                dim=-1).cpu().numpy()
            for row in probs:
                adjusted = row / (thresholds + 1e-8)
                all_preds.append(
                    int(np.argmax(adjusted)))
            all_labels.extend(
                batch['emotion'].numpy().tolist())
    return all_labels, all_preds


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Plot Utilities                                                  │
# └──────────────────────────────────────────────────────────────────────────┘

DARK_BG  = '#0f0f1a'
PANEL_BG = '#1a1a2e'
ACCENT   = ['#2ecc71','#e74c3c','#f39c12',
            '#3498db','#9b59b6','#1abc9c','#e67e22']

def _sax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors='#cccccc', labelsize=8)
    ax.set_title(title, color='white', fontsize=10,
                 fontweight='bold', pad=6)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#333355')


def plot_training_curves(history):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(
        'Module 1 v7 — Phase 2 MELD Fine-tuning',
        color='white', fontsize=11, fontweight='bold')
    ep_tr = [h['epoch'] for h in history['train']]
    ep_va = [h['epoch'] for h in history['val']]
    axes[0].plot(ep_tr,
                  [h['loss'] for h in history['train']],
                  color='#3498db', lw=2, marker='o',
                  ms=3, label='Train')
    axes[0].plot(ep_va,
                  [h['loss'] for h in history['val']],
                  color='#e74c3c', lw=2, marker='s',
                  ms=3, ls='--', label='Val')
    axes[0].legend(fontsize=8, facecolor=PANEL_BG,
                    labelcolor='white')
    _sax(axes[0], 'Loss', 'Epoch', 'Loss')
    axes[1].plot(ep_tr,
                  [h['f1']*100 for h in history['train']],
                  color='#2ecc71', lw=2, marker='o',
                  ms=3, label='Train')
    axes[1].plot(ep_va,
                  [h['f1']*100 for h in history['val']],
                  color='#f39c12', lw=2, marker='s',
                  ms=3, ls='--', label='Val (MELD dev)')
    axes[1].legend(fontsize=8, facecolor=PANEL_BG,
                    labelcolor='white')
    axes[1].set_ylim(0, 100)
    _sax(axes[1], 'Macro F1', 'Epoch', 'Macro F1 (%)')
    plt.tight_layout()
    p = f"{PLOT_DIR}/p2_01_training_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight',
                facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ p2_01_training_curves.png")


def plot_confusion_matrix(y_true, y_pred,
                           title, filename):
    cm     = confusion_matrix(
        y_true, y_pred, labels=list(range(7)))
    cm_pct = cm.astype(float) / (
        cm.sum(axis=1, keepdims=True) + 1e-6)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(title, color='white',
                 fontsize=11, fontweight='bold')
    for ax, data, fmt, t in zip(
            axes, [cm, cm_pct],
            ['d', '.2f'],
            ['Count', 'Row-normalised']):
        sns.heatmap(
            data, annot=True, fmt=fmt,
            xticklabels=EMOTION_CLASSES,
            yticklabels=EMOTION_CLASSES,
            cmap='Blues', ax=ax,
            linewidths=0.5, linecolor='#333355')
        ax.set_facecolor(PANEL_BG)
        ax.set_title(t, color='white',
                      fontsize=9, fontweight='bold')
        ax.set_xlabel('Predicted', color='#aaaaaa')
        ax.set_ylabel('True', color='#aaaaaa')
        ax.tick_params(colors='#cccccc')
    plt.tight_layout()
    p = f"{PLOT_DIR}/{filename}"
    plt.savefig(p, dpi=150, bbox_inches='tight',
                facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ {filename}")


def plot_per_class_f1(y_true, y_pred,
                       title, filename):
    f1s   = f1_score(y_true, y_pred, average=None,
                      zero_division=0,
                      labels=list(range(7)))
    macro = np.mean(f1s)
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK_BG)
    bars = ax.bar(EMOTION_CLASSES, f1s*100,
                   color=ACCENT, alpha=0.85,
                   edgecolor='white', linewidth=0.5)
    ax.axhline(macro*100, color='white',
                ls='--', lw=1.2,
                label=f'Macro={macro*100:.2f}%')
    for bar, v in zip(bars, f1s):
        ax.text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f'{v*100:.1f}%',
                 ha='center', va='bottom',
                 color='white', fontsize=8,
                 fontweight='bold')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=9, facecolor=PANEL_BG,
               labelcolor='white')
    _sax(ax, title, 'Emotion', 'F1 (%)')
    plt.tight_layout()
    p = f"{PLOT_DIR}/{filename}"
    plt.savefig(p, dpi=150, bbox_inches='tight',
                facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ {filename}")


def plot_meld_distribution(train_df, dev_df, test_df):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('MELD Emotion Distribution — v7',
                  color='white', fontsize=11,
                  fontweight='bold')
    for ax, df, title in zip(
            axes,
            [train_df, dev_df, test_df],
            ['Train', 'Dev', 'Test']):
        mapped = [MELD_MERGE.get(
            str(e).lower(), 'neutral')
            for e in df['Emotion'].tolist()]
        cnts = Counter(mapped)
        vals = [cnts.get(e, 0) for e in EMOTION_CLASSES]
        ax.bar(EMOTION_CLASSES, vals, color=ACCENT,
                alpha=0.85, edgecolor='white',
                linewidth=0.5)
        total = sum(vals)
        for j, (bar, v) in enumerate(
                zip(ax.patches, vals)):
            ax.text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 5,
                     f'{v/total*100:.0f}%',
                     ha='center', va='bottom',
                     color='white', fontsize=7)
        ax.tick_params(axis='x', rotation=30)
        _sax(ax, f'MELD {title} ({total:,})',
              'Emotion', 'Count')
    plt.tight_layout()
    p = f"{PLOT_DIR}/p2_00_meld_distribution.png"
    plt.savefig(p, dpi=150, bbox_inches='tight',
                facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ p2_00_meld_distribution.png")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Load Data                                                      │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "="*60)
print("  LOADING MELD")
print("="*60)

train_df = pd.read_csv(CFG1.MELD_TRAIN)
dev_df   = pd.read_csv(CFG1.MELD_DEV)
test_df  = pd.read_csv(CFG1.MELD_TEST)

print(f"  MELD  train:{len(train_df):,}  "
      f"dev:{len(dev_df):,}  test:{len(test_df):,}")

plot_meld_distribution(train_df, dev_df, test_df)

# ── Load vocab from Phase 1 checkpoint ────────────────────────────────────
# We MUST use the same vocab Phase 1 was trained with.
# Token IDs must map to the same embedding rows — any mismatch
# means the pretrained weights load into wrong embedding slots.
#
# Try order:
#   1. module1_phase1_best.pt  (full dict — has 'vocab' key)
#   2. phase1_pretrain.pt      (weights-only from v6 save — no vocab key)
#   3. Rebuild from MELD text  (last resort — emits warning)
print(f"\n  Loading vocab from Phase 1 …")
vocab = None
for _ckpt_path, _label in [
        (CFG1.PHASE1_BEST, "module1_phase1_best.pt"),
        (CFG1.PHASE1_CKPT, "phase1_pretrain.pt"),
]:
    if not Path(_ckpt_path).exists():
        print(f"  – {_label} not found, trying next …")
        continue
    _tmp = torch.load(_ckpt_path, map_location='cpu')
    if isinstance(_tmp, dict) and 'vocab' in _tmp:
        vocab = _tmp['vocab']
        print(f"  ✓ Vocab loaded from {_label}: "
              f"{len(vocab):,} tokens")
        break
    else:
        print(f"  – {_label} is weights-only (no vocab key)")

if vocab is None:
    # FIX: Build vocab to exactly VOCAB_SIZE=20,000 slots
    # even though MELD only has ~6,822 unique words.
    #
    # WHY THIS WORKS:
    #   Phase 1 tok_embed shape = (20000, 128)
    #   If we build vocab with max_size=20000, shape = (20000, 128)
    #   → shapes match → all 6,822 MELD words get their Phase 1
    #     embedding values transferred correctly.
    #   Rows 6823–19999 are unused padding (MELD never sees those tokens).
    #
    # WITHOUT THIS FIX (what just ran):
    #   vocab size = 6,822 → shape = (6822, 128)
    #   Shape mismatch → tok_embed reinitialised from scratch
    #   Phase 1's 75% F1 completely wasted, training from random init
    #   That's why epoch 1 started at F1=8% (random level)
    print(f"  ⚠ No vocab key in checkpoints — rebuilding to 20k slots")
    print(f"    Building to VOCAB_SIZE={CFG1.VOCAB_SIZE:,} to match "
          f"Phase 1 tok_embed shape ({CFG1.VOCAB_SIZE},128)")
    all_texts = (train_df['Utterance'].tolist() +
                 dev_df['Utterance'].tolist() +
                 test_df['Utterance'].tolist())
    vocab = build_vocab(all_texts, CFG1.VOCAB_SIZE)  # always 20k slots
    actual_unique = len([k for k in vocab if k not in ('<PAD>','<UNK>')])
    print(f"  Vocab built: {len(vocab):,} slots  "
          f"({actual_unique:,} real MELD words + padding)")
    print(f"  ✓ Shape will match Phase 1: ({len(vocab)},128)")

speaker2id = build_speaker_encoder(
    [train_df, dev_df, test_df])
# Clamp to MAX_SPEAKERS — just in case
if len(speaker2id) >= CFG1.MAX_SPEAKERS:
    print(f"  ⚠ {len(speaker2id)} speakers — "
          f"clamping to {CFG1.MAX_SPEAKERS}")

print(f"  Unique speakers: {len(speaker2id):,}")
print(f"  Speakers: "
      f"{sorted(speaker2id.keys())[:10]} ...")

# Build MELD datasets (all with real turn/history/speaker)
meld_train_ds = MELDDataset(
    train_df, vocab, speaker2id)
meld_dev_ds   = MELDDataset(
    dev_df,   vocab, speaker2id)
meld_test_ds  = MELDDataset(
    test_df,  vocab, speaker2id)

# Balanced sampler for training (handles 47% neutral)
meld_sampler   = make_balanced_sampler(meld_train_ds)
_kw = dict(collate_fn=collate_utt,
           num_workers=2, pin_memory=True)
train_loader = DataLoader(
    meld_train_ds,
    batch_size  = CFG1.BATCH_SIZE,
    sampler     = meld_sampler, **_kw)
dev_loader   = DataLoader(
    meld_dev_ds,
    batch_size  = CFG1.BATCH_SIZE,
    shuffle     = False, **_kw)
test_loader  = DataLoader(
    meld_test_ds,
    batch_size  = CFG1.BATCH_SIZE,
    shuffle     = False, **_kw)

print(f"\n  DataLoaders ready ✓")
print(f"  Train batches : {len(train_loader)}")
print(f"  Dev   batches : {len(dev_loader)}")
print(f"  Test  batches : {len(test_loader)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — Build Model + Load Phase 1 Weights                             │
# │                                                                          │
# │  IMPORTANT: We build the model with MAX_SPEAKERS=500 for MELD.          │
# │  Phase 1 used MAX_SPEAKERS=2 (only UNKNOWN).                            │
# │                                                                          │
# │  The speaker embedding layer is the ONLY mismatched layer.              │
# │  Strategy: load all weights except spk_embed, then re-initialise it.   │
# │  All other 7.6M params transfer perfectly.                              │
# └──────────────────────────────────────────────────────────────────────────┘

utt_encoder = DialogueTransformerEncoder(
    vocab_size  = len(vocab),
).to(DEVICE)

stage_encoder = StageDirectionEncoder(
    vocab_size      = len(vocab),
    embed_dim       = CFG1.EMBED_DIM,
    hidden_dim      = CFG1.D_MODEL // 2,
    num_scene_types = CFG1.NUM_SCENE_TYPES,
    dropout         = CFG1.DROPOUT,
).to(DEVICE)

total_params = sum(
    p.numel() for p in utt_encoder.parameters())
print(f"\n  DialogueTransformerEncoder: "
      f"{total_params:,} params")

# ── Load Phase 1 weights (partial — skip spk_embed) ──────────────────────
print(f"\n  Loading Phase 1 weights from:")
print(f"  {CFG1.PHASE1_CKPT}")

p1_state = torch.load(
    CFG1.PHASE1_CKPT, map_location=DEVICE)

# p1_state may be a plain state_dict or a dict with 'utt_encoder' key
if isinstance(p1_state, dict) and 'utt_encoder' in p1_state:
    p1_state = p1_state['utt_encoder']

current_state = utt_encoder.state_dict()
loaded, skipped = 0, 0

for key, val in p1_state.items():
    if key not in current_state:
        skipped += 1
        continue
    if current_state[key].shape != val.shape:
        # Shape mismatch — re-initialise this layer
        print(f"  ⚠ Shape mismatch '{key}': "
              f"Phase1={tuple(val.shape)} "
              f"Phase2={tuple(current_state[key].shape)}"
              f" — re-initialising")
        skipped += 1
        continue
    current_state[key] = val
    loaded += 1

utt_encoder.load_state_dict(current_state)
print(f"  ✓ Loaded {loaded} weight tensors")
print(f"  ⟳ Re-initialised {skipped} tensors "
      f"(shape mismatch — expected for spk_embed)")

# Re-initialise speaker embedding with small normal noise
# (not zeros — zero init causes all speakers to look identical
#  in epoch 1, slowing down speaker-specific learning)
nn.init.normal_(utt_encoder.spk_embed.weight, std=0.02)
print(f"  ✓ spk_embed re-initialised (std=0.02)")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — Phase 2 Training                                               │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  PHASE 2 — MELD FINE-TUNING")
print(f"  {len(meld_train_ds):,} train utterances")
print(f"  {CFG1.EPOCHS} epochs  LR={CFG1.LR}")
print(f"  turn_pos + utt_history + speaker_id ACTIVE")
print(f"  NeutralPenaltyLoss ACTIVE (×1.8 lazy-neutral)")
print("═"*60)

m_labels  = [s['emotion'].item()
              for s in meld_train_ds.samples]
m_weights = compute_class_weights(
    m_labels, CFG1.NUM_EMOTIONS,
    max_weight=15.0,
    extra_boost={'scared': 1.5,
                  'powerful': 1.5}
).to(DEVICE)

loss_fn = Phase2Loss(
    emotion_weights = m_weights,
    neutral_penalty = 1.8,
    label_smoothing = 0.1,
)

optimizer = optim.Adam(
    list(utt_encoder.parameters()) +
    list(stage_encoder.parameters()),
    lr=CFG1.LR, weight_decay=1e-5)

# Warmup + cosine (MELD is small → warmup critical)
_total_steps  = CFG1.EPOCHS * len(train_loader)
_warmup_steps = max(1, int(0.05 * _total_steps))
scheduler = make_warmup_cosine_scheduler(
    optimizer, _warmup_steps, _total_steps)
print(f"\n  Scheduler: {_warmup_steps} warmup steps "
      f"→ cosine over {_total_steps} total")

history    = {'train': [], 'val': []}
best_f1    = 0.0
patience_c = 0

for epoch in range(CFG1.EPOCHS):
    tl, tf = train_one_epoch(
        utt_encoder, train_loader, loss_fn,
        optimizer, epoch, CFG1.EPOCHS,
        phase_name='[MELD]',
        batch_scheduler=scheduler)
    vl, vf, _, _ = eval_model(
        utt_encoder, dev_loader,
        loss_fn, 'MELD-Dev')

    history['train'].append(
        {'epoch': epoch+1, 'loss': tl, 'f1': tf})
    history['val'].append(
        {'epoch': epoch+1, 'loss': vl, 'f1': vf})

    if vf > best_f1:
        best_f1    = vf
        patience_c = 0
        torch.save({
            # ── clearly named outputs ────────────────
            'utt_encoder':   utt_encoder.state_dict(),
            'stage_encoder': stage_encoder.state_dict(),
            'macro_f1':      best_f1,
            'vocab':         vocab,
            'speaker2id':    speaker2id,
            'config': {
                'vocab_size':    len(vocab),
                'd_model':       CFG1.D_MODEL,
                'num_heads':     CFG1.NUM_HEADS,
                'num_layers':    CFG1.NUM_LAYERS,
                'd_ff':          CFG1.D_FF,
                'embed_dim':     CFG1.EMBED_DIM,
                'speaker_dim':   CFG1.SPEAKER_DIM,
                'turn_dim':      CFG1.TURN_DIM,
                'max_turns':     CFG1.MAX_TURNS,
                'num_speakers':  CFG1.MAX_SPEAKERS,
                'num_emotions':  CFG1.NUM_EMOTIONS,
                'num_prev':      CFG1.NUM_PREV_UTT,
                'dropout':       CFG1.DROPOUT,
            },
        }, CFG1.PHASE2_BEST)
        print(f"    ✓ Saved → module1_phase2_best.pt "
              f"(f1={best_f1*100:.2f}%)")
    else:
        patience_c += 1
        if patience_c >= CFG1.EARLY_STOP:
            print(f"\n  ⏹ Early stop at epoch "
                  f"{epoch+1}")
            break

plot_training_curves(history)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Final Evaluation on MELD Test                                  │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  FINAL EVALUATION — MELD TEST SET")
print("═"*60)

ckpt = torch.load(CFG1.PHASE2_BEST,
                   map_location=DEVICE)
utt_encoder.load_state_dict(ckpt['utt_encoder'])
utt_encoder.eval()

_, test_f1, yt, yp = eval_model(
    utt_encoder, test_loader,
    loss_fn, 'MELD-TEST (argmax)')

# Threshold calibration on dev set
print("\n  Calibrating thresholds on MELD dev …")
thresholds = calibrate_thresholds(
    utt_encoder, dev_loader)

yt_cal, yp_cal = predict_with_thresholds(
    utt_encoder, test_loader, thresholds)
cal_f1 = f1_score(yt_cal, yp_cal,
                   average='macro', zero_division=0)

print(f"\n  F1 comparison:")
print(f"    Argmax        : {test_f1*100:.2f}%")
print(f"    Calibrated    : {cal_f1*100:.2f}%")

if cal_f1 > test_f1:
    print(f"  → Using calibrated "
          f"(+{(cal_f1-test_f1)*100:.2f}%)")
    yt, yp   = yt_cal, yp_cal
    test_f1  = cal_f1
    np.save(CFG1.THRESHOLDS, thresholds)
    print(f"  ✓ Saved → decision_thresholds.npy")

print(f"\n  Classification Report — MELD Test:")
# FIX: labels=list(range(7)) ensures all 7 classes appear in report
# even if the model never predicted one of them (e.g. peaceful).
# Without this, sklearn raises ValueError when predictions skip a class.
print(classification_report(
    yt, yp,
    labels       = list(range(CFG1.NUM_EMOTIONS)),
    target_names = EMOTION_CLASSES,
    zero_division= 0))

# Plots
plot_confusion_matrix(
    yt, yp,
    'Confusion Matrix — MELD Test (Phase 2)',
    'p2_02_confusion_matrix.png')
plot_per_class_f1(
    yt, yp,
    f'Per-Class F1 — MELD Test '
    f'(Macro={test_f1*100:.2f}%)',
    'p2_03_per_class_f1.png')


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 14 — Extract Scene Emotion Matrices → Module 2                      │
# │                                                                          │
# │  Groups utterances by (Season, Episode, Dialogue_ID) — real scenes.    │
# │  Each scene = one Friends dialogue (avg 7 utterances).                  │
# │                                                                          │
# │  This replaces v6's pseudo-scenes (fixed window=10 random sentences).   │
# │  Module 2 now receives genuine emotional arcs with:                     │
# │    - Real emotion progressions (calm → tense → explosion → relief)      │
# │    - Real speaker patterns (who speaks when)                             │
# │    - Real scene lengths (3–30 utterances, not fixed 10)                 │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═"*60)
print("  EXTRACTING SCENE EMOTION MATRICES → MODULE 2")
print("  Grouping by (Season, Episode, Dialogue_ID)")
print("═"*60)

utt_encoder.eval()

# Pool all splits into scene groups
all_samples_by_scene = defaultdict(list)
for ds in [meld_train_ds, meld_dev_ds, meld_test_ds]:
    for s in ds.samples:
        all_samples_by_scene[s['scene_id']].append(s)

scene_data = []

for scene_id in tqdm(
        sorted(all_samples_by_scene.keys()),
        desc="  Extracting scenes"):

    utts = sorted(
        all_samples_by_scene[scene_id],
        key=lambda x: x['position'])

    if not utts:
        continue

    ids_b  = torch.stack(
        [u['token_ids']   for u in utts]).to(DEVICE)
    spk_b  = torch.stack(
        [u['speaker_id']  for u in utts]).to(DEVICE)
    trn_b  = torch.stack(
        [u['turn_pos']    for u in utts]).to(DEVICE)
    hist_b = torch.stack(
        [u['utt_history'] for u in utts]).to(DEVICE)

    with torch.no_grad():
        _, vectors, _ = utt_encoder(
            ids_b, spk_b, trn_b, hist_b,
            return_vector=True)

    scene_data.append({
        'scene_id':       scene_id,
        'emotion_matrix': vectors.cpu(),  # (U, 256)
        'labels':         [u['emotion'].item()
                            for u in utts],
        'speakers':       [u['speaker']
                            for u in utts],
        'texts':          [u['text']
                            for u in utts],
        'positions':      [u['position']
                            for u in utts],
    })

torch.save(scene_data, CFG1.SCENE_OUT)

# Save label encoder
label_enc = {
    'LABEL2IDX':        {e: i for i, e in
                          enumerate(EMOTION_CLASSES)},
    'IDX2LABEL':        dict(enumerate(EMOTION_CLASSES)),
    'CLASS_NAMES':      EMOTION_CLASSES,
    'SCENE_TYPE_NAMES': SCENE_TYPE_NAMES,
    'SPEAKER2ID':       speaker2id,
}
with open(CFG1.LABEL_ENC, 'wb') as f:
    pickle.dump(label_enc, f)

with open(f"{CFG1.CKPT_DIR}/m1_p2_history.json",
          'w') as f:
    json.dump(history, f, indent=2)

# Scene length distribution
lengths = [len(s['labels']) for s in scene_data]
print(f"\n  ✓ {len(scene_data):,} real scenes extracted")
print(f"  Scene length — "
      f"min:{min(lengths)}  "
      f"max:{max(lengths)}  "
      f"avg:{np.mean(lengths):.1f}  "
      f"median:{int(np.median(lengths))}")

# Checkpoint file summary
print(f"\n  ✓ module1_phase2_best.pt  "
      f"→ {CFG1.PHASE2_BEST}")
print(f"  ✓ decision_thresholds.npy "
      f"→ {CFG1.THRESHOLDS}")
print(f"  ✓ scene_emotion_matrix.pt "
      f"→ {CFG1.SCENE_OUT}")
print(f"  ✓ label_encoder.pkl       "
      f"→ {CFG1.LABEL_ENC}")

print(f"\n  {'═'*55}")
print(f"  MODULE 1 v7 — PHASE 2 COMPLETE")
print(f"  {'═'*55}")
print(f"  MELD Test Macro F1   : {test_f1*100:.2f}%")
print(f"  Best Dev F1          : {best_f1*100:.2f}%")
print(f"  Real scenes          : {len(scene_data):,}")
print(f"  Phase 2 checkpoint   : module1_phase2_best.pt")
print(f"  {'═'*55}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 15 — Zip Outputs for Download                                       │
# └──────────────────────────────────────────────────────────────────────────┘

import zipfile
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/module1_phase2_outputs.zip"

print("\n" + "═"*60)
print("  ZIPPING PHASE 2 OUTPUTS")
print("="*60)

_dirs = [
    ("/kaggle/working/checkpoints", "checkpoints"),
    ("/kaggle/working/plots",       "plots"),
]

with zipfile.ZipFile(ZIP_PATH, 'w',
                      zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc in _dirs:
        for fpath in Path(src_dir).rglob('*'):
            if fpath.is_file():
                arcname = (arc + '/'
                           + str(fpath.relative_to(
                               src_dir)))
                zf.write(str(fpath), arcname)
                print(f"  + {arcname}  "
                      f"({fpath.stat().st_size/1024:.1f} KB)")

zip_mb = Path(ZIP_PATH).stat().st_size / (1024*1024)
print(f"\n  ✓ module1_phase2_outputs.zip  ({zip_mb:.1f} MB)")
print(f"\n  Contents:")
print(f"    checkpoints/module1_phase2_best.pt")
print(f"    checkpoints/decision_thresholds.npy")
print(f"    checkpoints/label_encoder.pkl")
print(f"    checkpoints/m1_p2_history.json")
print(f"    checkpoints/final_emotion_vectors/"
      f"scene_emotion_matrix.pt")
print(f"    plots/p2_*.png  (4 plots)")

display(FileLink(ZIP_PATH,
                  result_html_prefix="⬇️  Download: "))

  Device : cuda
  GPU    : Tesla T4
  Config ready ✓
  Phase 1 checkpoint : /kaggle/input/datasets/suyashnpande/module1-phase1/checkpoints/phase1_pretrain.pt
  Phase 2 output     : /kaggle/working/checkpoints/module1_phase2_best.pt
  Emotion classes: ['neutral', 'joyful', 'sad', 'mad', 'scared', 'peaceful', 'powerful']
  MELD mapping: 16 entries

  LOADING MELD
  MELD  train:9,989  dev:1,109  test:2,610
  ✓ p2_00_meld_distribution.png

  Loading vocab from Phase 1 …
  ✓ Vocab loaded from module1_phase1_best.pt: 20,000 tokens
  Unique speakers: 305
  Speakers: ['1st Customer', '2nd Customer', '3rd Customer', 'A Female Student', 'A Student', 'A Waiter', 'Airline Employee', 'Alan', 'Alice', 'All'] ...
  MELD dataset: 9,989 utterances from 1,038 dialogues
  MELD dataset: 1,109 utterances from 114 dialogues
  MELD dataset: 2,610 utterances from 280 dialogues

  Balanced sampler: 9,989 samples/epoch
    neutral     count=4710  ×1.0
    joyful      count=1743  ×2.7
    sad         count= 683 

[MELD] Ep 01/20 ▶ Train:   0%|                         | 0/157 [00:00<?, ?it/s]

    ↳ TRAIN  loss=3.9331  macro_f1=28.26%  (0m04s)
    ↳ MELD-DEV      loss=2.7835  macro_f1=17.62%  acc=16.14%
    ✓ Saved → module1_phase2_best.pt (f1=17.62%)


[MELD] Ep 02/20 ▶ Train:   0%|                         | 0/157 [00:00<?, ?it/s]

    ↳ TRAIN  loss=2.8079  macro_f1=35.95%  (0m04s)
    ↳ MELD-DEV      loss=2.8449  macro_f1=21.00%  acc=23.26%
    ✓ Saved → module1_phase2_best.pt (f1=21.00%)


...

[MELD] Ep 12/20 ▶ Train:   0%|                         | 0/157 [00:00<?, ?it/s]

    ↳ TRAIN  loss=1.7042  macro_f1=64.62%  (0m04s)
    ↳ MELD-DEV      loss=3.3370  macro_f1=23.08%  acc=27.95%

  ⏹ Early stop at epoch 12
  ✓ p2_01_training_curves.png

════════════════════════════════════════════════════════════
  FINAL EVALUATION — MELD TEST SET
════════════════════════════════════════════════════════════
    ↳ MELD-TEST (ARGMAX)  loss=2.7479  macro_f1=20.89%  acc=25.48%

  Calibrating thresholds on MELD dev …

  Calibrated thresholds (MELD dev):
    neutral     0.05 ↓
    joyful      0.20 ↓
    sad         0.20 ↓
    mad         0.15 ↓
    scared      0.40 ↓
    peaceful    0.50 ↑
    powerful    0.30 ↓

  F1 comparison:
    Argmax        : 20.89%
    Calibrated    : 31.95%
  → Using calibrated (+11.06%)
  ✓ Saved → decision_thresholds.npy

  Classification Report — MELD Test:
              precision    recall  f1-score   support

     neutral       0.59      0.44      0.50      1256
      joyful       0.37      0.45      0.40       402
         sad       0.20    

  Extracting scenes:   0%|          | 0/1423 [00:00<?, ?it/s]


  ✓ 1,423 real scenes extracted
  Scene length — min:1  max:33  avg:9.6  median:9

  ✓ module1_phase2_best.pt  → /kaggle/working/checkpoints/module1_phase2_best.pt
  ✓ decision_thresholds.npy → /kaggle/working/checkpoints/decision_thresholds.npy
  ✓ scene_emotion_matrix.pt → /kaggle/working/checkpoints/final_emotion_vectors/scene_emotion_matrix.pt
  ✓ label_encoder.pkl       → /kaggle/working/checkpoints/label_encoder.pkl

  ═══════════════════════════════════════════════════════
  MODULE 1 v7 — PHASE 2 COMPLETE
  ═══════════════════════════════════════════════════════
  MELD Test Macro F1   : 31.95%
  Best Dev F1          : 23.40%
  Real scenes          : 1,423
  Phase 2 checkpoint   : module1_phase2_best.pt
  ═══════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  ZIPPING PHASE 2 OUTPUTS
  + checkpoints/label_encoder.pkl  (4.5 KB)
  + checkpoints/decision_thresholds.npy  (0.2 KB)
  + checkpoints/m1_p2_history.json  (2.3

/kaggle/working/module1_phase2_outputs.zip

In [5]:
import torch
import numpy as np
from collections import Counter

scenes = torch.load(
    '/kaggle/working/checkpoints/final_emotion_vectors/scene_emotion_matrix.pt',
    map_location='cpu')

emo = ['neutral','joyful','sad','mad','scared','peaceful','powerful']
all_labels = []
scene_lengths = []
for s in scenes:
    all_labels.extend(s['labels'])
    scene_lengths.append(len(s['labels']))

cnt = Counter(all_labels)
total = len(all_labels)

print(f"Total scenes    : {len(scenes)}")
print(f"Total utterances: {total}")
print(f"Scene length    : min={min(scene_lengths)}  max={max(scene_lengths)}  avg={np.mean(scene_lengths):.1f}")
print()
print("Label distribution:")
for i, e in enumerate(emo):
    c = cnt.get(i, 0)
    bar = '█' * int(c/total*30)
    print(f"  {e:<10} {c:>5}  {c/total*100:5.1f}%  {bar}")

# Check vector health
all_vecs = torch.cat([s['emotion_matrix'] for s in scenes], dim=0)
print(f"\nVector stats (all {all_vecs.shape[0]} utterance vectors, dim={all_vecs.shape[1]}):")
print(f"  mean : {all_vecs.mean():.4f}")
print(f"  std  : {all_vecs.std():.4f}")
print(f"  min  : {all_vecs.min():.4f}")
print(f"  max  : {all_vecs.max():.4f}")
print(f"  NaN  : {torch.isnan(all_vecs).sum().item()}")
print(f"  Inf  : {torch.isinf(all_vecs).sum().item()}")
print()

# Check a real scene
s = scenes[0]
print(f"Sample scene {s['scene_id']}:")
print(f"  utterances  : {len(s['labels'])}")
print(f"  labels      : {[emo[l] for l in s['labels']]}")
print(f"  speakers    : {s['speakers']}")
print(f"  matrix shape: {s['emotion_matrix'].shape}")

Total scenes    : 1423
Total utterances: 13708
Scene length    : min=1  max=33  avg=9.6

Label distribution:
  neutral     6436   47.0%  ██████████████
  joyful      2308   16.8%  █████
  sad         1002    7.3%  ██
  mad         1968   14.4%  ████
  scared       358    2.6%  
  peaceful       0    0.0%  
  powerful    1636   11.9%  ███

Vector stats (all 13708 utterance vectors, dim=256):
  mean : 0.3987
  std  : 0.5747
  min  : 0.0000
  max  : 3.7323
  NaN  : 0
  Inf  : 0

Sample scene (1, 1, 19):
  utterances  : 3
  labels      : ['neutral', 'powerful', 'neutral']
  speakers    : ['Joey', 'Joey', 'Phoebe']
  matrix shape: torch.Size([3, 256])


In [10]:
print("Module 2 starts here ")

Module 2 starts here 


In [13]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODULE 2 — Scene Encoder V3  (Fixed)                                  ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  ROLE IN THE PIPELINE:                                                  ║
# ║    Module 1 processes one utterance at a time → 256d vector per utt.   ║
# ║    Module 2 takes ALL utterance vectors in a scene together and         ║
# ║    compresses them into ONE 256d scene vector for Module 3.             ║
# ║                                                                         ║
# ║  INPUTS  (from Module 1 scene_emotion_matrix.pt):                      ║
# ║    emotion_matrix : (U, 256)  ← DTE utterance vectors (U = num utts)  ║
# ║    labels         : [int]     ← per-utterance emotion labels 0-6       ║
# ║    speakers       : [str]     ← per-utterance speaker names            ║
# ║                                                                         ║
# ║  ARCHITECTURE:                                                          ║
# ║    Scene Type Conditioning  → 32d embedding added to each utt vector   ║
# ║    Input Projection         : 256+32 → 256                             ║
# ║    2-layer BiLSTM           : processes utterance sequence             ║
# ║    Attention Pooling        → single 256d scene_vector                 ║
# ║    Per-Utterance Heads      → emotion_curve (U,7), intensity (U,1)     ║
# ║    Scene-Level Heads        → 9 prediction targets for Module 3/4      ║
# ║                                                                         ║
# ║  OUTPUTS (saved to scene_vectors.pt → Module 3):                       ║
# ║    scene_vector     : (256,) → Module 3 narrative LSTM                 ║
# ║    emotion_curve    : (U, 7) → Module 4 hit-point logic                ║
# ║    intensity_curve  : (U,)   → Module 4 dynamic shape                  ║
# ║    transition_matrix: (7,7)  → Module 4 harmonic planning              ║
# ║    scene_type       : int    → Module 4 texture selection              ║
# ║    arousal_score    : float  → Module 4 tempo estimation               ║
# ║    conflict_score   : float  → Module 4 dissonance level               ║
# ║    trajectory_score : float  → Module 4 dynamic shape                  ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  FIXES APPLIED VS PRIOR VERSION:                                        ║
# ║    FIX 1 — peaceful class weight clamped to 15.0 (was ~162, crashes)   ║
# ║    FIX 2 — emotion_curve loss weight 0.4→0.2 (labels from 32% F1 M1)  ║
# ║    FIX 3 — BATCH_SIZE 32→16 (only 1,423 scenes; 32 = 31 batches/ep)   ║
# ║    FIX 4 — Input paths updated for Phase 2 checkpoint                  ║
# ║    FIX 5 — SCENE_TYPE_NAMES defined as a constant, not fragile .get()  ║
# ║    FIX 6 — Gradient clipping added to training loop                    ║
# ║    FIX 7 — plot_emotion_curves_sample guarded for post-extraction call ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import os, json, pickle, time
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (f1_score, accuracy_score,
                              classification_report,
                              confusion_matrix)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available()
                         else "cpu")
PLOT_DIR = "/kaggle/working/plots"
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

class CFG2:
    # ── Input: Module 1 Phase 2 outputs ──────────────────────────────────
    # FIX 4: Updated to Phase 2 checkpoint paths.
    # Phase 2 produced scene_emotion_matrix.pt from real MELD dialogues
    # (1,423 scenes, 13,708 utterances).
    # Phase 1 produced pseudo-scenes (5,634 sliding windows) — those are
    # less useful because they are artificial, not real dialogue groups.
    # Use Phase 2 outputs here.
    SCENE_MATRIX = ("/kaggle/input/datasets/suyashnpande/module1-phase2/checkpoints"
                    "/final_emotion_vectors/scene_emotion_matrix.pt")
    M1_LABEL_ENC = ("/kaggle/input/datasets/suyashnpande/module1-phase2/checkpoints"
                    "/label_encoder.pkl")

    # ── Output paths ─────────────────────────────────────────────────────
    CKPT_DIR      = "/kaggle/working/m2_checkpoints"
    SCENE_VEC_OUT = "/kaggle/working/m2_checkpoints/scene_vectors.pt"
    BEST_CKPT     = "/kaggle/working/m2_checkpoints/scene_encoder_best.pt"

    # ── Architecture ─────────────────────────────────────────────────────
    # INPUT_DIM = 256: Module 1 DTE outputs 256d utterance vectors.
    # The old BiLSTM (before v6) output 512d — this MUST be 256 to match.
    INPUT_DIM       = 256
    SCENE_TYPE_DIM  = 32     # scene type conditioning embedding
    HIDDEN_DIM      = 128    # BiLSTM hidden → 256 bidirectional
    SCENE_VEC_DIM   = 256    # final scene vector (sent to Module 3)
    NUM_EMOTIONS    = 7
    NUM_SCENE_TYPES = 6
    NUM_SENTIMENTS  = 4      # positive / negative / neutral / mixed

    # ── Training ─────────────────────────────────────────────────────────
    # FIX 3: BATCH_SIZE reduced 32→16.
    # Dataset has only 1,423 scenes after Phase 2 MELD extraction.
    # 70% train = ~996 scenes. With batch=32 → 31 batches/epoch (too coarse).
    # With batch=16 → 62 batches/epoch → smoother gradient updates.
    BATCH_SIZE  = 16
    LR          = 3e-4    # slightly reduced from 5e-4 for stability
    EPOCHS      = 30      # more epochs to compensate for small dataset
    DROPOUT     = 0.20    # reduced from 0.25 — small dataset needs less reg
    EARLY_STOP  = 8       # more patience — val F1 oscillates on small data
    MAX_WEIGHT  = 15.0    # FIX 1: class weight cap (peaceful was → 162)

    # ── Scene constraints ─────────────────────────────────────────────────
    MAX_SCENE_LEN = 30   # pad/truncate to this length
    MIN_SCENE_LEN = 2    # skip scenes shorter than this

    # ── Split ratios ───────────────────────────────────────────────────────
    VAL_SPLIT  = 0.15
    TEST_SPLIT = 0.15
    # → train ≈ 70% ≈ 996 scenes
    # → val   ≈ 15% ≈ 213 scenes
    # → test  ≈ 15% ≈ 213 scenes

Path(CFG2.CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("  Config ready ✓")
print(f"  INPUT_DIM  = {CFG2.INPUT_DIM}  (DTE 256d output)")
print(f"  BATCH_SIZE = {CFG2.BATCH_SIZE}  "
      f"(~{int(CFG2.BATCH_SIZE):d} scenes/batch)")
print(f"  MAX_WEIGHT = {CFG2.MAX_WEIGHT}  (class weight cap)")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Labels & Scene Types                                            │
# │                                                                          │
# │  FIX 5: SCENE_TYPE_NAMES defined as a module-level constant.            │
# │  Previous version used _le.get('SCENE_TYPE_NAMES', fallback).           │
# │  label_encoder.pkl from Phase 2 does contain SCENE_TYPE_NAMES but       │
# │  we define it explicitly anyway so all downstream code is consistent     │
# │  regardless of whether the pkl key exists.                               │
# └──────────────────────────────────────────────────────────────────────────┘

# Canonical scene type names — order is the integer index
SCENE_TYPE_NAMES = [
    'dialogue',         # 0 — standard back-and-forth conversation
    'action',           # 1 — physical conflict, high-mad utterances
    'investigation',    # 2 — single-speaker monologue/interrogation
    'suspense',         # 3 — high-scared utterances
    'dramatic_reveal',  # 4 — high-powerful utterances
    'transition',       # 5 — mostly neutral, scene change
]

# Emotion groupings used by derive_scene_features()
POS_EMOTIONS = {'joyful', 'peaceful'}
NEG_EMOTIONS = {'sad', 'mad', 'scared'}
HIGH_AROUSAL = {'joyful', 'mad', 'scared', 'powerful'}
LOW_AROUSAL  = {'peaceful', 'neutral', 'sad'}

# Load Module 1 label encoder
with open(CFG2.M1_LABEL_ENC, 'rb') as f:
    _le = pickle.load(f)

LABEL2IDX   = _le['LABEL2IDX']
IDX2LABEL   = _le['IDX2LABEL']
CLASS_NAMES = _le['CLASS_NAMES']
# Verify SCENE_TYPE_NAMES consistency
if 'SCENE_TYPE_NAMES' in _le:
    assert _le['SCENE_TYPE_NAMES'] == SCENE_TYPE_NAMES, (
        f"SCENE_TYPE_NAMES mismatch: pkl={_le['SCENE_TYPE_NAMES']} "
        f"vs module={SCENE_TYPE_NAMES}")

print(f"  Emotion classes : {CLASS_NAMES}")
print(f"  Scene types     : {SCENE_TYPE_NAMES}")
print(f"  LABEL2IDX       : {LABEL2IDX}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Scene Feature Derivation                                        │
# │                                                                          │
# │  These functions compute all supervised training targets from the        │
# │  raw utterance data that Module 1 produced. All targets are derived     │
# │  automatically — Module 2 has no external labels of its own.            │
# │                                                                          │
# │  WHY SELF-SUPERVISED TARGETS ARE OK HERE:                               │
# │    Module 2's job is to compress a variable-length sequence of          │
# │    utterance vectors into a fixed 256d scene vector. The "labels"       │
# │    we derive (dominant emotion, arousal score, etc.) are not the        │
# │    end goal — they are regularisation signals that force the BiLSTM      │
# │    to encode emotionally meaningful features into the scene vector.     │
# │    If dominant emotion F1 is 60%, that's fine — we care more about      │
# │    whether the 256d scene vector captures emotional arc patterns.       │
# └──────────────────────────────────────────────────────────────────────────┘

def attention_weighted_dominant(labels: list, vectors: torch.Tensor) -> int:
    """
    Compute dominant emotion weighted by utterance vector L2 norm.

    WHY: Simple majority vote treats a whispered "fine" equally to a
    screamed "I hate you!". DTE vectors have higher L2 norm for
    emotionally intense utterances (higher activation from attention).
    Weighting by norm gives stronger utterances more influence.

    Args:
        labels  : list of int, length U
        vectors : (U, 256) float tensor
    Returns:
        int — dominant emotion index 0-6
    """
    norms = torch.norm(vectors.float(), dim=1)   # (U,)
    total = norms.sum().item()
    if total < 1e-6:
        w = torch.ones(len(labels)) / len(labels)
    else:
        w = norms / total                         # (U,) normalised weights
    scores = torch.zeros(CFG2.NUM_EMOTIONS)
    for i, lab in enumerate(labels):
        scores[lab] += w[i].item()
    return int(scores.argmax().item())


def compute_transition_matrix(labels: list) -> np.ndarray:
    """
    Build a (7, 7) row-normalised emotion transition matrix.

    Entry [i, j] = probability of emotion j following emotion i
    within this scene. This tells Module 4 what harmonic transitions
    are likely — e.g. if [mad→powerful] is common, expect sudden
    accent chords.

    For scenes where emotion never changes, the diagonal is 1.0.
    """
    M = np.zeros((CFG2.NUM_EMOTIONS, CFG2.NUM_EMOTIONS),
                 dtype=np.float32)
    for i in range(len(labels) - 1):
        M[labels[i]][labels[i+1]] += 1.0
    row_sums = M.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    return (M / row_sums).astype(np.float32)


def compute_intensity_curve(labels: list,
                             vectors: torch.Tensor) -> np.ndarray:
    """
    Per-utterance emotional intensity = L2 norm of DTE vector,
    min-max normalised to [0, 1] within the scene.

    WHY L2 norm: The DTE's EmotionQueryToken + fusion layer outputs
    larger magnitude vectors for emotionally charged utterances
    (many activated attention heads). Neutral "OK" → small norm.
    "Get OUT of my house!" → large norm.

    This is what Module 4 uses to shape musical dynamics over time.
    """
    norms = torch.norm(vectors.float(), dim=1).numpy()
    vmin, vmax = norms.min(), norms.max()
    if vmax - vmin < 1e-6:
        return np.full(len(labels), 0.5, dtype=np.float32)
    return ((norms - vmin) / (vmax - vmin)).astype(np.float32)


def infer_scene_type(labels: list, speakers: list = None) -> int:
    """
    Heuristically classify scene type from emotion pattern.

    Priority order (highest wins):
      1. ≥60% scared  → suspense
      2. ≥50% mad     → action
      3. ≥50% powerful→ dramatic_reveal
      4. ≥70% neutral → transition
      5. single speaker with ≥4 utterances → investigation
      6. default → dialogue

    This is intentionally simple — it only needs to be right ~70%
    of the time to provide useful conditioning for the BiLSTM.
    """
    n      = len(labels)
    counts = Counter(IDX2LABEL[l] for l in labels)

    if counts.get('scared',   0) / n >= 0.60:
        return SCENE_TYPE_NAMES.index('suspense')
    if counts.get('mad',      0) / n >= 0.50:
        return SCENE_TYPE_NAMES.index('action')
    if counts.get('powerful', 0) / n >= 0.50:
        return SCENE_TYPE_NAMES.index('dramatic_reveal')
    if counts.get('neutral',  0) / n >= 0.70:
        return SCENE_TYPE_NAMES.index('transition')
    if (speakers is not None
            and len(set(speakers)) == 1
            and n >= 4):
        return SCENE_TYPE_NAMES.index('investigation')
    return SCENE_TYPE_NAMES.index('dialogue')


def derive_scene_features(labels: list,
                           vectors: torch.Tensor,
                           speakers: list = None,
                           scene_type: int = None) -> dict:
    """
    Derive all supervised training targets from raw utterance data.

    This is the only "label" source Module 2 has. All targets come
    from the emotion labels and vectors that Module 1 produced.

    Returns a dict with:
      dominant_emotion   : int       — most impactful emotion (7-class)
      transition_matrix  : (7,7)     — pairwise emotion transitions
      emotion_distribution: (7,)     — fraction of each emotion
      scene_type         : int       — 6-class scene category
      variance_score     : float     — fraction of turns that change emotion
      conflict_score     : float     — pos vs neg valence balance (-1 to +1)
      arousal_score      : float     — high vs low arousal balance (-1 to +1)
      pacing_score       : float     — scene density (0=short, 1=full MAX_LEN)
      sentiment_label    : int       — 4-class: pos/neg/neutral/mixed
      trajectory_score   : float     — valence change first→last (-1 to +1)
      intensity_curve    : (U,)      — per-utt emotional intensity [0,1]
      emotion_curve      : (U,)      — per-utt dominant emotion index
    """
    n      = len(labels)
    dom    = attention_weighted_dominant(labels, vectors)
    trans  = compute_transition_matrix(labels)
    counts = np.bincount(labels,
                         minlength=CFG2.NUM_EMOTIONS).astype(float)
    dist   = (counts / counts.sum()).astype(np.float32)

    stype  = (scene_type if scene_type is not None
              else infer_scene_type(labels, speakers))

    # Variance: fraction of consecutive turns that change emotion
    variance = (sum(1 for i in range(n - 1)
                    if labels[i] != labels[i + 1])
                / (n - 1)) if n > 1 else 0.0

    # Conflict: how much positive vs negative emotion
    # +1 = all positive, -1 = all negative, 0 = balanced/neutral
    pos_c = sum(1 for l in labels if IDX2LABEL[l] in POS_EMOTIONS)
    neg_c = sum(1 for l in labels if IDX2LABEL[l] in NEG_EMOTIONS)
    conflict = (pos_c - neg_c) / (pos_c + neg_c + 1e-6)

    # Arousal: high-energy vs low-energy emotion balance
    # +1 = all high-arousal, -1 = all low-arousal
    hi_c = sum(1 for l in labels if IDX2LABEL[l] in HIGH_AROUSAL)
    lo_c = sum(1 for l in labels if IDX2LABEL[l] in LOW_AROUSAL)
    arousal = (hi_c - lo_c) / (hi_c + lo_c + 1e-6)

    # Pacing: what fraction of MAX_SCENE_LEN is actually used
    pacing = min(n / CFG2.MAX_SCENE_LEN, 1.0)

    # Sentiment (4-class)
    pos_r, neg_r = pos_c / n, neg_c / n
    if   pos_r > 0.6:          sentiment = 0  # positive
    elif neg_r > 0.6:          sentiment = 1  # negative
    elif pos_r + neg_r < 0.3:  sentiment = 2  # neutral
    else:                       sentiment = 3  # mixed

    # Trajectory: valence arc (does scene end happier/sadder than it starts?)
    def valence(idx):
        e = IDX2LABEL[idx]
        return (1 if e in POS_EMOTIONS else
                -1 if e in NEG_EMOTIONS else 0)

    trajectory = float(valence(labels[-1]) - valence(labels[0])) / 2.0

    return {
        'dominant_emotion':     dom,
        'transition_matrix':    trans,
        'emotion_distribution': dist,
        'scene_type':           stype,
        'variance_score':       float(variance),
        'conflict_score':       float(conflict),
        'arousal_score':        float(arousal),
        'pacing_score':         float(pacing),
        'sentiment_label':      sentiment,
        'trajectory_score':     trajectory,
        'first_emotion':        labels[0],
        'last_emotion':         labels[-1],
        'intensity_curve':      compute_intensity_curve(labels, vectors),
        'emotion_curve':        np.array(labels, dtype=np.int32),
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — Scene Dataset                                                   │
# │                                                                          │
# │  Each training sample = one scene (one MELD dialogue).                  │
# │  Variable-length utterance sequence → padded to MAX_SCENE_LEN.          │
# │  The mask tensor tells the model which positions are real utterances    │
# │  (True) vs padding (False).                                              │
# └──────────────────────────────────────────────────────────────────────────┘

class SceneDatasetV3(Dataset):
    def __init__(self, scene_data: list,
                 max_len: int = CFG2.MAX_SCENE_LEN,
                 min_len: int = CFG2.MIN_SCENE_LEN):
        self.samples = []
        skipped      = 0
        feat_dim     = CFG2.INPUT_DIM

        for scene in tqdm(scene_data,
                          desc="  Building SceneDataset",
                          leave=False):
            labels = scene['labels']
            mat    = scene['emotion_matrix']   # (U, 256)
            U      = mat.shape[0]

            if U < min_len:
                skipped += 1
                continue

            # Safety: if Module 1 output dim differs, fix silently
            if mat.shape[1] != feat_dim:
                if mat.shape[1] > feat_dim:
                    mat = mat[:, :feat_dim]
                else:
                    pad = torch.zeros(U, feat_dim - mat.shape[1])
                    mat = torch.cat([mat, pad], dim=1)

            speakers   = scene.get('speakers', [])
            scene_type = scene.get('scene_type', None)
            feats      = derive_scene_features(
                labels, mat, speakers, scene_type)

            # Pad / truncate sequence to MAX_SCENE_LEN
            if U >= max_len:
                mat_p = mat[:max_len].float()
                mask  = torch.ones(max_len, dtype=torch.bool)
                ec    = feats['emotion_curve'][:max_len]
                ic    = feats['intensity_curve'][:max_len]
            else:
                pad_rows = torch.zeros(max_len - U, feat_dim)
                mat_p    = torch.cat([mat.float(), pad_rows], dim=0)
                mask     = torch.cat([
                    torch.ones(U, dtype=torch.bool),
                    torch.zeros(max_len - U, dtype=torch.bool)])
                ec = np.pad(feats['emotion_curve'],
                            (0, max_len - U), constant_values=0)
                ic = np.pad(feats['intensity_curve'],
                            (0, max_len - U), constant_values=0.0)

            self.samples.append({
                # ── Model inputs ──────────────────────────────────────
                'emotion_matrix':    mat_p,
                'mask':              mask,
                'scene_len':         torch.tensor(
                    min(U, max_len), dtype=torch.long),
                'scene_type_input':  torch.tensor(
                    feats['scene_type'], dtype=torch.long),
                # ── Supervised targets ────────────────────────────────
                'dominant_emotion':  torch.tensor(
                    feats['dominant_emotion'], dtype=torch.long),
                'transition_matrix': torch.tensor(
                    feats['transition_matrix'], dtype=torch.float32),
                'emotion_dist':      torch.tensor(
                    feats['emotion_distribution'], dtype=torch.float32),
                'sentiment_label':   torch.tensor(
                    feats['sentiment_label'], dtype=torch.long),
                'variance_score':    torch.tensor(
                    feats['variance_score'], dtype=torch.float32),
                'conflict_score':    torch.tensor(
                    feats['conflict_score'], dtype=torch.float32),
                'arousal_score':     torch.tensor(
                    feats['arousal_score'], dtype=torch.float32),
                'scene_type_target': torch.tensor(
                    feats['scene_type'], dtype=torch.long),
                'trajectory_score':  torch.tensor(
                    feats['trajectory_score'], dtype=torch.float32),
                'emotion_curve':     torch.tensor(
                    ec, dtype=torch.long),
                'intensity_curve':   torch.tensor(
                    ic, dtype=torch.float32),
                # ── Metadata (not used in training) ──────────────────
                'scene_id':          scene['scene_id'],
                'num_utterances':    U,
                'speakers':          speakers,
                'pacing_score':      torch.tensor(
                    feats['pacing_score'], dtype=torch.float32),
                'first_emotion':     feats['first_emotion'],
                'last_emotion':      feats['last_emotion'],
            })

        print(f"  Scenes built : {len(self.samples):,}  "
              f"(skipped {skipped} too-short scenes)")

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_scenes(batch: list) -> dict:
    """Stack all tensors; metadata (scene_id, speakers) kept as lists."""
    return {
        'emotion_matrix':    torch.stack([b['emotion_matrix']    for b in batch]),
        'mask':              torch.stack([b['mask']              for b in batch]),
        'scene_len':         torch.stack([b['scene_len']         for b in batch]),
        'scene_type_input':  torch.stack([b['scene_type_input']  for b in batch]),
        'dominant_emotion':  torch.stack([b['dominant_emotion']  for b in batch]),
        'transition_matrix': torch.stack([b['transition_matrix'] for b in batch]),
        'emotion_dist':      torch.stack([b['emotion_dist']      for b in batch]),
        'sentiment_label':   torch.stack([b['sentiment_label']   for b in batch]),
        'variance_score':    torch.stack([b['variance_score']    for b in batch]),
        'conflict_score':    torch.stack([b['conflict_score']    for b in batch]),
        'arousal_score':     torch.stack([b['arousal_score']     for b in batch]),
        'scene_type_target': torch.stack([b['scene_type_target'] for b in batch]),
        'trajectory_score':  torch.stack([b['trajectory_score']  for b in batch]),
        'emotion_curve':     torch.stack([b['emotion_curve']     for b in batch]),
        'intensity_curve':   torch.stack([b['intensity_curve']   for b in batch]),
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — Scene Encoder Model V3                                          │
# │                                                                          │
# │  Architecture walk-through:                                              │
# │                                                                          │
# │  INPUT: (B, L, 256)  — batch of padded utterance sequences              │
# │                                                                          │
# │  STEP 1 — Scene type conditioning                                        │
# │    Each scene has a single scene_type label (0-5).                      │
# │    We embed it into 32d and broadcast to every position in the          │
# │    sequence: (B, 32) → (B, L, 32). Then concatenate with the           │
# │    utterance vectors: (B, L, 256+32) = (B, L, 288).                    │
# │    WHY: The same emotion sequence in an "action" vs "suspense"          │
# │    scene should produce different music → conditioning lets the          │
# │    model learn this.                                                     │
# │                                                                          │
# │  STEP 2 — Input projection: 288 → 256                                   │
# │    Linear + LayerNorm + ReLU + Dropout.                                 │
# │                                                                          │
# │  STEP 3 — 2-layer BiLSTM over the utterance sequence                    │
# │    Processes utterances left-to-right AND right-to-left.                │
# │    At each position t, the hidden state encodes:                        │
# │      - Everything that happened before t (forward LSTM)                 │
# │      - Everything that happens after t (backward LSTM)                  │
# │    Output: (B, L, 256)  (128 forward + 128 backward)                   │
# │    WHY BiLSTM and not just forward: the emotional meaning of            │
# │    utterance 3 in a scene depends on what resolves in utterance 8.     │
# │                                                                          │
# │  STEP 4 — Per-utterance heads (run on EVERY LSTM hidden state)          │
# │    emotion_curve_logits : (B, L, 7) → per-utt emotion distribution      │
# │    intensity_curve      : (B, L, 1) → per-utt emotional intensity        │
# │    These are the temporal curves Module 4 uses for music hit points.    │
# │                                                                          │
# │  STEP 5 — Masked attention pooling                                       │
# │    A learnable attention head scores each real utterance (mask=True).   │
# │    Padding positions (mask=False) get -inf before softmax.              │
# │    Weighted sum → single (B, 256) scene_vector.                         │
# │    WHY attention over mean pooling: the emotionally significant          │
# │    utterances (e.g. a dramatic reveal) should dominate the scene        │
# │    vector, not be diluted by surrounding neutral dialogue.              │
# │                                                                          │
# │  STEP 6 — Scene-level prediction heads (all from scene_vector)          │
# │    8 heads predicting the derived targets from Cell 4.                  │
# └──────────────────────────────────────────────────────────────────────────┘

class SceneEncoderV3(nn.Module):
    def __init__(self,
                 input_dim      = CFG2.INPUT_DIM,
                 scene_type_dim = CFG2.SCENE_TYPE_DIM,
                 hidden_dim     = CFG2.HIDDEN_DIM,
                 scene_vec_dim  = CFG2.SCENE_VEC_DIM,
                 num_emotions   = CFG2.NUM_EMOTIONS,
                 num_scene_types= CFG2.NUM_SCENE_TYPES,
                 num_sentiments = CFG2.NUM_SENTIMENTS,
                 dropout        = CFG2.DROPOUT):
        super().__init__()
        self.hidden_dim = hidden_dim
        bilstm_out_dim  = hidden_dim * 2   # 256 (128 × 2 directions)

        # ── Step 1: Scene type conditioning ──────────────────────────────
        self.scene_type_emb = nn.Embedding(
            num_scene_types, scene_type_dim)
        nn.init.normal_(self.scene_type_emb.weight, std=0.02)

        # ── Step 2: Input projection 256+32 → 256 ────────────────────────
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim + scene_type_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # ── Step 3: 2-layer BiLSTM ────────────────────────────────────────
        self.bilstm = nn.LSTM(
            input_size    = 256,
            hidden_size   = hidden_dim,
            num_layers    = 2,
            batch_first   = True,
            bidirectional = True,
            dropout       = dropout,
        )

        # ── Step 4: Per-utterance temporal curve heads ────────────────────
        # Applied to EVERY LSTM hidden state before pooling
        self.per_utt_emo_head = nn.Linear(
            bilstm_out_dim, num_emotions)
        # → (B, L, 7) emotion logits per utterance

        self.per_utt_int_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid())
        # → (B, L, 1) intensity in [0,1] per utterance

        # ── Step 5: Masked attention pooling → scene_vector ───────────────
        self.attn_fc   = nn.Linear(bilstm_out_dim, 1)
        self.scene_ln  = nn.LayerNorm(bilstm_out_dim)
        self.scene_drop= nn.Dropout(dropout)

        # ── Step 6: Scene-level prediction heads ─────────────────────────
        # Dominant emotion (7-class classification)
        self.emotion_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_emotions))

        # Emotion transition matrix (7×7, row-softmaxed)
        self.transition_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_emotions * num_emotions))

        # Scene type verification (6-class)
        self.scene_type_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_scene_types))

        # Sentiment (4-class: positive/negative/neutral/mixed)
        self.sentiment_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_sentiments))

        # Continuous regression heads (all output in [-1,1] or [0,1])
        self.variance_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        self.conflict_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())

        self.arousal_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())

        self.trajectory_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())

    def forward(self,
                emotion_matrix,    # (B, L, 256)
                mask,              # (B, L) — True = real utterance
                scene_type_ids=None) -> dict:

        B, L, _ = emotion_matrix.shape

        # Step 1: scene type embedding broadcast over sequence
        if scene_type_ids is not None:
            st_e = self.scene_type_emb(scene_type_ids)  # (B, 32)
        else:
            st_e = torch.zeros(
                B, CFG2.SCENE_TYPE_DIM,
                device=emotion_matrix.device)
        st_e = st_e.unsqueeze(1).expand(-1, L, -1)      # (B, L, 32)

        # Step 2: project to 256
        x = self.input_proj(
            torch.cat([emotion_matrix, st_e], dim=-1))  # (B, L, 256)

        # Step 3: BiLSTM
        h, _ = self.bilstm(x)                           # (B, L, 256)

        # Step 4: per-utterance heads
        per_utt_emo = self.per_utt_emo_head(h)          # (B, L, 7)
        per_utt_int = self.per_utt_int_head(
            h).squeeze(-1)                               # (B, L)

        # Step 5: masked attention pooling
        scores = self.attn_fc(h).squeeze(-1)             # (B, L)
        scores = scores.masked_fill(~mask, float('-inf'))
        attn_w = torch.softmax(scores, dim=1)
        attn_w = torch.nan_to_num(attn_w, nan=0.0)      # guard all-pad
        scene_v = torch.bmm(
            attn_w.unsqueeze(1), h).squeeze(1)           # (B, 256)
        scene_v = self.scene_drop(self.scene_ln(scene_v))

        # Transition matrix: predict (B, 49) → reshape → row-softmax
        trans_flat = self.transition_head(scene_v)
        trans_mat  = trans_flat.view(
            B, CFG2.NUM_EMOTIONS, CFG2.NUM_EMOTIONS)
        trans_mat  = torch.softmax(trans_mat, dim=-1)

        # Step 6: scene-level heads
        return {
            # Primary output to Module 3
            'scene_vector':          scene_v,           # (B, 256)
            # Classification heads
            'emotion_logits':        self.emotion_head(scene_v),
            'scene_type_logits':     self.scene_type_head(scene_v),
            'sentiment_logits':      self.sentiment_head(scene_v),
            'transition_matrix':     trans_mat,         # (B, 7, 7)
            # Regression heads
            'variance_pred':         self.variance_head(scene_v).squeeze(-1),
            'conflict_pred':         self.conflict_head(scene_v).squeeze(-1),
            'arousal_pred':          self.arousal_head(scene_v).squeeze(-1),
            'trajectory_pred':       self.trajectory_head(scene_v).squeeze(-1),
            # Temporal curves (for Module 4 hit-point logic)
            'emotion_curve_logits':  per_utt_emo,       # (B, L, 7)
            'intensity_curve':       per_utt_int,       # (B, L)
            'attn_weights':          attn_w,            # (B, L)
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — Multi-Task Loss                                                 │
# │                                                                          │
# │  Module 2 trains on 9 simultaneous targets.                             │
# │                                                                          │
# │  WEIGHT RATIONALE:                                                       │
# │    dominant_emotion  ×1.0 — primary scene-level classification          │
# │    scene_type        ×0.5 — important for conditioning Module 4         │
# │    transition_matrix ×0.4 — harmonic planning for Module 4              │
# │    emotion_dist      ×0.4 — overall scene feel                          │
# │    emotion_curve     ×0.2 — FIX 2: reduced from 0.4 because            │
# │                              Module 1 labels have only 32% F1;          │
# │                              trusting noisy per-utt labels too much      │
# │                              degrades the scene-level signal            │
# │    intensity_curve   ×0.3 — norm-based, doesn't depend on label quality │
# │    sentiment         ×0.3 — coarse 4-class, relatively clean            │
# │    variance/conflict/arousal ×0.2 each — derived scalars               │
# │    trajectory        ×0.2 — first/last valence                          │
# └──────────────────────────────────────────────────────────────────────────┘

class SceneLossV3(nn.Module):
    def __init__(self, emotion_weights=None):
        super().__init__()
        self.ce     = nn.CrossEntropyLoss(
            weight=emotion_weights,
            label_smoothing=0.05)    # light smoothing — scene labels noisy
        self.ce_raw = nn.CrossEntropyLoss(
            label_smoothing=0.05)
        self.mse    = nn.MSELoss()

    def _trans_kl(self, pred: torch.Tensor,
                  true: torch.Tensor) -> torch.Tensor:
        """
        Row-wise KL divergence on (B, 7, 7) transition matrices.
        Skips rows where the true distribution is all-zero
        (transitions that never occurred in this scene).
        """
        B = pred.shape[0]
        pred_flat  = pred.view(B * CFG2.NUM_EMOTIONS,
                               CFG2.NUM_EMOTIONS)
        true_flat  = true.view(B * CFG2.NUM_EMOTIONS,
                               CFG2.NUM_EMOTIONS)
        valid      = (true_flat.sum(dim=-1) > 0)
        if not valid.any():
            return torch.tensor(0.0, device=pred.device)
        lp = torch.log(pred_flat[valid] + 1e-8)
        tg = true_flat[valid]
        return (tg * (torch.log(tg + 1e-8) - lp)).sum(dim=-1).mean()

    def _dist_kl(self, logits: torch.Tensor,
                 target: torch.Tensor) -> torch.Tensor:
        """KL on overall emotion distribution (B, 7)."""
        log_p = torch.log_softmax(logits, dim=-1)
        valid = (target.sum(dim=-1) > 0)
        if not valid.any():
            return torch.tensor(0.0, device=logits.device)
        tg    = target[valid]
        return (tg * (torch.log(tg + 1e-8) - log_p[valid])).sum(dim=-1).mean()

    def forward(self, preds: dict, batch: dict) -> dict:
        D    = DEVICE
        mask = batch['mask'].to(D)     # (B, L) bool
        B, L = mask.shape

        # ── Scene-level losses ─────────────────────────────────────────
        l_emo   = self.ce(
            preds['emotion_logits'],
            batch['dominant_emotion'].to(D))

        l_st    = self.ce_raw(
            preds['scene_type_logits'],
            batch['scene_type_target'].to(D))

        l_sent  = self.ce_raw(
            preds['sentiment_logits'],
            batch['sentiment_label'].to(D))

        l_trans = self._trans_kl(
            preds['transition_matrix'],
            batch['transition_matrix'].to(D))

        l_dist  = self._dist_kl(
            preds['emotion_logits'],
            batch['emotion_dist'].to(D))

        l_var   = self.mse(preds['variance_pred'],
                           batch['variance_score'].to(D))
        l_con   = self.mse(preds['conflict_pred'],
                           batch['conflict_score'].to(D))
        l_aro   = self.mse(preds['arousal_pred'],
                           batch['arousal_score'].to(D))
        l_traj  = self.mse(preds['trajectory_pred'],
                           batch['trajectory_score'].to(D))

        # ── Per-utterance curve losses (masked — skip padding) ─────────
        mf   = mask.view(B * L)
        ec_l = preds['emotion_curve_logits'].view(B * L, -1)
        ec_t = batch['emotion_curve'].to(D).view(B * L)
        ic_p = preds['intensity_curve'].view(B * L)
        ic_t = batch['intensity_curve'].to(D).view(B * L)

        if mf.sum() > 0:
            l_ec = self.ce_raw(ec_l[mf], ec_t[mf])
            l_ic = self.mse(ic_p[mf], ic_t[mf])
        else:
            l_ec = torch.tensor(0.0, device=D)
            l_ic = torch.tensor(0.0, device=D)

        # ── Weighted sum ───────────────────────────────────────────────
        # FIX 2: l_ec weight reduced 0.4 → 0.2 (noisy Module 1 labels)
        total = (1.0 * l_emo   +
                 0.5 * l_st    +
                 0.4 * l_trans +
                 0.4 * l_dist  +
                 0.3 * l_sent  +
                 0.3 * l_ic    +
                 0.2 * l_ec    +   # ← was 0.4, reduced due to label noise
                 0.2 * l_var   +
                 0.2 * l_con   +
                 0.2 * l_aro   +
                 0.2 * l_traj)

        return {
            'total':      total,
            'emo':        l_emo.item(),
            'scene_type': l_st.item(),
            'trans':      l_trans.item(),
            'curve_emo':  l_ec.item(),
            'curve_int':  l_ic.item(),
            'arousal':    l_aro.item(),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Training / Evaluation Loop                                      │
# └──────────────────────────────────────────────────────────────────────────┘

def run_epoch(model, loader, loss_fn,
              optimizer=None, phase='train',
              epoch=0, total=0) -> tuple:
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()

    tot_loss = 0.0
    lp       = defaultdict(float)
    all_t, all_p = [], []
    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=(f"Ep {epoch+1:02d}/{total} "
              f"{'▶ Train' if is_train else '  Val  '}"),
        leave=True,
        bar_format="{l_bar}{bar:25}{r_bar}")

    for step, batch in enumerate(pbar):
        mat  = batch['emotion_matrix'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)
        st   = batch['scene_type_input'].to(DEVICE)

        with torch.set_grad_enabled(is_train):
            preds  = model(mat, mask, st)
            losses = loss_fn(preds, batch)

        if is_train:
            optimizer.zero_grad()
            losses['total'].backward()
            # FIX 6: gradient clipping — small dataset → spiky gradients
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        all_t.extend(batch['dominant_emotion'].numpy().tolist())
        all_p.extend(
            preds['emotion_logits'].argmax(-1)
                 .detach().cpu().numpy().tolist())

        tot_loss += losses['total'].item()
        for k, v in losses.items():
            if k != 'total':
                lp[k] += (v if isinstance(v, float) else v.item())

        if (step + 1) % 5 == 0:
            f1 = f1_score(all_t, all_p,
                          average='macro', zero_division=0) * 100
            pbar.set_postfix(
                loss=f"{tot_loss/(step+1):.3f}",
                f1=f"{f1:.1f}%")

    n    = len(loader)
    avg  = tot_loss / n
    f1   = f1_score(all_t, all_p, average='macro', zero_division=0)
    acc  = accuracy_score(all_t, all_p)
    m, s = divmod(int(time.time() - t0), 60)

    print(f"    ↳ {phase.upper():5s}  loss={avg:.4f}  "
          f"[emo={lp['emo']/n:.3f}  "
          f"st={lp['scene_type']/n:.3f}  "
          f"cur={lp['curve_emo']/n:.3f}]  "
          f"f1={f1*100:.2f}%  ({m}m{s:02d}s)")

    return avg, {'macro_f1': f1, 'accuracy': acc, 'loss': avg}


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Plots                                                           │
# └──────────────────────────────────────────────────────────────────────────┘

DARK_BG  = '#0f0f1a'
PANEL_BG = '#1a1a2e'
ACCENT   = ['#2ecc71','#e74c3c','#f39c12',
            '#3498db','#9b59b6','#1abc9c','#e67e22']


def _ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors='#cccccc', labelsize=8)
    ax.set_title(title, color='white',
                 fontsize=9, fontweight='bold', pad=5)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#333355')


def plot_m2_training(history: dict):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 2 — Scene Encoder Training',
                 color='white', fontsize=11, fontweight='bold')

    ep_tr = [h['epoch'] for h in history['train']]
    ep_va = [h['epoch'] for h in history['val']]

    axes[0].plot(ep_tr, [h['loss'] for h in history['train']],
                 color='#3498db', lw=2, marker='o', ms=3, label='Train')
    axes[0].plot(ep_va, [h['loss'] for h in history['val']],
                 color='#e74c3c', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[0].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    _ax(axes[0], 'Loss', 'Epoch', 'Loss')

    axes[1].plot(ep_tr, [h['f1']*100 for h in history['train']],
                 color='#2ecc71', lw=2, marker='o', ms=3, label='Train')
    axes[1].plot(ep_va, [h['f1']*100 for h in history['val']],
                 color='#f39c12', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[1].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    axes[1].set_ylim(0, 100)
    _ax(axes[1], 'Dominant Emotion F1', 'Epoch', 'Macro F1 (%)')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m2_01_training_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m2_01_training_curves.png")


def plot_scene_type_distribution(dataset: SceneDatasetV3):
    type_counts = Counter(
        s['scene_type_input'].item() for s in dataset.samples)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 2 — Scene Type Distribution',
                 color='white', fontsize=11, fontweight='bold')

    labels = SCENE_TYPE_NAMES
    values = [type_counts.get(i, 0)
              for i in range(CFG2.NUM_SCENE_TYPES)]

    axes[0].pie(values, labels=labels, colors=ACCENT,
                autopct='%1.1f%%', startangle=90,
                textprops={'color': 'white', 'fontsize': 8},
                wedgeprops={'edgecolor': DARK_BG, 'linewidth': 1})
    axes[0].set_facecolor(DARK_BG)
    _ax(axes[0], 'Scene Type Distribution')

    axes[1].barh(labels, values, color=ACCENT,
                 alpha=0.85, edgecolor='white', linewidth=0.5)
    for i, v in enumerate(values):
        axes[1].text(v + max(values) * 0.01, i, str(v),
                     va='center', color='white', fontsize=8)
    _ax(axes[1], 'Scene Counts', 'Count', 'Scene Type')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m2_02_scene_types.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m2_02_scene_types.png")


def plot_emotion_curves_sample(all_meta: list, n_scenes: int = 4):
    """
    FIX 7: all_meta entries must have 'emotion_curve' as (U, 7) tensor
    and 'intensity_curve' as (U,) tensor — these are set in Cell 13.
    This function is only called AFTER Cell 13 populates all_meta.
    """
    samples = [m for m in all_meta
               if m['num_utterances'] >= 4][:n_scenes]
    if not samples:
        print("  ⚠ No scenes with ≥4 utterances for curve plot")
        return

    fig, axes = plt.subplots(
        len(samples), 2, figsize=(14, 3 * len(samples)))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle(
        'Module 2 — Emotion & Intensity Curves\n'
        '(Temporal signals sent to Module 4 for music hit-point planning)',
        color='white', fontsize=10, fontweight='bold')

    if len(samples) == 1:
        axes = [axes]

    for row, meta in enumerate(samples):
        U   = meta['num_utterances']
        ec  = meta['emotion_curve']        # (U, 7) softmax probs
        ic  = meta['intensity_curve']      # (U,)  intensity [0,1]
        dom = ec.argmax(dim=-1).numpy()    # (U,) dominant per utt
        sid = meta['scene_id']

        # Left: dominant emotion colour bar
        ax0 = axes[row][0]
        colors_utt = [ACCENT[d % len(ACCENT)] for d in dom]
        ax0.bar(range(U), [1] * U, color=colors_utt,
                alpha=0.7, edgecolor='white', linewidth=0.3)
        for j, d in enumerate(dom):
            ax0.text(j, 0.5, CLASS_NAMES[d][:3],
                     ha='center', va='center',
                     color='white', fontsize=6, fontweight='bold')
        _ax(ax0, f'Scene {sid} — Emotion Sequence', 'Utterance', '')
        ax0.set_yticks([])

        # Right: intensity curve
        ax1 = axes[row][1]
        ic_np = ic.numpy() if isinstance(ic, torch.Tensor) else ic
        ax1.fill_between(range(U), ic_np, alpha=0.4, color='#f39c12')
        ax1.plot(range(U), ic_np, color='#f39c12',
                 lw=2, marker='o', ms=4)
        ax1.set_ylim(0, 1.1)
        _ax(ax1, f'Scene {sid} — Intensity', 'Utterance', 'Intensity')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m2_03_emotion_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m2_03_emotion_curves.png")


def plot_scene_vector_heatmap(all_meta: list, n: int = 30):
    vecs  = torch.stack([m['scene_vector'] for m in all_meta[:n]])
    doms  = [m['dominant_emotion'] for m in all_meta[:n]]
    labels = [CLASS_NAMES[d] for d in doms]

    v_norm = (vecs - vecs.mean(0)) / (vecs.std(0) + 1e-6)

    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    im = ax.imshow(v_norm.numpy(), aspect='auto',
                   cmap='RdBu_r', vmin=-2, vmax=2)
    plt.colorbar(im, ax=ax, label='Normalised Value')
    ax.set_yticks(range(n))
    ax.set_yticklabels(
        [f'{i}: {l}' for i, l in enumerate(labels)],
        fontsize=6, color='white')
    ax.set_xlabel('Scene Vector Dimension', color='#aaaaaa')
    _ax(ax, f'Scene Vectors — first {n} scenes')
    ax.tick_params(colors='#cccccc')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m2_04_scene_vectors.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m2_04_scene_vectors.png")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Load Data + Build Dataset                                      │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "=" * 60)
print("  LOADING MODULE 1 OUTPUTS")
print("=" * 60)

scene_data = torch.load(CFG2.SCENE_MATRIX, map_location='cpu')
print(f"  Loaded {len(scene_data):,} scenes from Module 1")

# Auto-check and fix input dimension if needed
actual_dim = scene_data[0]['emotion_matrix'].shape[-1]
if actual_dim != CFG2.INPUT_DIM:
    print(f"  ⚠ scene_emotion_matrix dim={actual_dim} "
          f"but CFG2.INPUT_DIM={CFG2.INPUT_DIM}")
    print(f"  Auto-adjusting CFG2.INPUT_DIM → {actual_dim}")
    CFG2.INPUT_DIM = actual_dim
else:
    print(f"  ✓ Input dim = {actual_dim} (matches CFG2.INPUT_DIM)")

# Print scene stats before building dataset
all_labels_flat = []
all_lengths     = []
for s in scene_data:
    all_labels_flat.extend(s['labels'])
    all_lengths.append(len(s['labels']))

print(f"\n  Scene stats:")
print(f"    Total scenes    : {len(scene_data):,}")
print(f"    Total utterances: {len(all_labels_flat):,}")
print(f"    Lengths min/max/avg: "
      f"{min(all_lengths)} / {max(all_lengths)} / "
      f"{np.mean(all_lengths):.1f}")

cnt   = Counter(all_labels_flat)
total = len(all_labels_flat)
print(f"\n  Label distribution (from Module 1):")
for i, e in enumerate(CLASS_NAMES):
    c = cnt.get(i, 0)
    print(f"    {e:<10} {c:>5}  {c/total*100:5.1f}%  "
          f"{'█' * int(c/total*30)}")

# Build dataset
full_dataset = SceneDatasetV3(scene_data)

# Stratified 70/15/15 split on dominant emotion label
all_idx = list(range(len(full_dataset)))
all_dom = [full_dataset.samples[i]['dominant_emotion'].item()
           for i in all_idx]

tr_va, te = train_test_split(
    all_idx, test_size=CFG2.TEST_SPLIT,
    stratify=all_dom, random_state=SEED)
tr_va_dom = [all_dom[i] for i in tr_va]
tr, va = train_test_split(
    tr_va,
    test_size=CFG2.VAL_SPLIT / (1.0 - CFG2.TEST_SPLIT),
    stratify=tr_va_dom, random_state=SEED)

print(f"\n  Split: train={len(tr)}  val={len(va)}  test={len(te)}")

_kw = dict(collate_fn=collate_scenes,
           num_workers=2, pin_memory=True)
train_loader = DataLoader(
    Subset(full_dataset, tr),
    CFG2.BATCH_SIZE, shuffle=True,  **_kw)
val_loader   = DataLoader(
    Subset(full_dataset, va),
    CFG2.BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(
    Subset(full_dataset, te),
    CFG2.BATCH_SIZE, shuffle=False, **_kw)

# ── Class weights (with cap) ──────────────────────────────────────────────
# FIX 1: clamp to MAX_WEIGHT=15.0.
# 'peaceful' has 0 training scenes → count set to 1 → raw weight ≈ 142.
# Without clamping this crashes training in epoch 1 (loss → NaN).
tr_labels = [full_dataset.samples[i]['dominant_emotion'].item()
             for i in tr]
counts  = np.bincount(tr_labels,
                      minlength=CFG2.NUM_EMOTIONS).astype(float)
counts  = np.where(counts == 0, 1, counts)    # guard zero-count classes
raw_w   = len(tr) / (CFG2.NUM_EMOTIONS * counts)
clamped = np.clip(raw_w, 0.1, CFG2.MAX_WEIGHT)
weights = torch.tensor(clamped, dtype=torch.float32).to(DEVICE)

print(f"\n  Class weights (clamped ≤ {CFG2.MAX_WEIGHT}):")
for i, (e, c, rw, cw) in enumerate(
        zip(CLASS_NAMES, counts, raw_w, clamped)):
    flag = ' ← CLAMPED' if rw > CFG2.MAX_WEIGHT else ''
    print(f"    {e:<10}  count={int(c):>4}  "
          f"raw={rw:6.1f}  clamped={cw:.2f}{flag}")

# Plot before training
plot_scene_type_distribution(full_dataset)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — Build Model + Train                                            │
# └──────────────────────────────────────────────────────────────────────────┘

model2 = SceneEncoderV3(
    input_dim      = CFG2.INPUT_DIM,
    scene_type_dim = CFG2.SCENE_TYPE_DIM,
    hidden_dim     = CFG2.HIDDEN_DIM,
    scene_vec_dim  = CFG2.SCENE_VEC_DIM,
    num_emotions   = CFG2.NUM_EMOTIONS,
    num_scene_types= CFG2.NUM_SCENE_TYPES,
    num_sentiments = CFG2.NUM_SENTIMENTS,
    dropout        = CFG2.DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model2.parameters())
print(f"\n  SceneEncoderV3 params: {total_params:,}")

loss_fn2   = SceneLossV3(emotion_weights=weights)
optimizer2 = optim.Adam(
    model2.parameters(), lr=CFG2.LR, weight_decay=1e-5)

# ReduceLROnPlateau: halves LR when val loss stalls for 3 epochs
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode='min', patience=3,
    factor=0.5, min_lr=1e-6)

print("\n" + "═" * 60)
print("  TRAINING SCENE ENCODER V3")
print(f"  Input : {CFG2.INPUT_DIM}d DTE vectors per utterance")
print(f"  Model : 2-layer BiLSTM({CFG2.HIDDEN_DIM}×2) + 9 heads")
print(f"  Train : {len(tr)} scenes  "
      f"Val : {len(va)} scenes  "
      f"Test : {len(te)} scenes")
print(f"  Epochs {CFG2.EPOCHS}  Batch {CFG2.BATCH_SIZE}  "
      f"LR {CFG2.LR}")
print(f"  Early stop patience : {CFG2.EARLY_STOP}")
print("═" * 60)

history    = {'train': [], 'val': []}
best_f1    = 0.0
patience_c = 0

for epoch in range(CFG2.EPOCHS):
    tr_l, tr_m = run_epoch(
        model2, train_loader, loss_fn2,
        optimizer2, 'train', epoch, CFG2.EPOCHS)
    va_l, va_m = run_epoch(
        model2, val_loader, loss_fn2,
        None, 'val', epoch, CFG2.EPOCHS)

    scheduler2.step(va_l)

    history['train'].append({
        'epoch': epoch + 1,
        'loss':  tr_l,
        'f1':    tr_m['macro_f1']})
    history['val'].append({
        'epoch': epoch + 1,
        'loss':  va_l,
        'f1':    va_m['macro_f1']})

    if va_m['macro_f1'] > best_f1:
        best_f1    = va_m['macro_f1']
        patience_c = 0
        torch.save({
            'model_state':  model2.state_dict(),
            'macro_f1':     best_f1,
            'config': {
                'input_dim':       CFG2.INPUT_DIM,
                'hidden_dim':      CFG2.HIDDEN_DIM,
                'scene_vec_dim':   CFG2.SCENE_VEC_DIM,
                'num_emotions':    CFG2.NUM_EMOTIONS,
                'num_scene_types': CFG2.NUM_SCENE_TYPES,
                'num_sentiments':  CFG2.NUM_SENTIMENTS,
                'dropout':         CFG2.DROPOUT,
                'class_names':     CLASS_NAMES,
                'scene_type_names':SCENE_TYPE_NAMES,
            },
        }, CFG2.BEST_CKPT)
        print(f"    ✓ Saved scene_encoder_best.pt "
              f"(f1={best_f1*100:.2f}%)")
    else:
        patience_c += 1
        if patience_c >= CFG2.EARLY_STOP:
            print(f"\n  ⏹ Early stop at epoch {epoch + 1}")
            break

plot_m2_training(history)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — Final Evaluation on Test Set                                   │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 60)
print("  FINAL EVALUATION — TEST SET")
print("═" * 60)

ckpt = torch.load(CFG2.BEST_CKPT, map_location=DEVICE)
model2.load_state_dict(ckpt['model_state'])
model2.eval()

all_t, all_p = [], []
with torch.no_grad():
    for batch in test_loader:
        mat  = batch['emotion_matrix'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)
        st   = batch['scene_type_input'].to(DEVICE)
        out  = model2(mat, mask, st)
        all_t.extend(batch['dominant_emotion'].numpy().tolist())
        all_p.extend(
            out['emotion_logits'].argmax(-1).cpu().numpy().tolist())

test_f1  = f1_score(all_t, all_p, average='macro', zero_division=0)
test_acc = accuracy_score(all_t, all_p)

print(f"\n  Test Macro F1 : {test_f1*100:.2f}%")
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Best Val F1   : {best_f1*100:.2f}%")
print(f"\n  Classification Report:")
print(classification_report(
    all_t, all_p,
    target_names=CLASS_NAMES,
    labels=list(range(CFG2.NUM_EMOTIONS)),
    zero_division=0))


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Extract Scene Vectors → Module 3                               │
# │                                                                          │
# │  Runs the trained model over ALL scenes (train+val+test).               │
# │  Saves a scene_vectors.pt file containing:                              │
# │    scene_vector     : (N, 256)  — the compressed scene representations  │
# │    scene_metadata   : list[dict] — all per-scene signals for Module 3/4 │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 60)
print("  EXTRACTING SCENE VECTORS → MODULE 3")
print("═" * 60)

full_loader = DataLoader(
    full_dataset, batch_size=64,
    collate_fn=collate_scenes,
    shuffle=False, num_workers=2)

all_vecs, all_meta = [], []

with torch.no_grad():
    for i, batch in enumerate(tqdm(full_loader,
                                    desc="  Extracting")):
        mat  = batch['emotion_matrix'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)
        st   = batch['scene_type_input'].to(DEVICE)
        out  = model2(mat, mask, st)
        B    = mat.size(0)

        for b in range(B):
            g = i * 64 + b
            if g >= len(full_dataset):
                break
            s  = full_dataset.samples[g]
            U  = s['num_utterances']
            sv = out['scene_vector'][b].cpu()
            all_vecs.append(sv)

            # emotion_curve: softmax probs per real utterance (U, 7)
            ec_soft = torch.softmax(
                out['emotion_curve_logits'][b, :U], dim=-1).cpu()
            # intensity_curve: (U,)
            ic      = out['intensity_curve'][b, :U].cpu()

            all_meta.append({
                # ── Primary for Module 3 ──────────────────────────
                'scene_id':           s['scene_id'],
                'scene_vector':       sv,              # (256,)
                # ── Scene-level predictions ──────────────────────
                'emotion_probs':      torch.softmax(
                    out['emotion_logits'][b], dim=0).cpu(),   # (7,)
                'dominant_emotion':   s['dominant_emotion'].item(),
                'predicted_emotion':  out['emotion_logits'][b].argmax().item(),
                'scene_type':         out['scene_type_logits'][b].argmax().item(),
                'scene_type_name':    SCENE_TYPE_NAMES[
                    out['scene_type_logits'][b].argmax().item()],
                'transition_matrix':  out['transition_matrix'][b].cpu(),  # (7,7)
                'arousal_score':      out['arousal_pred'][b].item(),
                'conflict_score':     out['conflict_pred'][b].item(),
                'variance_score':     out['variance_pred'][b].item(),
                'trajectory_score':   out['trajectory_pred'][b].item(),
                # ── Temporal curves for Module 4 ─────────────────
                'emotion_curve':      ec_soft,         # (U, 7)
                'intensity_curve':    ic,              # (U,)
                'attn_weights':       out['attn_weights'][b, :U].cpu(),
                # ── Metadata ─────────────────────────────────────
                'num_utterances':     U,
                'pacing_score':       s['pacing_score'].item(),
                'speakers':           s['speakers'],
                'emotion_dist':       s['emotion_dist'].numpy(),
                'first_emotion':      s['first_emotion'],
                'last_emotion':       s['last_emotion'],
            })

# Generate visualisation plots (uses all_meta populated above)
plot_emotion_curves_sample(all_meta, n_scenes=4)
plot_scene_vector_heatmap(all_meta, n=30)

# Save to disk
vec_tensor = torch.stack(all_vecs)              # (N, 256)
torch.save({
    'scene_vectors':     vec_tensor,
    'scene_metadata':    all_meta,
    'vector_dim':        CFG2.SCENE_VEC_DIM,
    'num_scenes':        len(all_meta),
    'label_map':         LABEL2IDX,
    'scene_type_names':  SCENE_TYPE_NAMES,
    'class_names':       CLASS_NAMES,
}, CFG2.SCENE_VEC_OUT)

with open(f"{CFG2.CKPT_DIR}/m2_history.json", 'w') as f:
    json.dump(history, f, indent=2)

# Summary
print(f"\n  ✓ scene_vectors.pt     shape={tuple(vec_tensor.shape)}")
print(f"  ✓ {len(all_meta):,} scene metadata entries saved")
print(f"\n  Plots:")
print(f"    m2_01_training_curves.png")
print(f"    m2_02_scene_types.png")
print(f"    m2_03_emotion_curves.png")
print(f"    m2_04_scene_vectors.png")
print(f"\n  {'═'*55}")
print(f"  MODULE 2 COMPLETE")
print(f"  {'═'*55}")
print(f"  Architecture : SceneEncoderV3")
print(f"  Input dim    : {CFG2.INPUT_DIM}d  (DTE utterance vectors)")
print(f"  Output dim   : {CFG2.SCENE_VEC_DIM}d  (scene_vector → M3)")
print(f"  Scenes       : {len(all_meta):,}")
print(f"  Test F1      : {test_f1*100:.2f}%")
print(f"  Best val F1  : {best_f1*100:.2f}%")
print(f"  {'═'*55}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 14 — Zip Outputs for Download                                       │
# └──────────────────────────────────────────────────────────────────────────┘

import zipfile
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/module2_outputs.zip"
print("\n" + "═" * 60)
print("  ZIPPING MODULE 2 OUTPUTS")
print("═" * 60)

_dirs = [
    ("/kaggle/working/m2_checkpoints", "m2_checkpoints"),
    ("/kaggle/working/plots",          "plots"),
]
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc in _dirs:
        for fpath in Path(src_dir).rglob('*'):
            if fpath.is_file():
                arcname = arc + '/' + str(
                    fpath.relative_to(src_dir))
                zf.write(str(fpath), arcname)
                print(f"  + {arcname}  "
                      f"({fpath.stat().st_size/1024:.1f} KB)")

zip_mb = Path(ZIP_PATH).stat().st_size / (1024 * 1024)
print(f"\n  ✓ module2_outputs.zip  ({zip_mb:.1f} MB)")
print(f"\n  Contents:")
print(f"    m2_checkpoints/scene_encoder_best.pt")
print(f"    m2_checkpoints/scene_vectors.pt")
print(f"    m2_checkpoints/m2_history.json")
print(f"    plots/m2_01_training_curves.png")
print(f"    plots/m2_02_scene_types.png")
print(f"    plots/m2_03_emotion_curves.png")
print(f"    plots/m2_04_scene_vectors.png")

display(FileLink(ZIP_PATH, result_html_prefix="⬇️  Download: "))

  Device : cuda
  GPU    : Tesla T4
  Config ready ✓
  INPUT_DIM  = 256  (DTE 256d output)
  BATCH_SIZE = 16  (~16 scenes/batch)
  MAX_WEIGHT = 15.0  (class weight cap)
  Emotion classes : ['neutral', 'joyful', 'sad', 'mad', 'scared', 'peaceful', 'powerful']
  Scene types     : ['dialogue', 'action', 'investigation', 'suspense', 'dramatic_reveal', 'transition']
  LABEL2IDX       : {'neutral': 0, 'joyful': 1, 'sad': 2, 'mad': 3, 'scared': 4, 'peaceful': 5, 'powerful': 6}

  LOADING MODULE 1 OUTPUTS
  Loaded 1,423 scenes from Module 1
  ✓ Input dim = 256 (matches CFG2.INPUT_DIM)

  Scene stats:
    Total scenes    : 1,423
    Total utterances: 13,708
    Lengths min/max/avg: 1 / 33 / 9.6

  Label distribution (from Module 1):
    neutral     6436   47.0%  ██████████████
    joyful      2308   16.8%  █████
    sad         1002    7.3%  ██
    mad         1968   14.4%  ████
    scared       358    2.6%  
    peaceful       0    0.0%  
    powerful    1636   11.9%  ███


  Building SceneDataset:   0%|          | 0/1423 [00:00<?, ?it/s]

  Scenes built : 1,337  (skipped 86 too-short scenes)

  Split: train=935  val=201  test=201

  Class weights (clamped ≤ 15.0):
    neutral     count= 601  raw=   0.2  clamped=0.22
    joyful      count= 127  raw=   1.1  clamped=1.05
    sad         count=  33  raw=   4.0  clamped=4.05
    mad         count= 114  raw=   1.2  clamped=1.17
    scared      count=   9  raw=  14.8  clamped=14.84
    peaceful    count=   1  raw= 133.6  clamped=15.00 ← CLAMPED
    powerful    count=  51  raw=   2.6  clamped=2.62
  ✓ m2_02_scene_types.png

  SceneEncoderV3 params: 1,015,631

════════════════════════════════════════════════════════════
  TRAINING SCENE ENCODER V3
  Input : 256d DTE vectors per utterance
  Model : 2-layer BiLSTM(128×2) + 9 heads
  Train : 935 scenes  Val : 201 scenes  Test : 201 scenes
  Epochs 30  Batch 16  LR 0.0003
  Early stop patience : 8
════════════════════════════════════════════════════════════


Ep 01/30 ▶ Train:   0%|                         | 0/59 [00:00<?, ?it/s]

    ↳ TRAIN  loss=4.7331  [emo=2.550  st=1.024  cur=1.797]  f1=22.98%  (0m01s)


...

Ep 24/30   Val  :   0%|                         | 0/13 [00:00<?, ?it/s]

    ↳ VAL    loss=3.4355  [emo=2.135  st=0.247  cur=1.121]  f1=53.90%  (0m00s)

  ⏹ Early stop at epoch 24
  ✓ m2_01_training_curves.png

════════════════════════════════════════════════════════════
  FINAL EVALUATION — TEST SET
════════════════════════════════════════════════════════════

  Test Macro F1 : 55.32%
  Test Accuracy : 70.15%
  Best Val F1   : 55.63%

  Classification Report:
              precision    recall  f1-score   support

     neutral       0.86      0.74      0.79       129
      joyful       0.47      0.71      0.56        28
         sad       0.29      0.29      0.29         7
         mad       0.65      0.62      0.64        24
      scared       0.33      0.50      0.40         2
    peaceful       0.00      0.00      0.00         0
    powerful       0.57      0.73      0.64        11

    accuracy                           0.70       201
   macro avg       0.45      0.51      0.47       201
weighted avg       0.74      0.70      0.71       201


══════════

  Extracting:   0%|          | 0/21 [00:00<?, ?it/s]

  ✓ m2_03_emotion_curves.png
  ✓ m2_04_scene_vectors.png

  ✓ scene_vectors.pt     shape=(1337, 256)
  ✓ 1,337 scene metadata entries saved

  Plots:
    m2_01_training_curves.png
    m2_02_scene_types.png
    m2_03_emotion_curves.png
    m2_04_scene_vectors.png

  ═══════════════════════════════════════════════════════
  MODULE 2 COMPLETE
  ═══════════════════════════════════════════════════════
  Architecture : SceneEncoderV3
  Input dim    : 256d  (DTE utterance vectors)
  Output dim   : 256d  (scene_vector → M3)
  Scenes       : 1,337
  Test F1      : 55.32%
  Best val F1  : 55.63%
  ═══════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  ZIPPING MODULE 2 OUTPUTS
════════════════════════════════════════════════════════════
  + m2_checkpoints/m2_history.json  (4.5 KB)
  + m2_checkpoints/scene_encoder_best.pt  (3987.1 KB)
  + m2_checkpoints/scene_vectors.pt  (5802.0 KB)
  + plots/p2_02_confusion_matrix.png  (133.5 KB)
  

/kaggle/working/module2_outputs.zip

In [17]:
import torch
import numpy as np
from collections import Counter

scenes = torch.load(
    '/kaggle/input/datasets/suyashnpande/module1-phase2/checkpoints/final_emotion_vectors/scene_emotion_matrix.pt',
    map_location='cpu')

emo = ['neutral','joyful','sad','mad','scared','peaceful','powerful']
all_labels = []
scene_lengths = []
for s in scenes:
    all_labels.extend(s['labels'])
    scene_lengths.append(len(s['labels']))

cnt = Counter(all_labels)
total = len(all_labels)

print(f"Total scenes    : {len(scenes)}")
print(f"Total utterances: {total}")
print(f"Scene length    : min={min(scene_lengths)}  max={max(scene_lengths)}  avg={np.mean(scene_lengths):.1f}")
print()
print("Label distribution:")
for i, e in enumerate(emo):
    c = cnt.get(i, 0)
    bar = '█' * int(c/total*30)
    print(f"  {e:<10} {c:>5}  {c/total*100:5.1f}%  {bar}")

# Check vector health
all_vecs = torch.cat([s['emotion_matrix'] for s in scenes], dim=0)
print(f"\nVector stats (all {all_vecs.shape[0]} utterance vectors, dim={all_vecs.shape[1]}):")
print(f"  mean : {all_vecs.mean():.4f}")
print(f"  std  : {all_vecs.std():.4f}")
print(f"  min  : {all_vecs.min():.4f}")
print(f"  max  : {all_vecs.max():.4f}")
print(f"  NaN  : {torch.isnan(all_vecs).sum().item()}")
print(f"  Inf  : {torch.isinf(all_vecs).sum().item()}")
print()

# Check a real scene
s = scenes[0]
print(f"Sample scene {s['scene_id']}:")
print(f"  utterances  : {len(s['labels'])}")
print(f"  labels      : {[emo[l] for l in s['labels']]}")
print(f"  speakers    : {s['speakers']}")
print(f"  matrix shape: {s['emotion_matrix'].shape}")

Total scenes    : 1423
Total utterances: 13708
Scene length    : min=1  max=33  avg=9.6

Label distribution:
  neutral     6436   47.0%  ██████████████
  joyful      2308   16.8%  █████
  sad         1002    7.3%  ██
  mad         1968   14.4%  ████
  scared       358    2.6%  
  peaceful       0    0.0%  
  powerful    1636   11.9%  ███

Vector stats (all 13708 utterance vectors, dim=256):
  mean : 0.3987
  std  : 0.5747
  min  : 0.0000
  max  : 3.7323
  NaN  : 0
  Inf  : 0

Sample scene (1, 1, 19):
  utterances  : 3
  labels      : ['neutral', 'powerful', 'neutral']
  speakers    : ['Joey', 'Joey', 'Phoebe']
  matrix shape: torch.Size([3, 256])


In [14]:
print("Module 3 starts here")

Module 3 starts here


In [22]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODULE 3 — Narrative LSTM  v1                                         ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  WHERE IT SITS IN THE PIPELINE:                                         ║
# ║                                                                         ║
# ║    M1 (utterance) → M2 (scene) → M3 (narrative) → M4 (music)          ║
# ║                                                                         ║
# ║    M2 gave us one 256d vector per scene.                               ║
# ║    M3 reads the full SEQUENCE of scene vectors for an episode           ║
# ║    and outputs a 256d narrative vector per scene — i.e. each scene     ║
# ║    now knows WHERE it sits in the story arc.                           ║
# ║                                                                         ║
# ║  WHY CAUSAL LSTM (not BiLSTM, not Transformer):                        ║
# ║    A composer writing music for scene 12 has only HEARD scenes 1-11.   ║
# ║    They cannot know what happens in scene 13.                          ║
# ║    Causal LSTM enforces this — scene t only sees scenes 0..t-1.        ║
# ║    BiLSTM would leak future information. Wrong for music scoring.       ║
# ║                                                                         ║
# ║  THREE MECHANISMS FUSED:                                                ║
# ║    1. Causal 2-layer LSTM   — basic narrative flow                     ║
# ║    2. TensionMemoryCell     — accumulates/releases dramatic pressure   ║
# ║    3. CharacterMemoryBank   — tracks each character's emotional state  ║
# ║                                                                         ║
# ║  INPUTS  (from Module 2 scene_vectors.pt):                             ║
# ║    scene_vectors   : (S, 256)   per scene                              ║
# ║    scene_metadata  : list[dict] speakers, arousal, trajectory…         ║
# ║                                                                         ║
# ║  OUTPUTS (to scene_narrative.pt → Module 4):                           ║
# ║    narrative_vector : (256,)   the fused narrative context             ║
# ║    tension_score    : float    accumulated narrative pressure [0,1]     ║
# ║    arc_phase        : int      5-class: setup/rising/climax/fall/res   ║
# ║    reversal_score   : float    did emotion flip from last scene? [0,1] ║
# ║    char_context     : (64,)    dominant character emotional state       ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  DATASET NOTE:                                                          ║
# ║    Module 3 has NO external labels. All targets are derived from       ║
# ║    scene_metadata already in scene_vectors.pt:                         ║
# ║      arc_phase    ← scene position within episode (0-4 quintiles)     ║
# ║      tension      ← cumulative arousal - trajectory running signal     ║
# ║      reversal     ← sign change in trajectory between consecutive      ║
# ║                     scenes (binary: did emotion direction flip?)        ║
# ║    This is fully self-supervised — no human annotation needed.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import os, json, pickle, time, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (f1_score, accuracy_score,
                              classification_report)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available()
                         else "cpu")
PLOT_DIR = "/kaggle/working/plots"
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

class CFG3:
    # ── Input: Module 2 outputs ───────────────────────────────────────────
    SCENE_VEC_PT = ("/kaggle/input/datasets/suyashnpande/module2-outputs/m2_checkpoints"
                    "/scene_vectors.pt")

    # ── Output paths ─────────────────────────────────────────────────────
    CKPT_DIR      = "/kaggle/working/m3_checkpoints"
    BEST_CKPT     = "/kaggle/working/m3_checkpoints/narrative_lstm_best.pt"
    NARR_OUT      = "/kaggle/working/m3_checkpoints/scene_narrative.pt"

    # ── Architecture ─────────────────────────────────────────────────────
    SCENE_DIM   = 256     # input from Module 2 — must match
    HIDDEN_DIM  = 128     # LSTM hidden (→ 128d causal output)
    NARR_DIM    = 256     # fused narrative vector dimension
    CHAR_DIM    = 64      # per-character state dimension
    MAX_CHARS   = 20      # max unique characters tracked per episode
    DROPOUT     = 0.20

    # Arc phases (position-based — no external label needed)
    ARC_PHASES    = ['setup', 'rising', 'climax', 'falling', 'resolution']
    NUM_ARC       = 5

    # ── Training ─────────────────────────────────────────────────────────
    # Episodes are the training unit — each episode = one sequence of scenes.
    # MELD has ~1,152 episodes across train/dev/test splits.
    # After grouping scenes by (Season, Episode), we get episode sequences.
    BATCH_SIZE  = 16     # episodes per batch
    LR          = 2e-4
    EPOCHS      = 30
    EARLY_STOP  = 8
    MAX_WEIGHT  = 15.0   # class weight cap for arc phase CE loss

    # ── Sequence constraints ─────────────────────────────────────────────
    MAX_SCENES  = 20     # pad/truncate episodes to this many scenes
    MIN_SCENES  = 2      # skip episodes shorter than this

    # ── Split ratios ───────────────────────────────────────────────────────
    VAL_SPLIT   = 0.15
    TEST_SPLIT  = 0.15

Path(CFG3.CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("  Config ready ✓")
print(f"  SCENE_DIM={CFG3.SCENE_DIM}  HIDDEN_DIM={CFG3.HIDDEN_DIM}  "
      f"NARR_DIM={CFG3.NARR_DIM}  CHAR_DIM={CFG3.CHAR_DIM}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Label Constants                                                 │
# └──────────────────────────────────────────────────────────────────────────┘

# Arc phase labels — derived from scene position within episode
# quintile 0 = first 20% of episode → setup
# quintile 1 = 20–40%              → rising
# quintile 2 = 40–60%              → climax
# quintile 3 = 60–80%              → falling
# quintile 4 = 80–100%             → resolution
ARC_PHASES    = CFG3.ARC_PHASES
EMOTION_NAMES = ['neutral', 'joyful', 'sad', 'mad',
                 'scared', 'peaceful', 'powerful']

# Emotion valence for reversal detection
# +1 = positive, -1 = negative, 0 = neutral
VALENCE = {
    'neutral':  0,
    'joyful':   1,
    'sad':     -1,
    'mad':     -1,
    'scared':  -1,
    'peaceful': 1,
    'powerful': 0,   # dramatic — valence ambiguous
}


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Episode Grouping + Target Derivation                            │
# │                                                                          │
# │  Module 2 saved one scene per entry in scene_metadata.                 │
# │  scene_id = (Season, Episode, Dialogue_ID)                              │
# │                                                                          │
# │  Module 3 needs EPISODES: sequences of scenes from the same            │
# │  (Season, Episode) grouped in Dialogue_ID order.                       │
# │                                                                          │
# │  WHY we group by episode (not arbitrary windows):                      │
# │    A MELD "episode" = one Friends TV episode.                          │
# │    Scenes within an episode share the same narrative arc.              │
# │    The tension that builds in scene 4 resolves in scene 12.           │
# │    Randomly windowing across episode boundaries would mix arcs         │
# │    from unrelated stories — the LSTM would learn noise.               │
# └──────────────────────────────────────────────────────────────────────────┘

def group_scenes_into_episodes(scene_metadata: list) -> list:
    """
    Groups scene_metadata entries by (Season, Episode).
    Returns list of episodes, each episode = ordered list of scene dicts.
    Sorts within each episode by Dialogue_ID (ascending).
    """
    episodes = defaultdict(list)
    for meta in scene_metadata:
        sid = meta['scene_id']   # (season, episode, dialogue_id)
        ep_key = (int(sid[0]), int(sid[1]))
        episodes[ep_key].append(meta)

    # Sort each episode's scenes by dialogue_id
    episode_list = []
    for ep_key in sorted(episodes.keys()):
        scenes = sorted(episodes[ep_key],
                        key=lambda m: int(m['scene_id'][2]))
        if len(scenes) >= CFG3.MIN_SCENES:
            episode_list.append({
                'episode_id': ep_key,
                'scenes':     scenes,
            })

    print(f"  Episodes grouped: {len(episode_list):,}  "
          f"(from {len(scene_metadata):,} scenes)")

    lengths = [len(e['scenes']) for e in episode_list]
    print(f"  Episode length min/max/avg: "
          f"{min(lengths)} / {max(lengths)} / "
          f"{np.mean(lengths):.1f}")
    return episode_list


def build_char_encoder(episode_list: list) -> dict:
    """
    Maps speaker name → integer ID across all episodes.
    ID 0 is reserved for UNKNOWN / padding.
    Capped at MAX_CHARS unique characters.
    """
    all_speakers = set()
    for ep in episode_list:
        for scene in ep['scenes']:
            for spk in scene.get('speakers', []):
                all_speakers.add(str(spk))

    char2id = {'UNKNOWN': 0}
    for spk in sorted(all_speakers):
        if len(char2id) >= CFG3.MAX_CHARS:
            break
        if spk not in char2id:
            char2id[spk] = len(char2id)

    print(f"  Character encoder: {len(char2id)} unique speakers")
    print(f"  Top speakers: {list(char2id.keys())[:10]}")
    return char2id


def derive_arc_phase(scene_idx: int, total_scenes: int) -> int:
    """
    Derive arc phase label from scene position within episode.

    Uses quintile bucketing — no external annotation needed.
    A 10-scene episode:
      scenes 0-1   → setup (0)
      scenes 2-3   → rising (1)
      scenes 4-5   → climax (2)
      scenes 6-7   → falling (3)
      scenes 8-9   → resolution (4)

    This is an approximation of story structure theory (Freytag's pyramid).
    Real arcs vary — but position is a reliable proxy on average.
    """
    if total_scenes <= 1:
        return 0
    frac = scene_idx / (total_scenes - 1)   # 0.0 to 1.0
    return min(int(frac * CFG3.NUM_ARC), CFG3.NUM_ARC - 1)


def derive_tension_target(scenes: list) -> list:
    """
    Compute running tension targets for an episode's scenes.

    Tension model:
      t_0 = arousal_0
      t_i = clamp(0.85 * t_{i-1} + arousal_i - release_i, 0, 1)
    where:
      arousal_i  = scene arousal_score from Module 2   (high = adds tension)
      release_i  = max(0, -trajectory_i)              (positive traj = release)

    This gives a smooth tension curve the TensionMemoryCell learns to predict.
    """
    tension = 0.0
    targets = []
    for scene in scenes:
        arousal = float(scene.get('arousal_score',
                                  scene.get('arousal_pred', 0.0)))
        traj    = float(scene.get('trajectory_score',
                                  scene.get('trajectory_pred', 0.0)))
        # trajectory > 0 = valence improving (tension releasing)
        release = max(0.0, traj)
        tension = np.clip(0.85 * tension + arousal * 0.5 - release * 0.3,
                          0.0, 1.0)
        targets.append(float(tension))
    return targets


def derive_reversal_target(scenes: list) -> list:
    """
    Binary: did the dominant emotion's valence FLIP between consecutive scenes?

    reversal[0] = 0 (no previous scene to compare to)
    reversal[i] = 1 if sign(valence[i]) != sign(valence[i-1])

    Useful for Module 4: reversals need musical transition markers
    (hard cut, silence gap, stinger).
    """
    targets = [0.0]   # first scene has no reversal
    for i in range(1, len(scenes)):
        dom_curr = scenes[i].get('dominant_emotion', 0)
        dom_prev = scenes[i-1].get('dominant_emotion', 0)
        v_curr = VALENCE.get(EMOTION_NAMES[dom_curr], 0)
        v_prev = VALENCE.get(EMOTION_NAMES[dom_prev], 0)
        # Reversal = nonzero valences that have opposite signs
        if v_curr != 0 and v_prev != 0 and v_curr != v_prev:
            targets.append(1.0)
        else:
            targets.append(0.0)
    return targets


def encode_char_ids(speakers: list, char2id: dict,
                    k: int = 5) -> list:
    """
    Encode up to k unique speakers from a scene into integer IDs.
    Pads with -1 (excluded from CharacterMemoryBank update).
    """
    unique = list(dict.fromkeys(str(s) for s in speakers))[:k]
    ids    = [char2id.get(s, 0) for s in unique]
    ids   += [-1] * (k - len(ids))   # -1 = padding
    return ids


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — Episode Dataset                                                 │
# │                                                                          │
# │  Each training sample = one TV episode = one sequence of scenes.       │
# │  Variable-length sequences padded to MAX_SCENES.                       │
# │                                                                          │
# │  Unlike Modules 1 and 2, the LSTM here is stateful across the         │
# │  sequence — the hidden state carries information from scene to scene.  │
# └──────────────────────────────────────────────────────────────────────────┘

class EpisodeDataset(Dataset):
    """
    One sample = one episode (sequence of scenes).

    Inputs:
      scene_vectors : (MAX_SCENES, 256) — padded scene vectors
      char_ids      : (MAX_SCENES, K)   — speaker IDs per scene
      mask          : (MAX_SCENES,)     — True = real scene

    Targets (all self-supervised from Module 2 metadata):
      arc_phases    : (MAX_SCENES,)     — 5-class
      tension_targets: (MAX_SCENES,)   — float [0,1]
      reversal_targets:(MAX_SCENES,)   — float {0,1}
      emotion_seq   : (MAX_SCENES,)    — dominant emotion sequence
    """
    CHARS_PER_SCENE = 5   # how many speakers to encode per scene

    def __init__(self, episode_list: list, char2id: dict,
                 max_scenes: int = CFG3.MAX_SCENES):
        self.samples = []
        skipped      = 0

        for ep in tqdm(episode_list,
                       desc="  Building EpisodeDataset",
                       leave=False):
            scenes = ep['scenes']
            S      = len(scenes)
            if S < CFG3.MIN_SCENES:
                skipped += 1
                continue

            S_eff = min(S, max_scenes)   # clamp long episodes

            # ── Scene vectors ──────────────────────────────────────────
            vecs = []
            for scene in scenes[:S_eff]:
                sv = scene.get('scene_vector')
                if sv is None:
                    sv = torch.zeros(CFG3.SCENE_DIM)
                elif not isinstance(sv, torch.Tensor):
                    sv = torch.tensor(sv, dtype=torch.float32)
                vecs.append(sv.float())

            # Pad to max_scenes
            pad_vec = torch.zeros(CFG3.SCENE_DIM)
            while len(vecs) < max_scenes:
                vecs.append(pad_vec)
            scene_mat = torch.stack(vecs)   # (max_scenes, 256)

            # ── Mask ──────────────────────────────────────────────────
            mask = torch.cat([
                torch.ones(S_eff, dtype=torch.bool),
                torch.zeros(max_scenes - S_eff, dtype=torch.bool)])

            # ── Character IDs per scene ────────────────────────────────
            char_mat = []
            for scene in scenes[:S_eff]:
                ids = encode_char_ids(
                    scene.get('speakers', []),
                    char2id,
                    k=self.CHARS_PER_SCENE)
                char_mat.append(ids)
            # Pad with all -1 for padding scenes
            pad_chars = [-1] * self.CHARS_PER_SCENE
            while len(char_mat) < max_scenes:
                char_mat.append(pad_chars)
            char_tensor = torch.tensor(
                char_mat, dtype=torch.long)  # (max_scenes, K)

            # ── Derive targets ─────────────────────────────────────────
            real_scenes     = scenes[:S_eff]
            arc_list        = [derive_arc_phase(i, S_eff)
                               for i in range(S_eff)]
            tension_list    = derive_tension_target(real_scenes)
            reversal_list   = derive_reversal_target(real_scenes)
            emotion_seq_list= [s.get('dominant_emotion', 0)
                               for s in real_scenes]

            # Pad targets with zeros
            def _pad(lst, val=0):
                return lst + [val] * (max_scenes - len(lst))

            arc_tensor     = torch.tensor(
                _pad(arc_list), dtype=torch.long)
            tension_tensor = torch.tensor(
                _pad(tension_list, 0.0), dtype=torch.float32)
            reversal_tensor= torch.tensor(
                _pad(reversal_list, 0.0), dtype=torch.float32)
            emotion_tensor = torch.tensor(
                _pad(emotion_seq_list), dtype=torch.long)

            # ── Scene-level auxiliary signals ──────────────────────────
            arousal_seq = torch.tensor(
                _pad([float(s.get('arousal_score',
                                  s.get('arousal_pred', 0.0)))
                      for s in real_scenes], 0.0),
                dtype=torch.float32)
            conflict_seq= torch.tensor(
                _pad([float(s.get('conflict_score',
                                  s.get('conflict_pred', 0.0)))
                      for s in real_scenes], 0.0),
                dtype=torch.float32)
            traj_seq    = torch.tensor(
                _pad([float(s.get('trajectory_score',
                                  s.get('trajectory_pred', 0.0)))
                      for s in real_scenes], 0.0),
                dtype=torch.float32)

            self.samples.append({
                # Inputs
                'scene_vectors':    scene_mat,     # (S, 256)
                'char_ids':         char_tensor,   # (S, K)
                'mask':             mask,          # (S,)
                'episode_id':       ep['episode_id'],
                'num_scenes':       S_eff,
                # Targets
                'arc_phases':       arc_tensor,    # (S,)
                'tension_targets':  tension_tensor,# (S,)
                'reversal_targets': reversal_tensor,# (S,)
                'emotion_seq':      emotion_tensor,# (S,)
                # Auxiliary continuous signals
                'arousal_seq':      arousal_seq,   # (S,)
                'conflict_seq':     conflict_seq,  # (S,)
                'traj_seq':         traj_seq,      # (S,)
            })

        print(f"  Episodes built : {len(self.samples):,}  "
              f"(skipped {skipped})")

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx): return self.samples[idx]


def collate_episodes(batch: list) -> dict:
    return {
        'scene_vectors':    torch.stack([b['scene_vectors']    for b in batch]),
        'char_ids':         torch.stack([b['char_ids']         for b in batch]),
        'mask':             torch.stack([b['mask']             for b in batch]),
        'arc_phases':       torch.stack([b['arc_phases']       for b in batch]),
        'tension_targets':  torch.stack([b['tension_targets']  for b in batch]),
        'reversal_targets': torch.stack([b['reversal_targets'] for b in batch]),
        'emotion_seq':      torch.stack([b['emotion_seq']      for b in batch]),
        'arousal_seq':      torch.stack([b['arousal_seq']      for b in batch]),
        'conflict_seq':     torch.stack([b['conflict_seq']     for b in batch]),
        'traj_seq':         torch.stack([b['traj_seq']         for b in batch]),
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — TensionMemoryCell                                               │
# │                                                                          │
# │  Models how narrative pressure accumulates and decays across scenes.    │
# │                                                                          │
# │  Learned gate mechanism:                                                │
# │    γ (decay)   — how much tension carries over from the last scene     │
# │    α (arousal) — how much this scene ADDS tension                      │
# │    ρ (release) — how much this scene RELEASES tension                  │
# │                                                                          │
# │  t_new = clamp(γ·t_prev + α - ρ,  0, 1)                               │
# │                                                                          │
# │  All three gates are learned from the scene vector — the model          │
# │  discovers from data which scene types add vs release tension.          │
# │  E.g. "mad scenes add tension" and "peaceful scenes release it"        │
# │  emerge naturally without being hard-coded.                            │
# └──────────────────────────────────────────────────────────────────────────┘

class TensionMemoryCell(nn.Module):
    def __init__(self, scene_dim: int = CFG3.SCENE_DIM):
        super().__init__()
        # γ ∈ (0,1): how much prior tension persists
        # High γ → tension accumulates across many scenes (thriller pacing)
        # Low γ  → tension resets per scene (comedy pacing)
        self.decay_gate = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())

        # α ∈ (0,1): how much THIS scene adds to tension
        self.arousal_gate = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())

        # ρ ∈ (0,1): how much THIS scene releases tension
        self.release_gate = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid())

    def forward(self,
                scene_vector: torch.Tensor,   # (B, 256)
                prev_tension: torch.Tensor    # (B, 1)
                ) -> torch.Tensor:            # (B, 1)
        gamma   = self.decay_gate(scene_vector)
        arousal = self.arousal_gate(scene_vector)
        release = self.release_gate(scene_vector)
        tension = gamma * prev_tension + arousal - release
        return torch.clamp(tension, 0.0, 1.0)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — CharacterMemoryBank                                             │
# │                                                                          │
# │  Maintains a 64d emotional state per character, updated every time      │
# │  that character appears in a scene.                                     │
# │                                                                          │
# │  Motivation: a film composer tracks each character's emotional arc.     │
# │  Ross has been increasingly sad for 5 scenes → even his neutral scene  │
# │  should have sad-coloured music. The character memory bank enables     │
# │  this "emotional continuity per character" reasoning.                  │
# │                                                                          │
# │  Implementation uses a GRUCell per update — lightweight and avoids     │
# │  the inner-loop overhead of a full GRU over variable-length sequences. │
# └──────────────────────────────────────────────────────────────────────────┘

class CharacterMemoryBank(nn.Module):
    def __init__(self,
                 scene_dim: int = CFG3.SCENE_DIM,
                 char_dim:  int = CFG3.CHAR_DIM,
                 max_chars: int = CFG3.MAX_CHARS):
        super().__init__()
        self.char_dim  = char_dim
        self.max_chars = max_chars

        # GRUCell: updates one character's memory with the scene vector
        self.gru_cell = nn.GRUCell(
            input_size  = scene_dim,
            hidden_size = char_dim)

        # Project aggregated character context to fixed dim
        self.proj = nn.Linear(char_dim, char_dim)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self,
                scene_vector: torch.Tensor,   # (B, 256)
                char_ids:     torch.Tensor,   # (B, K) — -1 = padding
                char_memory:  torch.Tensor    # (B, max_chars, 64)
                ) -> tuple:
        """
        Returns:
          updated_memory : (B, max_chars, 64)
          char_context   : (B, 64) — mean of active character states
        """
        B, K      = char_ids.shape
        updated   = char_memory.clone()

        # Update each character slot that appears in this scene
        for k in range(K):
            ids   = char_ids[:, k]           # (B,)
            valid = (ids >= 0)               # not padding
            if not valid.any():
                continue

            # Clamp IDs to [0, max_chars-1] for index safety
            safe_ids = ids.clamp(0, self.max_chars - 1)

            # Gather current memory for these chars: (B, 64)
            curr_mem = char_memory[
                torch.arange(B, device=scene_vector.device),
                safe_ids]

            # GRU update: new_mem = GRU(scene_vector, curr_mem)
            new_mem = self.gru_cell(scene_vector, curr_mem)  # (B, 64)

            # Write back only for valid (non-padding) characters
            valid_b = valid.nonzero(as_tuple=True)[0]
            for b in valid_b:
                updated[b, safe_ids[b]] = new_mem[b].detach()

        # Character context = mean of all active character memories
        # Active = any slot that is NOT the zero initial state
        # We approximate: use IDs ≥ 0 across all K slots
        active_mask = (char_ids >= 0).float()       # (B, K)
        safe_all    = char_ids.clamp(0, self.max_chars - 1)  # (B, K)

        # Gather: (B, K, 64)
        active_mems = updated[
            torch.arange(B, device=scene_vector.device)
                 .unsqueeze(1)
                 .expand(B, K),
            safe_all]

        # Weighted mean (ignore padding)
        denom     = active_mask.sum(dim=1, keepdim=True) + 1e-6   # (B, 1)
        char_ctx  = (active_mems *
                     active_mask.unsqueeze(-1)).sum(dim=1) / denom  # (B, 64)

        return updated, self.proj(char_ctx)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — NarrativeLSTM                                                   │
# │                                                                          │
# │  The three mechanisms are combined step-by-step (not in parallel).      │
# │  For each scene t:                                                       │
# │    1. Causal LSTM processes scene_vector → 128d hidden state            │
# │    2. TensionMemoryCell updates tension scalar                          │
# │    3. CharacterMemoryBank updates character states → 64d context        │
# │    4. Fusion: LSTM(128) + tension(1) + char(64) → 256d narr_vector     │
# │    5. Prediction heads: arc_phase, reversal_score, emotion_continuity   │
# └──────────────────────────────────────────────────────────────────────────┘

class NarrativeLSTM(nn.Module):
    def __init__(self,
                 scene_dim  = CFG3.SCENE_DIM,
                 hidden_dim = CFG3.HIDDEN_DIM,
                 narr_dim   = CFG3.NARR_DIM,
                 char_dim   = CFG3.CHAR_DIM,
                 max_chars  = CFG3.MAX_CHARS,
                 num_arc    = CFG3.NUM_ARC,
                 num_emotions = 7,
                 dropout    = CFG3.DROPOUT):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.char_dim   = char_dim
        self.max_chars  = max_chars

        # ── 1. Causal LSTM over scene sequence ────────────────────────
        # batch_first=True: input (B, S, 256) → output (B, S, 128)
        # 2 layers for richer representation; NOT bidirectional (causal)
        self.lstm = nn.LSTM(
            input_size  = scene_dim,
            hidden_size = hidden_dim,
            num_layers  = 2,
            batch_first = True,
            bidirectional=False,
            dropout     = dropout)

        # ── 2. TensionMemoryCell ──────────────────────────────────────
        self.tension_cell = TensionMemoryCell(scene_dim)

        # ── 3. CharacterMemoryBank ────────────────────────────────────
        self.char_bank = CharacterMemoryBank(
            scene_dim, char_dim, max_chars)

        # ── 4. Fusion: LSTM(128) + tension(1) + char(64) = 193 → 256 ─
        fuse_in = hidden_dim + 1 + char_dim   # 128+1+64 = 193
        self.fusion = nn.Sequential(
            nn.Linear(fuse_in, narr_dim),
            nn.LayerNorm(narr_dim),
            nn.ReLU(),
            nn.Dropout(dropout))

        # ── 5. Prediction heads ────────────────────────────────────────
        # Arc phase (5-class: setup/rising/climax/falling/resolution)
        self.arc_head = nn.Sequential(
            nn.Linear(narr_dim, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_arc))

        # Reversal detector: did emotion direction flip? (binary)
        self.reversal_head = nn.Sequential(
            nn.Linear(narr_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        # Tension prediction: what is the current tension? (regression)
        self.tension_head = nn.Sequential(
            nn.Linear(narr_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        # Emotion continuity: predict next scene's dominant emotion
        # Auxiliary task — forces LSTM to track emotional trajectory
        self.emotion_head = nn.Sequential(
            nn.Linear(narr_dim, 64), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_emotions))

    def forward(self,
                scene_vectors: torch.Tensor,  # (B, S, 256)
                char_ids:      torch.Tensor,  # (B, S, K)
                ) -> dict:
        """
        Processes an episode's scene sequence causally.

        Returns per-scene outputs for all S positions.
        All outputs respect causality: position t only uses 0..t.
        """
        B, S, _   = scene_vectors.shape

        # ── Full causal LSTM pass (efficient: all S at once) ──────────
        lstm_out, _ = self.lstm(scene_vectors)  # (B, S, 128)

        # ── Per-scene sequential pass for tension + character bank ────
        # Cannot vectorise these — they depend on previous timestep state
        tension   = torch.zeros(B, 1, device=scene_vectors.device)
        char_mem  = torch.zeros(B, self.max_chars, self.char_dim,
                                device=scene_vectors.device)

        narr_vecs    = []
        tension_preds= []
        arc_list     = []
        reversal_list= []
        emotion_list = []

        for t in range(S):
            sv   = scene_vectors[:, t, :]  # (B, 256) current scene
            lh   = lstm_out[:, t, :]       # (B, 128) causal LSTM output
            cids = char_ids[:, t, :]       # (B, K)

            # Update tension (depends on previous tension)
            tension = self.tension_cell(sv, tension)   # (B, 1)

            # Update character memories
            char_mem, char_ctx = self.char_bank(
                sv, cids, char_mem)                    # char_ctx: (B, 64)

            # Fuse: LSTM + tension + char_context → narrative vector
            fused = self.fusion(
                torch.cat([lh, tension, char_ctx],
                           dim=-1))                    # (B, 256)

            narr_vecs.append(fused)
            tension_preds.append(tension)

            # Prediction heads
            arc_list.append(self.arc_head(fused))      # (B, 5)
            reversal_list.append(
                self.reversal_head(fused))             # (B, 1)
            emotion_list.append(
                self.emotion_head(fused))              # (B, 7)

        # Stack timesteps → (B, S, dim)
        narr_out     = torch.stack(narr_vecs,     dim=1)  # (B,S,256)
        tension_out  = torch.stack(tension_preds, dim=1)  # (B,S,1)
        arc_out      = torch.stack(arc_list,      dim=1)  # (B,S,5)
        reversal_out = torch.stack(reversal_list, dim=1)  # (B,S,1)
        emotion_out  = torch.stack(emotion_list,  dim=1)  # (B,S,7)

        return {
            'narrative_vectors': narr_out,         # → Module 4
            'tension_scores':    tension_out,      # → Module 4
            'arc_logits':        arc_out,          # trained on arc_phases
            'reversal_scores':   reversal_out,     # trained on reversal targets
            'emotion_logits':    emotion_out,      # auxiliary next-emotion pred
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Loss Function                                                   │
# │                                                                          │
# │  All losses are masked — padding positions (mask=False) excluded.       │
# │                                                                          │
# │  Loss terms:                                                             │
# │    arc_phase CE     ×1.0  — main task: where in story arc?              │
# │    emotion CE       ×0.5  — next-scene emotion prediction               │
# │    tension MSE      ×0.4  — tension tracking                            │
# │    reversal BCE     ×0.3  — reversal detection                          │
# └──────────────────────────────────────────────────────────────────────────┘

class NarrativeLoss(nn.Module):
    def __init__(self, arc_weights=None):
        super().__init__()
        self.arc_ce  = nn.CrossEntropyLoss(
            weight=arc_weights, label_smoothing=0.05,
            reduction='none')
        self.emo_ce  = nn.CrossEntropyLoss(
            label_smoothing=0.05, reduction='none')
        self.mse     = nn.MSELoss(reduction='none')
        self.bce     = nn.BCELoss(reduction='none')

    def _masked_mean(self, loss_per_elem: torch.Tensor,
                     mask: torch.Tensor) -> torch.Tensor:
        """Mean over real (non-padding) positions only."""
        m = mask.float()
        return (loss_per_elem * m).sum() / (m.sum() + 1e-6)

    def forward(self, preds: dict, batch: dict) -> dict:
        D    = DEVICE
        mask = batch['mask'].to(D)          # (B, S)
        B, S = mask.shape

        # Flatten to (B*S, ...) for loss computation
        arc_t   = batch['arc_phases'].to(D).view(B * S)
        emo_t   = batch['emotion_seq'].to(D).view(B * S)
        ten_t   = batch['tension_targets'].to(D).view(B * S)
        rev_t   = batch['reversal_targets'].to(D).view(B * S)
        mf      = mask.view(B * S)          # (B*S,) flat mask

        arc_p   = preds['arc_logits'].view(B * S, -1)
        emo_p   = preds['emotion_logits'].view(B * S, -1)
        ten_p   = preds['tension_scores'].view(B * S)
        rev_p   = preds['reversal_scores'].view(B * S)

        # Masked losses
        l_arc = self._masked_mean(self.arc_ce(arc_p, arc_t), mf)
        l_emo = self._masked_mean(self.emo_ce(emo_p, emo_t), mf)
        l_ten = self._masked_mean(self.mse(ten_p, ten_t), mf)
        l_rev = self._masked_mean(self.bce(rev_p, rev_t), mf)

        total = (1.0 * l_arc +
                 0.5 * l_emo +
                 0.4 * l_ten +
                 0.3 * l_rev)

        return {
            'total':   total,
            'arc':     l_arc.item(),
            'emo':     l_emo.item(),
            'tension': l_ten.item(),
            'reversal':l_rev.item(),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Train / Eval Loop                                              │
# └──────────────────────────────────────────────────────────────────────────┘

def run_epoch(model, loader, loss_fn,
              optimizer=None, phase='train',
              epoch=0, total=0) -> tuple:
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()

    tot_loss = 0.0
    lp       = defaultdict(float)
    all_t, all_p = [], []
    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=(f"Ep {epoch+1:02d}/{total} "
              f"{'▶ Train' if is_train else '  Val  '}"),
        leave=True,
        bar_format="{l_bar}{bar:25}{r_bar}")

    for step, batch in enumerate(pbar):
        sv   = batch['scene_vectors'].to(DEVICE)   # (B, S, 256)
        cids = batch['char_ids'].to(DEVICE)        # (B, S, K)
        mask = batch['mask'].to(DEVICE)            # (B, S)

        with torch.set_grad_enabled(is_train):
            preds  = model(sv, cids)
            losses = loss_fn(preds, batch)

        if is_train:
            optimizer.zero_grad()
            losses['total'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # Track arc_phase F1 (main metric)
        B, S  = mask.shape
        mf    = mask.view(B * S).cpu()
        arc_t = batch['arc_phases'].view(B * S)
        arc_p = preds['arc_logits'].view(B * S, -1).argmax(-1).detach().cpu()
        # arc_t_np = arc_t.numpy()
        # arc_p_np = arc_p.numpy()
        # real_mask = mf.numpy().astype(bool)
        arc_t_np = arc_t.cpu().numpy()
        arc_p_np = arc_p.cpu().numpy()
        real_mask = mf.numpy().astype(bool)
        all_t.extend(arc_t_np[real_mask].tolist())
        all_p.extend(arc_p_np[real_mask].tolist())

        tot_loss += losses['total'].item()
        for k, v in losses.items():
            if k != 'total':
                lp[k] += (v if isinstance(v, float) else v.item())

        if (step + 1) % 5 == 0:
            f1 = f1_score(all_t, all_p, average='macro',
                          zero_division=0) * 100
            pbar.set_postfix(
                loss=f"{tot_loss/(step+1):.3f}",
                arc_f1=f"{f1:.1f}%")

    n    = len(loader)
    avg  = tot_loss / n
    f1   = f1_score(all_t, all_p, average='macro', zero_division=0)
    acc  = accuracy_score(all_t, all_p)
    m, s = divmod(int(time.time() - t0), 60)

    print(f"    ↳ {phase.upper():5s}  loss={avg:.4f}  "
          f"[arc={lp['arc']/n:.3f}  "
          f"ten={lp['tension']/n:.3f}  "
          f"rev={lp['reversal']/n:.3f}]  "
          f"arc_f1={f1*100:.2f}%  ({m}m{s:02d}s)")

    return avg, {'macro_f1': f1, 'accuracy': acc, 'loss': avg}


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — Plots                                                          │
# └──────────────────────────────────────────────────────────────────────────┘

DARK_BG  = '#0f0f1a'
PANEL_BG = '#1a1a2e'
ACCENT   = ['#2ecc71','#e74c3c','#f39c12',
            '#3498db','#9b59b6','#1abc9c','#e67e22']


def _ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors='#cccccc', labelsize=8)
    ax.set_title(title, color='white',
                 fontsize=9, fontweight='bold', pad=5)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#333355')


def plot_m3_training(history: dict):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 3 — Narrative LSTM Training',
                 color='white', fontsize=11, fontweight='bold')

    ep_tr = [h['epoch'] for h in history['train']]
    ep_va = [h['epoch'] for h in history['val']]

    axes[0].plot(ep_tr, [h['loss'] for h in history['train']],
                 color='#3498db', lw=2, marker='o', ms=3, label='Train')
    axes[0].plot(ep_va, [h['loss'] for h in history['val']],
                 color='#e74c3c', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[0].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    _ax(axes[0], 'Loss', 'Epoch', 'Loss')

    axes[1].plot(ep_tr, [h['f1']*100 for h in history['train']],
                 color='#2ecc71', lw=2, marker='o', ms=3, label='Train')
    axes[1].plot(ep_va, [h['f1']*100 for h in history['val']],
                 color='#f39c12', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[1].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    axes[1].set_ylim(0, 100)
    _ax(axes[1], 'Arc Phase Macro F1', 'Epoch', 'Macro F1 (%)')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m3_01_training_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m3_01_training_curves.png")


def plot_tension_curves(all_narr: list, n_eps: int = 4):
    """
    Show predicted tension curves for the first N episodes.
    Each plot: tension score per scene with arc phase colour-coded.
    """
    samples = all_narr[:n_eps]
    fig, axes = plt.subplots(len(samples), 1,
                             figsize=(14, 3 * len(samples)))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 3 — Tension Curves per Episode',
                 color='white', fontsize=10, fontweight='bold')

    if len(samples) == 1:
        axes = [axes]

    phase_colors = {0:'#3498db', 1:'#f39c12', 2:'#e74c3c',
                    3:'#9b59b6', 4:'#2ecc71'}

    for row, ep in enumerate(samples):
        ax    = axes[row]
        tens  = ep['tension_scores']     # list of floats
        phases= ep['arc_phases']         # list of ints
        S     = len(tens)

        for i in range(S - 1):
            c = phase_colors.get(phases[i], '#aaaaaa')
            ax.fill_between([i, i+1],
                            [tens[i], tens[i+1]],
                            alpha=0.3, color=c)
            ax.plot([i, i+1], [tens[i], tens[i+1]],
                    color=c, lw=2)

        ax.scatter(range(S), tens, c=[
            phase_colors.get(p, '#aaaaaa') for p in phases],
                   s=40, zorder=5)

        # Legend
        for ph_i, ph_n in enumerate(ARC_PHASES):
            ax.plot([], [], color=phase_colors[ph_i],
                    label=ph_n, lw=3)
        ax.legend(fontsize=7, facecolor=PANEL_BG,
                  labelcolor='white', ncol=5)
        ax.set_ylim(0, 1.1)
        _ax(ax, f"Episode {ep['episode_id']} — Tension Arc",
            'Scene', 'Tension')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m3_02_tension_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m3_02_tension_curves.png")


def plot_arc_phase_distribution(dataset: EpisodeDataset):
    """Bar chart of arc phase label distribution in training data."""
    all_phases = []
    for s in dataset.samples:
        mask = s['mask'].numpy()
        arcs = s['arc_phases'].numpy()
        all_phases.extend(arcs[mask].tolist())

    counts = Counter(all_phases)
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor(DARK_BG)
    vals = [counts.get(i, 0) for i in range(CFG3.NUM_ARC)]
    bars = ax.bar(ARC_PHASES, vals, color=ACCENT[:5],
                  alpha=0.85, edgecolor='white', linewidth=0.5)
    total = sum(vals)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 5,
                f'{v/total*100:.0f}%',
                ha='center', va='bottom',
                color='white', fontsize=9)
    _ax(ax, 'Arc Phase Distribution', 'Arc Phase', 'Count')
    plt.tight_layout()
    p = f"{PLOT_DIR}/m3_00_arc_distribution.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m3_00_arc_distribution.png")


def plot_narrative_vectors(all_narr: list, n: int = 40):
    """Heatmap of narrative vectors coloured by arc phase."""
    vecs   = torch.stack([m['narrative_vectors'][0]
                          for m in all_narr[:n]])
    phases = [m['arc_phases'][0] for m in all_narr[:n]]
    labels = [ARC_PHASES[p] for p in phases]

    v_norm = (vecs - vecs.mean(0)) / (vecs.std(0) + 1e-6)
    fig, ax = plt.subplots(figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    im = ax.imshow(v_norm.numpy(), aspect='auto',
                   cmap='RdBu_r', vmin=-2, vmax=2)
    plt.colorbar(im, ax=ax, label='Normalised')
    ax.set_yticks(range(n))
    ax.set_yticklabels([f'{i}:{l}' for i, l in enumerate(labels)],
                       fontsize=6, color='white')
    ax.set_xlabel('Narrative Vector Dimension', color='#aaaaaa')
    _ax(ax, f'Narrative Vectors — first {n} episodes')
    ax.tick_params(colors='#cccccc')
    plt.tight_layout()
    p = f"{PLOT_DIR}/m3_03_narrative_vectors.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m3_03_narrative_vectors.png")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — Load Data + Build Dataset                                      │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "=" * 60)
print("  LOADING MODULE 2 OUTPUTS")
print("=" * 60)

m2_data = torch.load(CFG3.SCENE_VEC_PT, map_location='cpu',weights_only=False)
scene_metadata  = m2_data['scene_metadata']
CLASS_NAMES     = m2_data.get('class_names',
                               ['neutral','joyful','sad','mad',
                                'scared','peaceful','powerful'])
SCENE_TYPE_NAMES= m2_data.get('scene_type_names',
                               ['dialogue','action','investigation',
                                'suspense','dramatic_reveal','transition'])

print(f"  Loaded {len(scene_metadata):,} scenes")
print(f"  Emotion classes : {CLASS_NAMES}")

# Verify scene_vector dimension
sv_dim = scene_metadata[0]['scene_vector'].shape[0]
if sv_dim != CFG3.SCENE_DIM:
    print(f"  ⚠ scene_vector dim={sv_dim}, "
          f"expected {CFG3.SCENE_DIM} — adjusting")
    CFG3.SCENE_DIM = sv_dim
else:
    print(f"  ✓ scene_vector dim = {sv_dim}")

# Group scenes into episodes
episode_list = group_scenes_into_episodes(scene_metadata)

# Build character encoder
char2id = build_char_encoder(episode_list)

# Build dataset
full_dataset = EpisodeDataset(episode_list, char2id)

# Stratified split on first arc phase of each episode
all_idx  = list(range(len(full_dataset)))
# Use dominant emotion of first scene as stratify key
strat_key= [full_dataset.samples[i]['arc_phases'][0].item()
            for i in all_idx]

tr_va, te = train_test_split(
    all_idx, test_size=CFG3.TEST_SPLIT,
    stratify=strat_key, random_state=SEED)
tr_va_key = [strat_key[i] for i in tr_va]
tr, va = train_test_split(
    tr_va,
    test_size=CFG3.VAL_SPLIT / (1.0 - CFG3.TEST_SPLIT),
    stratify=tr_va_key, random_state=SEED)

print(f"\n  Split: train={len(tr)}  val={len(va)}  test={len(te)}")

_kw = dict(collate_fn=collate_episodes,
           num_workers=2, pin_memory=True)
train_loader = DataLoader(
    Subset(full_dataset, tr),
    CFG3.BATCH_SIZE, shuffle=True,  **_kw)
val_loader   = DataLoader(
    Subset(full_dataset, va),
    CFG3.BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(
    Subset(full_dataset, te),
    CFG3.BATCH_SIZE, shuffle=False, **_kw)

# Arc phase class weights (position-based → roughly uniform)
tr_phases = []
for i in tr:
    s    = full_dataset.samples[i]
    mask = s['mask'].numpy()
    tr_phases.extend(
        s['arc_phases'].numpy()[mask].tolist())

counts  = np.bincount(tr_phases,
                      minlength=CFG3.NUM_ARC).astype(float)
counts  = np.where(counts == 0, 1, counts)
raw_w   = len(tr_phases) / (CFG3.NUM_ARC * counts)
clamped = np.clip(raw_w, 0.1, CFG3.MAX_WEIGHT)
arc_weights = torch.tensor(clamped, dtype=torch.float32).to(DEVICE)

print(f"\n  Arc phase weights:")
for i, (ph, rw, cw) in enumerate(
        zip(ARC_PHASES, raw_w, clamped)):
    flag = ' ← CLAMPED' if rw > CFG3.MAX_WEIGHT else ''
    print(f"    {ph:<12}  count={int(counts[i]):>5}  "
          f"weight={cw:.2f}{flag}")

# Visualise arc distribution
plot_arc_phase_distribution(full_dataset)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Build Model + Train                                            │
# └──────────────────────────────────────────────────────────────────────────┘

model3 = NarrativeLSTM(
    scene_dim   = CFG3.SCENE_DIM,
    hidden_dim  = CFG3.HIDDEN_DIM,
    narr_dim    = CFG3.NARR_DIM,
    char_dim    = CFG3.CHAR_DIM,
    max_chars   = CFG3.MAX_CHARS,
    num_arc     = CFG3.NUM_ARC,
    num_emotions= len(CLASS_NAMES),
    dropout     = CFG3.DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model3.parameters())
print(f"\n  NarrativeLSTM params: {total_params:,}")

loss_fn3   = NarrativeLoss(arc_weights=arc_weights)
optimizer3 = optim.Adam(
    model3.parameters(), lr=CFG3.LR, weight_decay=1e-5)
scheduler3 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer3, mode='min', patience=3,
    factor=0.5, min_lr=1e-6)

print("\n" + "═" * 60)
print("  TRAINING NARRATIVE LSTM")
print(f"  Input  : 256d scene vectors from Module 2")
print(f"  Model  : Causal LSTM + TensionCell + CharacterBank")
print(f"  Train  : {len(tr)} episodes  "
      f"Val : {len(va)} episodes  "
      f"Test : {len(te)} episodes")
print(f"  Epochs {CFG3.EPOCHS}  Batch {CFG3.BATCH_SIZE}  LR {CFG3.LR}")
print("═" * 60)

history    = {'train': [], 'val': []}
best_f1    = 0.0
patience_c = 0

for epoch in range(CFG3.EPOCHS):
    tr_l, tr_m = run_epoch(
        model3, train_loader, loss_fn3,
        optimizer3, 'train', epoch, CFG3.EPOCHS)
    va_l, va_m = run_epoch(
        model3, val_loader, loss_fn3,
        None, 'val', epoch, CFG3.EPOCHS)

    scheduler3.step(va_l)

    history['train'].append({
        'epoch': epoch + 1, 'loss': tr_l,
        'f1':   tr_m['macro_f1']})
    history['val'].append({
        'epoch': epoch + 1, 'loss': va_l,
        'f1':   va_m['macro_f1']})

    if va_m['macro_f1'] > best_f1:
        best_f1    = va_m['macro_f1']
        patience_c = 0
        torch.save({
            'model_state':  model3.state_dict(),
            'macro_f1':     best_f1,
            'char2id':      char2id,
            'config': {
                'scene_dim':   CFG3.SCENE_DIM,
                'hidden_dim':  CFG3.HIDDEN_DIM,
                'narr_dim':    CFG3.NARR_DIM,
                'char_dim':    CFG3.CHAR_DIM,
                'max_chars':   CFG3.MAX_CHARS,
                'num_arc':     CFG3.NUM_ARC,
                'num_emotions':len(CLASS_NAMES),
                'dropout':     CFG3.DROPOUT,
                'arc_phases':  ARC_PHASES,
                'class_names': CLASS_NAMES,
            },
        }, CFG3.BEST_CKPT)
        print(f"    ✓ Saved narrative_lstm_best.pt "
              f"(arc_f1={best_f1*100:.2f}%)")
    else:
        patience_c += 1
        if patience_c >= CFG3.EARLY_STOP:
            print(f"\n  ⏹ Early stop at epoch {epoch + 1}")
            break

plot_m3_training(history)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 14 — Final Evaluation                                               │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 60)
print("  FINAL EVALUATION — TEST SET")
print("═" * 60)

ckpt = torch.load(CFG3.BEST_CKPT, map_location=DEVICE)
model3.load_state_dict(ckpt['model_state'])
model3.eval()

all_t, all_p = [], []
with torch.no_grad():
    for batch in test_loader:
        sv   = batch['scene_vectors'].to(DEVICE)
        cids = batch['char_ids'].to(DEVICE)
        mask = batch['mask']

        out  = model3(sv, cids)
        B, S = mask.shape
        mf   = mask.view(B * S).numpy().astype(bool)
        arc_t= batch['arc_phases'].view(B * S).numpy()
        arc_p= out['arc_logits'].view(B * S, -1).argmax(-1).cpu().numpy()
        all_t.extend(arc_t[mf].tolist())
        all_p.extend(arc_p[mf].tolist())

test_f1  = f1_score(all_t, all_p, average='macro', zero_division=0)
test_acc = accuracy_score(all_t, all_p)

print(f"\n  Test Arc Phase Macro F1 : {test_f1*100:.2f}%")
print(f"  Test Accuracy           : {test_acc*100:.2f}%")
print(f"  Best Val F1             : {best_f1*100:.2f}%")
print(f"\n  Classification Report (Arc Phases):")
print(classification_report(
    all_t, all_p,
    target_names=ARC_PHASES,
    labels=list(range(CFG3.NUM_ARC)),
    zero_division=0))


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 15 — Extract Narrative Vectors → Module 4                           │
# │                                                                          │
# │  Run the trained model over ALL episodes.                               │
# │  For each scene, save:                                                  │
# │    narrative_vector : (256,)  → Module 4 main input                    │
# │    tension_score    : float   → Module 4 tempo/dynamics                │
# │    arc_phase        : int     → Module 4 transition style              │
# │    reversal_score   : float   → Module 4 hit point marker             │
# │    char_context     : None    → embedded in narrative_vector           │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 60)
print("  EXTRACTING NARRATIVE VECTORS → MODULE 4")
print("═" * 60)

full_loader = DataLoader(
    full_dataset, batch_size=16,
    collate_fn=collate_episodes,
    shuffle=False, num_workers=2)

all_narr = []   # list of episode-level dicts

with torch.no_grad():
    for i, batch in enumerate(tqdm(full_loader,
                                    desc="  Extracting")):
        sv   = batch['scene_vectors'].to(DEVICE)
        cids = batch['char_ids'].to(DEVICE)
        out  = model3(sv, cids)
        B    = sv.size(0)

        for b in range(B):
            ep_idx = i * 16 + b
            if ep_idx >= len(full_dataset):
                break
            sample   = full_dataset.samples[ep_idx]
            S_real   = sample['num_scenes']
            mask_b   = sample['mask'].numpy()

            # Extract only REAL (non-padding) scenes
            narr_v   = out['narrative_vectors'][b, :S_real].cpu()   # (S, 256)
            tens_v   = out['tension_scores'][b, :S_real, 0].cpu()   # (S,)
            arc_pred = out['arc_logits'][b, :S_real].argmax(-1).cpu()
            rev_pred = out['reversal_scores'][b, :S_real, 0].cpu()  # (S,)

            all_narr.append({
                'episode_id':        sample['episode_id'],
                # Per-scene outputs (lists of length S_real)
                'narrative_vectors': narr_v,        # (S, 256) → M4
                'tension_scores':    tens_v.tolist(),
                'arc_phases':        arc_pred.tolist(),
                'reversal_scores':   rev_pred.tolist(),
                # Ground truth for verification
                'arc_phases_true':   sample['arc_phases'][:S_real].tolist(),
                'emotion_seq':       sample['emotion_seq'][:S_real].tolist(),
                # Linking key for Module 4 to join with scene_vectors.pt
                'scene_ids': [
                    ep['scenes'][j]['scene_id']
                    for ep in [episode_list[ep_idx]]
                    for j in range(S_real)
                ] if ep_idx < len(episode_list) else [],
                'num_scenes':        S_real,
            })

# Generate plots
plot_tension_curves(all_narr, n_eps=4)
plot_narrative_vectors(all_narr, n=40)

# Save to disk
torch.save({
    'episode_narratives': all_narr,
    'num_episodes':       len(all_narr),
    'narr_dim':           CFG3.NARR_DIM,
    'arc_phases':         ARC_PHASES,
    'class_names':        CLASS_NAMES,
    'char2id':            char2id,
}, CFG3.NARR_OUT)

with open(f"{CFG3.CKPT_DIR}/m3_history.json", 'w') as f:
    json.dump(history, f, indent=2)

# Summary statistics
all_tensions  = [t for ep in all_narr
                 for t in ep['tension_scores']]
all_reversals = [r for ep in all_narr
                 for r in ep['reversal_scores']]

print(f"\n  ✓ scene_narrative.pt")
print(f"    Episodes  : {len(all_narr):,}")
print(f"    Total scenes in output: "
      f"{sum(ep['num_scenes'] for ep in all_narr):,}")
print(f"    Tension   mean={np.mean(all_tensions):.3f}  "
      f"std={np.std(all_tensions):.3f}")
print(f"    Reversals : "
      f"{sum(1 for r in all_reversals if r > 0.5):,} predicted")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 16 — Zip Outputs                                                    │
# └──────────────────────────────────────────────────────────────────────────┘

import zipfile
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/module3_outputs.zip"
print("\n" + "═" * 60)
print("  ZIPPING MODULE 3 OUTPUTS")
print("═" * 60)

_dirs = [
    ("/kaggle/working/m3_checkpoints", "m3_checkpoints"),
    ("/kaggle/working/plots",          "plots"),
]
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc_name in _dirs:
        for fpath in Path(src_dir).rglob('*'):
            if fpath.is_file():
                arcname = arc_name + '/' + str(
                    fpath.relative_to(src_dir))
                zf.write(str(fpath), arcname)
                print(f"  + {arcname}  "
                      f"({fpath.stat().st_size/1024:.1f} KB)")

zip_mb = Path(ZIP_PATH).stat().st_size / (1024*1024)
print(f"\n  ✓ module3_outputs.zip  ({zip_mb:.1f} MB)")
print(f"\n  Contents:")
print(f"    m3_checkpoints/narrative_lstm_best.pt")
print(f"    m3_checkpoints/scene_narrative.pt")
print(f"    m3_checkpoints/m3_history.json")
print(f"    plots/m3_00_arc_distribution.png")
print(f"    plots/m3_01_training_curves.png")
print(f"    plots/m3_02_tension_curves.png")
print(f"    plots/m3_03_narrative_vectors.png")

print(f"\n  {'═'*55}")
print(f"  MODULE 3 COMPLETE")
print(f"  {'═'*55}")
print(f"  Architecture  : NarrativeLSTM (causal)")
print(f"  Mechanisms    : LSTM + TensionCell + CharBank")
print(f"  Input         : 256d scene vectors")
print(f"  Output        : 256d narrative vectors")
print(f"  Episodes      : {len(all_narr):,}")
print(f"  Test Arc F1   : {test_f1*100:.2f}%")
print(f"  Best Val F1   : {best_f1*100:.2f}%")
print(f"  {'═'*55}")

display(FileLink(ZIP_PATH, result_html_prefix="⬇️  Download: "))

  Device : cuda
  GPU    : Tesla T4
  Config ready ✓
  SCENE_DIM=256  HIDDEN_DIM=128  NARR_DIM=256  CHAR_DIM=64

  LOADING MODULE 2 OUTPUTS
  Loaded 1,337 scenes
  Emotion classes : ['neutral', 'joyful', 'sad', 'mad', 'scared', 'peaceful', 'powerful']
  ✓ scene_vector dim = 256
  Episodes grouped: 180  (from 1,337 scenes)
  Episode length min/max/avg: 2 / 17 / 7.4
  Character encoder: 20 unique speakers
  Top speakers: ['UNKNOWN', '1st Customer', '2nd Customer', '3rd Customer', 'A Female Student', 'A Student', 'A Waiter', 'Airline Employee', 'Alan', 'Alice']


  Building EpisodeDataset:   0%|          | 0/180 [00:00<?, ?it/s]

  Episodes built : 180  (skipped 0)

  Split: train=126  val=27  test=27

  Arc phase weights:
    setup         count=  215  weight=0.85
    rising        count=  153  weight=1.19
    climax        count=  156  weight=1.17
    falling       count=  153  weight=1.19
    resolution    count=  233  weight=0.78
  ✓ m3_00_arc_distribution.png

  NarrativeLSTM params: 545,617

════════════════════════════════════════════════════════════
  TRAINING NARRATIVE LSTM
  Input  : 256d scene vectors from Module 2
  Model  : Causal LSTM + TensionCell + CharacterBank
  Train  : 126 episodes  Val : 27 episodes  Test : 27 episodes
  Epochs 30  Batch 16  LR 0.0002
════════════════════════════════════════════════════════════


Ep 01/30 ▶ Train:   0%|                         | 0/8 [00:00<?, ?it/s]

    ↳ TRAIN  loss=2.6855  [arc=1.621  ten=0.015  rev=0.615]  arc_f1=16.11%  (0m01s)


...

Ep 29/30   Val  :   0%|                         | 0/2 [00:00<?, ?it/s]

    ↳ VAL    loss=1.7793  [arc=1.323  ten=0.001  rev=0.178]  arc_f1=42.37%  (0m00s)

  ⏹ Early stop at epoch 29
  ✓ m3_01_training_curves.png

════════════════════════════════════════════════════════════
  FINAL EVALUATION — TEST SET
════════════════════════════════════════════════════════════

  Test Arc Phase Macro F1 : 52.47%
  Test Accuracy           : 55.07%
  Best Val F1             : 48.26%

  Classification Report (Arc Phases):
              precision    recall  f1-score   support

       setup       0.85      0.88      0.86        50
      rising       0.50      0.60      0.55        40
      climax       0.32      0.30      0.31        40
     falling       0.31      0.28      0.29        40
  resolution       0.64      0.60      0.62        57

    accuracy                           0.55       227
   macro avg       0.52      0.53      0.52       227
weighted avg       0.55      0.55      0.55       227


════════════════════════════════════════════════════════════
  EXTRACT

  Extracting:   0%|          | 0/12 [00:00<?, ?it/s]

  ✓ m3_02_tension_curves.png
  ✓ m3_03_narrative_vectors.png

  ✓ scene_narrative.pt
    Episodes  : 180
    Total scenes in output: 1,333
    Tension   mean=0.084  std=0.114
    Reversals : 0 predicted

════════════════════════════════════════════════════════════
  ZIPPING MODULE 3 OUTPUTS
════════════════════════════════════════════════════════════
  + m3_checkpoints/scene_narrative.pt  (1445.7 KB)
  + m3_checkpoints/m3_history.json  (5.5 KB)
  + m3_checkpoints/narrative_lstm_best.pt  (2147.7 KB)
  + plots/p2_02_confusion_matrix.png  (133.5 KB)
  + plots/p2_03_per_class_f1.png  (45.2 KB)
  + plots/m2_03_emotion_curves.png  (169.4 KB)
  + plots/p2_00_meld_distribution.png  (72.4 KB)
  + plots/p2_01_training_curves.png  (73.9 KB)
  + plots/m2_04_scene_vectors.png  (117.8 KB)
  + plots/m2_02_scene_types.png  (80.6 KB)
  + plots/m3_03_narrative_vectors.png  (140.3 KB)
  + plots/m2_01_training_curves.png  (79.9 KB)
  + plots/m3_01_training_curves.png  (85.4 KB)
  + plots/m3_00_arc_distrib

/kaggle/working/module3_outputs.zip

In [23]:
print("Module 4 starts here")

Module 4 starts here


In [36]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODULE 4 — Music Planner  v2                                          ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  POSITION IN PIPELINE:                                                  ║
# ║                                                                         ║
# ║    M1 (utterance/256d) → M2 (scene/256d) → M3 (narrative/256d)        ║
# ║                                                    ↓                   ║
# ║                                             M4 (music descriptors)     ║
# ║                                                                         ║
# ║  INPUT — 576d:                                                          ║
# ║    scene_vector      (256d) — from M2: what THIS scene feels like      ║
# ║    narrative_vector  (256d) — from M3: where we are in the story arc   ║
# ║    emotion_context   ( 64d) — from M2 emotion_probs(7) → learned proj  ║
# ║                                                                         ║
# ║  WHY 576d AND NOT 512d:                                                 ║
# ║    The design plan specified scene(256)+narrative(256)+char_ctx(64).   ║
# ║    M3 fuses char_ctx internally into narrative_vector, so there is no  ║
# ║    separate 64d output from M3. However the 64d slot still carries     ║
# ║    meaning: here it encodes the per-scene emotion probability dist      ║
# ║    (7d softmax from M2 → EmotionContextEncoder → 64d). This preserves  ║
# ║    the design plan's input shape and intent — the 64d encodes "which   ║
# ║    emotional character state dominates this scene", which was the       ║
# ║    original purpose of char_ctx in the plan.                           ║
# ║                                                                         ║
# ║  TRUNK — 3 bottleneck layers (matches design plan exactly):            ║
# ║    576 → 512 → 256 → 128                                               ║
# ║    All 13 heads read from the same 128d shared representation.         ║
# ║    Shared trunk forces internal consistency across heads.               ║
# ║                                                                         ║
# ║  13 OUTPUT HEADS:                                                       ║
# ║    Regression  (3):  tempo_score, intensity, valence                   ║
# ║    Classification(9):tonality(3), harmonic_style(8), motion_type(6),  ║
# ║                       rhythm_style(6), rhythm_density(3),              ║
# ║                       dynamic_shape(8), texture(6), transition(5),     ║
# ║                       arc_verify(5)                                    ║
# ║    Multi-label (1):  instruments(15) — BCELoss                         ║
# ║                                                                         ║
# ║  TRAINING — fully self-supervised:                                      ║
# ║    All 13 targets derived from M2 + M3 signals via film-scoring rules. ║
# ║    No human annotation required.                                        ║
# ║                                                                         ║
# ║  FINAL OUTPUT — assemble_prompt():                                      ║
# ║    Converts raw head outputs → structured music descriptor string      ║
# ║    suitable for a music generation API or human composer brief.        ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import os, json, pickle, time, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import (f1_score, accuracy_score,
                              classification_report)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PLOT_DIR = "/kaggle/working/plots"
Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)

print(f"  Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Configuration                                                   │
# └──────────────────────────────────────────────────────────────────────────┘

class CFG4:
    # ── Input paths ───────────────────────────────────────────────────────
    SCENE_VEC_PT = ("/kaggle/input/datasets/suyashnpande/module2-outputs/m2_checkpoints"
                    "/scene_vectors.pt")
    NARR_PT      = ("/kaggle/input/datasets/suyashnpande/module3-outputs/m3_checkpoints"
                    "/scene_narrative.pt")

    # ── Output paths ──────────────────────────────────────────────────────
    CKPT_DIR     = "/kaggle/working/m4_checkpoints"
    BEST_CKPT    = "/kaggle/working/m4_checkpoints/music_planner_best.pt"
    PROMPTS_OUT  = "/kaggle/working/m4_checkpoints/music_prompts.json"

    # ── Architecture (matches design plan exactly) ────────────────────────
    SCENE_DIM    = 256    # M2 scene_vector
    NARR_DIM     = 256    # M3 narrative_vector
    EMO_PROB_DIM = 7      # M2 emotion_probs (input to EmotionContextEncoder)
    EMO_CTX_DIM  = 64     # EmotionContextEncoder output (fills char_ctx slot)
    INPUT_DIM    = SCENE_DIM + NARR_DIM + EMO_CTX_DIM   # 576

    # Trunk: 576 → 512 → 256 → 128  (design plan: 3 bottleneck layers)
    TRUNK_DIMS   = [512, 256, 128]

    # Head output sizes (exact from design plan)
    NUM_TONALITY   = 3    # major / minor / ambiguous
    NUM_HARMONIC   = 8    # harmonic texture types
    NUM_MOTION     = 6    # musical motion types
    NUM_RHYTHM     = 6    # rhythm pattern types
    NUM_DENSITY    = 3    # minimal / moderate / dense
    NUM_DYNAMIC    = 8    # dynamic arc shapes
    NUM_TEXTURE    = 6    # orchestration textures
    NUM_TRANSITION = 5    # scene transition styles
    NUM_ARC        = 5    # arc phase verify (matches M3)
    NUM_INSTRUMENTS= 15   # multi-label instrument groups

    # Tempo: BPM = 40 + sigmoid_output * 120   (design plan: 40-160 BPM)
    TEMPO_BASE   = 40
    TEMPO_RANGE  = 120    # so max = 40 + 120 = 160

    # ── Training ──────────────────────────────────────────────────────────
    BATCH_SIZE   = 32
    LR           = 3e-4
    EPOCHS       = 35
    EARLY_STOP   = 8
    DROPOUT      = 0.20
    MAX_WEIGHT   = 10.0   # class weight cap

    # ── Splits ────────────────────────────────────────────────────────────
    VAL_SPLIT    = 0.15
    TEST_SPLIT   = 0.15

Path(CFG4.CKPT_DIR).mkdir(parents=True, exist_ok=True)
print("  Config ready ✓")
print(f"  INPUT_DIM = {CFG4.INPUT_DIM}  "
      f"(scene(256) + narrative(256) + emo_ctx(64))")
print(f"  TRUNK     = {CFG4.INPUT_DIM} → "
      f"{' → '.join(str(d) for d in CFG4.TRUNK_DIMS)}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Music Vocabulary                                                │
# │                                                                          │
# │  All label names from the design plan, verbatim and in order.          │
# │  Index = class integer used by heads. Order is canonical.              │
# └──────────────────────────────────────────────────────────────────────────┘

# ── Classification vocabularies (from design plan) ─────────────────────────

TONALITY_NAMES    = ['major', 'minor', 'ambiguous']

HARMONIC_NAMES    = [
    'diatonic_warm',          # 0
    'melodic_minor',          # 1
    'dissonant_sparse',       # 2
    'cluster_dissonance',     # 3
    'chromatic_aggressive',   # 4
    'modal_mystical',         # 5
    'pedal_tone',             # 6
    'atonal',                 # 7
]

MOTION_NAMES      = [
    'static_slow',            # 0
    'rising_tension',         # 1
    'chaotic',                # 2
    'slow_release',           # 3
    'steady_drive',           # 4
    'sudden_drop',            # 5
]

RHYTHM_NAMES      = [
    'sustained',              # 0
    'ostinato',               # 1
    'sparse_pulse',           # 2
    'rubato',                 # 3
    'driving_8th',            # 4
    'silence',                # 5
]

DENSITY_NAMES     = ['minimal', 'moderate', 'dense']

DYNAMIC_NAMES     = [
    'slow_build',             # 0
    'crescendo',              # 1
    'sudden_hit_then_drive',  # 2
    'drop_then_soft_build',   # 3
    'sustained_peak',         # 4
    'decrescendo',            # 5
    'flat',                   # 6
    'pulse_and_release',      # 7
]

TEXTURE_NAMES     = [
    'ambient_minimal',        # 0
    'layered_ambient',        # 1
    'hybrid_orchestral',      # 2
    'emotional_orchestral',   # 3
    'electronic_dark',        # 4
    'acoustic_intimate',      # 5
]

TRANSITION_NAMES  = [
    'attacca',                # 0
    'crossfade',              # 1
    'hard_cut',               # 2
    'silence_gap',            # 3
    'motif_bridge',           # 4
]

# ── Multi-label: instruments (from design plan, exact order) ───────────────
INSTRUMENT_NAMES  = [
    'low_strings',            # 0
    'high_strings',           # 1
    'solo_violin',            # 2
    'solo_cello',             # 3
    'solo_piano',             # 4
    'brass_stabs',            # 5
    'french_horn',            # 6
    'ambient_synth',          # 7
    'sub_bass_drone',         # 8
    'metallic_percussion',    # 9
    'cinematic_drums',        # 10
    'reverse_piano',          # 11
    'distorted_synth_bass',   # 12
    'choir',                  # 13
    'woodwinds',              # 14
]

# ── Shared with upstream modules ───────────────────────────────────────────
EMOTION_NAMES     = ['neutral', 'joyful', 'sad', 'mad',
                     'scared', 'peaceful', 'powerful']
SCENE_TYPE_NAMES  = ['dialogue', 'action', 'investigation',
                     'suspense', 'dramatic_reveal', 'transition']
ARC_PHASES        = ['setup', 'rising', 'climax', 'falling', 'resolution']

print("  Music vocabulary loaded:")
print(f"    Harmonic styles : {HARMONIC_NAMES}")
print(f"    Dynamic shapes  : {DYNAMIC_NAMES}")
print(f"    Instruments     : {len(INSTRUMENT_NAMES)} groups")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Self-Supervised Target Derivation                               │
# │                                                                          │
# │  All 13 music targets derived from M2 + M3 signals.                    │
# │  No human annotation. Rules grounded in film scoring theory.            │
# │                                                                          │
# │  Rule philosophy:                                                        │
# │    Rules are deliberately simple — the neural network learns the        │
# │    COMBINATION and INTERPOLATION between rules. Rules provide clean     │
# │    directional signal; the shared trunk generalises beyond them.        │
# │                                                                          │
# │  Regression targets are always in [0,1] or [-1,1] matching head range. │
# │  Classification targets are always ints matching head output size.      │
# └──────────────────────────────────────────────────────────────────────────┘

def derive_music_targets(
        # ── From M2 scene metadata ──────────────────────────────────────
        dominant_emotion  : int,    # 0-6 (index into EMOTION_NAMES)
        scene_type        : int,    # 0-5 (index into SCENE_TYPE_NAMES)
        arousal_score     : float,  # scalar (any range, clipped internally)
        conflict_score    : float,  # scalar (any range, clipped internally)
        trajectory_score  : float,  # scalar: positive=improving, negative=worsening
        variance_score    : float,  # [0,1]: how much emotion changes within scene
        pacing_score      : float,  # [0,1]: utterance density
        emotion_dist      : np.ndarray,  # (7,) softmax emotion distribution
        # ── From M3 narrative metadata ──────────────────────────────────
        tension_score     : float,  # [0,1]: accumulated narrative pressure
        arc_phase         : int,    # 0-4
        reversal_score    : float,  # [0,1]: probability of emotion direction flip
) -> dict:
    """
    Maps M2+M3 scene signals to 13 music targets.
    All values clipped to valid ranges before use.
    Returns dict with fields matching MusicDataset keys.
    """
    emo   = EMOTION_NAMES[max(0, min(dominant_emotion, 6))]
    stype = SCENE_TYPE_NAMES[max(0, min(scene_type, 5))]
    arc   = arc_phase

    # Clip continuous inputs
    aros  = float(np.clip(arousal_score,    -1.0,  1.0))
    conf  = float(np.clip(conflict_score,   -1.0,  1.0))
    traj  = float(np.clip(trajectory_score, -1.0,  1.0))
    vari  = float(np.clip(variance_score,    0.0,  1.0))
    pace  = float(np.clip(pacing_score,      0.0,  1.0))
    tens  = float(np.clip(tension_score,     0.0,  1.0))
    rev   = float(np.clip(reversal_score,    0.0,  1.0))

    # ── REGRESSION TARGETS ──────────────────────────────────────────────

    # tempo_norm → sigmoid output → BPM = 40 + tempo_norm * 120
    # High arousal/tension = faster. Peaceful/setup = slower.
    tempo_norm = float(np.clip(
        0.40 * max(aros, 0.0)     # arousal drives tempo
        + 0.30 * tens              # tension pushes it up
        + 0.20 * vari              # emotional flux = busier
        + 0.10 * pace,             # denser scenes feel faster
        0.0, 1.0))

    # intensity → dynamics: pp to ff
    intensity  = float(np.clip(
        0.40 * tens
        + 0.30 * abs(aros)
        + 0.20 * vari
        + 0.10 * pace,
        0.0, 1.0))

    # valence → Tanh output: +1 = major/bright, -1 = minor/dark
    # conflict_score encodes positive/negative emotional balance
    valence    = float(np.clip(conf, -1.0, 1.0))

    # ── TONALITY ────────────────────────────────────────────────────────
    # 0=major  1=minor  2=ambiguous
    if   conf > 0.25:   tonality = 0   # positive balance → major
    elif conf < -0.20:  tonality = 1   # negative balance → minor
    else:               tonality = 2   # mixed → ambiguous

    # ── HARMONIC STYLE ──────────────────────────────────────────────────
    # Primary signal: dominant_emotion × scene_type
    if   emo == 'joyful'  and stype not in ['action']:   harmonic = 0  # diatonic_warm
    elif emo == 'sad':                                    harmonic = 1  # melodic_minor
    elif emo == 'scared'  and tens < 0.55:               harmonic = 2  # dissonant_sparse
    elif emo == 'scared'  and tens >= 0.55:              harmonic = 3  # cluster_dissonance
    elif emo == 'mad'     or stype == 'action':          harmonic = 4  # chromatic_aggressive
    elif stype == 'investigation':                        harmonic = 5  # modal_mystical
    elif emo in ['neutral', 'peaceful'] or stype == 'transition':
                                                          harmonic = 6  # pedal_tone
    elif emo == 'powerful' or stype == 'dramatic_reveal':harmonic = 7  # atonal
    else:                                                 harmonic = 6  # default

    # ── MOTION TYPE ─────────────────────────────────────────────────────
    if   rev > 0.55:                                      motion = 5  # sudden_drop
    elif arc in [1, 2] and tens > 0.35:                  motion = 1  # rising_tension
    elif stype == 'action' or (emo=='mad' and vari>0.5): motion = 2  # chaotic
    elif traj < -0.30:                                    motion = 3  # slow_release
    elif emo in ['joyful', 'peaceful'] and vari < 0.4:  motion = 4  # steady_drive
    else:                                                 motion = 0  # static_slow

    # ── RHYTHM STYLE ────────────────────────────────────────────────────
    if   stype == 'transition' and vari < 0.20:          rhythm = 5  # silence
    elif emo in ['sad', 'peaceful'] and pace < 0.45:     rhythm = 0  # sustained
    elif emo == 'sad' and vari > 0.30:                   rhythm = 3  # rubato
    elif stype == 'suspense' or (emo=='scared' and tens>0.30):
                                                          rhythm = 1  # ostinato
    elif emo == 'scared' and tens <= 0.30:               rhythm = 2  # sparse_pulse
    elif emo in ['joyful', 'mad'] or stype == 'action':  rhythm = 4  # driving_8th
    else:                                                 rhythm = 0  # sustained

    # ── RHYTHM DENSITY ──────────────────────────────────────────────────
    # 0=minimal  1=moderate  2=dense
    den_score = 0.4 * pace + 0.3 * tens + 0.3 * vari
    if   den_score < 0.30: density = 0
    elif den_score < 0.60: density = 1
    else:                  density = 2

    # ── DYNAMIC SHAPE ───────────────────────────────────────────────────
    if   arc == 2 and tens > 0.60:                        dynamic = 4  # sustained_peak
    elif arc in [1, 2] and tens > 0.35:                  dynamic = 1  # crescendo
    elif arc == 0 and tens < 0.30:                       dynamic = 0  # slow_build
    elif arc in [3, 4] and traj > 0.20:                  dynamic = 5  # decrescendo
    elif arc == 3 and tens > 0.50:                       dynamic = 2  # sudden_hit_then_drive
    elif arc in [3, 4] and tens < 0.30:                  dynamic = 3  # drop_then_soft_build
    elif stype == 'dramatic_reveal' or emo == 'powerful': dynamic = 7  # pulse_and_release
    else:                                                 dynamic = 6  # flat

    # ── TEXTURE ─────────────────────────────────────────────────────────
    if   stype == 'action' or emo == 'mad':              texture = 4  # electronic_dark
    elif stype == 'dramatic_reveal' or arc == 2:         texture = 3  # emotional_orchestral
    elif stype in ['suspense', 'investigation']:          texture = 2  # hybrid_orchestral
    elif emo in ['sad', 'peaceful'] and intensity < 0.40:texture = 5  # acoustic_intimate
    elif tens < 0.20 and arc in [0, 4]:                  texture = 0  # ambient_minimal
    else:                                                 texture = 1  # layered_ambient

    # ── TRANSITION STYLE ────────────────────────────────────────────────
    if   rev > 0.55:         transition = 2  # hard_cut
    elif arc in [1, 2]:      transition = 0  # attacca — keep momentum
    elif tens > 0.50:        transition = 4  # motif_bridge
    elif arc in [3, 4]:      transition = 3  # silence_gap
    else:                    transition = 1  # crossfade

    # ── ARC VERIFY (pass-through from M3) ───────────────────────────────
    arc_verify = int(arc_phase)   # M4 learns to verify arc prediction

    # ── INSTRUMENTS (multi-label, BCELoss) ──────────────────────────────
    # 15 binary flags, one per instrument group
    instr = np.zeros(CFG4.NUM_INSTRUMENTS, dtype=np.float32)

    # low_strings: negative emotion OR high tension
    if emo in ['sad', 'scared', 'mad'] or tens > 0.40:
        instr[0] = 1.0
    # high_strings: emotional peak, joyful, powerful
    if emo in ['joyful', 'sad', 'powerful'] or arc in [2, 3]:
        instr[1] = 1.0
    # solo_violin: intimate sadness
    if emo == 'sad' and stype != 'action':
        instr[2] = 1.0
    # solo_cello: sad/peaceful or falling/resolution arc
    if emo in ['sad', 'peaceful'] or arc in [3, 4]:
        instr[3] = 1.0
    # solo_piano: low density OR peaceful/sad emotion
    if density == 0 or emo in ['peaceful', 'sad']:
        instr[4] = 1.0
    # brass_stabs: action or climax with high tension
    if stype == 'action' or (arc == 2 and tens > 0.60):
        instr[5] = 1.0
    # french_horn: rising/climax arc, not joyful/peaceful
    if arc in [1, 2] and emo not in ['joyful', 'peaceful']:
        instr[6] = 1.0
    # ambient_synth: suspense, investigation, transition
    if stype in ['suspense', 'investigation', 'transition']:
        instr[7] = 1.0
    # sub_bass_drone: fear or sustained suspense
    if emo == 'scared' or (tens > 0.50 and stype == 'suspense'):
        instr[8] = 1.0
    # metallic_percussion: fear, investigation
    if emo == 'scared' or stype == 'investigation':
        instr[9] = 1.0
    # cinematic_drums: action, mad + tension
    if stype == 'action' or (emo == 'mad' and tens > 0.40):
        instr[10] = 1.0
    # reverse_piano: powerful, dramatic_reveal
    if emo == 'powerful' or stype == 'dramatic_reveal':
        instr[11] = 1.0
    # distorted_synth_bass: action + anger
    if stype == 'action' and emo == 'mad':
        instr[12] = 1.0
    # choir: climax peak, powerful, high-intensity joyful
    if (arc == 2 and intensity > 0.60) or emo == 'powerful':
        instr[13] = 1.0
    # woodwinds: peaceful, joyful, setup arc
    if emo in ['peaceful', 'joyful'] or arc == 0:
        instr[14] = 1.0

    return {
        # Regression [0,1] or [-1,1]
        'tempo_norm'  : np.float32(tempo_norm),
        'intensity'   : np.float32(intensity),
        'valence'     : np.float32(valence),
        # Classification (int)
        'tonality'         : tonality,
        'harmonic_style'   : harmonic,
        'motion_type'      : motion,
        'rhythm_style'     : rhythm,
        'rhythm_density'   : density,
        'dynamic_shape'    : dynamic,
        'texture'          : texture,
        'transition'       : transition,
        'arc_verify'       : arc_verify,
        # Multi-label
        'instruments'      : instr,          # (15,) float32
    }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — Dataset                                                         │
# │                                                                          │
# │  Joins M2 scene metadata with M3 episode narratives.                   │
# │  One training sample = one scene.                                       │
# │                                                                          │
# │  Each sample contains:                                                  │
# │    scene_vector    (256,) tensor  — M2 direct                          │
# │    narrative_vector(256,) tensor  — M3 per-scene                        │
# │    emotion_probs   (7,)   tensor  — M2 softmax (→ EmotionContextEncoder)│
# │    + all derived targets                                                │
# └──────────────────────────────────────────────────────────────────────────┘

class MusicDataset(Dataset):
    def __init__(self, samples: list):
        self.samples = samples

    def __len__(self):          return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            # Model inputs
            'scene_vector'     : s['scene_vector'],       # (256,)
            'narrative_vector' : s['narrative_vector'],   # (256,)
            'emotion_probs'    : s['emotion_probs'],      # (7,)
            # Regression targets
            'tempo_norm'       : torch.tensor(float(s['tempo_norm']),
                                              dtype=torch.float32),
            'intensity'        : torch.tensor(float(s['intensity']),
                                              dtype=torch.float32),
            'valence'          : torch.tensor(float(s['valence']),
                                              dtype=torch.float32),
            # Classification targets
            'tonality'         : torch.tensor(s['tonality'],
                                              dtype=torch.long),
            'harmonic_style'   : torch.tensor(s['harmonic_style'],
                                              dtype=torch.long),
            'motion_type'      : torch.tensor(s['motion_type'],
                                              dtype=torch.long),
            'rhythm_style'     : torch.tensor(s['rhythm_style'],
                                              dtype=torch.long),
            'rhythm_density'   : torch.tensor(s['rhythm_density'],
                                              dtype=torch.long),
            'dynamic_shape'    : torch.tensor(s['dynamic_shape'],
                                              dtype=torch.long),
            'texture'          : torch.tensor(s['texture'],
                                              dtype=torch.long),
            'transition'       : torch.tensor(s['transition'],
                                              dtype=torch.long),
            'arc_verify'       : torch.tensor(s['arc_verify'],
                                              dtype=torch.long),
            # Multi-label
            'instruments'      : torch.tensor(s['instruments'],
                                              dtype=torch.float32),
        }


def collate_music(batch: list) -> dict:
    return {k: torch.stack([b[k] for b in batch])
            for k in batch[0]}


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — EmotionContextEncoder                                           │
# │                                                                          │
# │  Projects M2 emotion_probs (7d softmax) → 64d.                         │
# │  This fills the char_ctx(64) slot from the design plan.                │
# │  Rationale: the emotion probability distribution encodes which          │
# │  emotional characters dominate this scene — joyful scenes have         │
# │  high P(joyful) mass, which represents the emotional "character"        │
# │  of the scene in much the same way char_ctx did in the plan.          │
# └──────────────────────────────────────────────────────────────────────────┘

class EmotionContextEncoder(nn.Module):
    """
    7d emotion probability distribution → 64d context vector.
    Simple 2-layer MLP with LayerNorm.
    Trained end-to-end with the rest of MusicPlannerV2.
    """
    def __init__(self,
                 in_dim   : int = CFG4.EMO_PROB_DIM,    # 7
                 out_dim  : int = CFG4.EMO_CTX_DIM,     # 64
                 dropout  : float = CFG4.DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, out_dim),
            nn.LayerNorm(out_dim),
            nn.ReLU(),
        )

    def forward(self, emotion_probs: torch.Tensor) -> torch.Tensor:
        """
        emotion_probs : (B, 7)  softmax distribution from M2
        returns       : (B, 64)
        """
        return self.net(emotion_probs)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — MusicPlannerV2                                                  │
# │                                                                          │
# │  Architecture exactly as specified in the design plan.                  │
# │                                                                          │
# │  WHY shared trunk matters:                                               │
# │    tempo=160 BPM + rhythm=sustained is contradictory.                  │
# │    texture=acoustic_intimate + dynamic=crescendo+dense is wrong.        │
# │    All 13 heads read from the same 128d "music state". The trunk       │
# │    learns a single coherent interpretation of the scene, and heads     │
# │    specialise on different facets of that interpretation.              │
# │    Contradictions are suppressed because they share the same root.     │
# └──────────────────────────────────────────────────────────────────────────┘

class MusicPlannerV2(nn.Module):
    """
    13-head music descriptor generator.
    Input: scene(256) + narrative(256) + emo_ctx(64) = 576d
    Trunk: 576 → 512 → 256 → 128   (3 bottleneck layers)
    """
    def __init__(self,
                 input_dim  : int   = CFG4.INPUT_DIM,    # 576
                 trunk_dims : list  = CFG4.TRUNK_DIMS,   # [512, 256, 128]
                 dropout    : float = CFG4.DROPOUT):
        super().__init__()

        # ── Emotion context encoder (fills char_ctx slot) ─────────────
        self.emo_encoder = EmotionContextEncoder(
            in_dim  = CFG4.EMO_PROB_DIM,
            out_dim = CFG4.EMO_CTX_DIM,
            dropout = dropout)

        # ── Shared trunk: 576 → 512 → 256 → 128 ──────────────────────
        layers = []
        in_d   = input_dim
        for out_d in trunk_dims:
            layers += [
                nn.Linear(in_d, out_d),
                nn.LayerNorm(out_d),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_d = out_d
        self.trunk = nn.Sequential(*layers)
        head_in = trunk_dims[-1]   # 128

        # ── Regression heads ──────────────────────────────────────────
        # tempo_score  → Sigmoid → BPM = 40 + output * 120
        self.tempo_head    = nn.Sequential(
            nn.Linear(head_in, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        # intensity    → Sigmoid → 0 (pp) to 1 (ff)
        self.intensity_head= nn.Sequential(
            nn.Linear(head_in, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        # valence      → Tanh → -1 (minor) to +1 (major)
        self.valence_head  = nn.Sequential(
            nn.Linear(head_in, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())

        # ── Classification heads ──────────────────────────────────────
        self.tonality_head    = nn.Linear(head_in, CFG4.NUM_TONALITY)     # 3
        self.harmonic_head    = nn.Linear(head_in, CFG4.NUM_HARMONIC)     # 8
        self.motion_head      = nn.Linear(head_in, CFG4.NUM_MOTION)       # 6
        self.rhythm_head      = nn.Linear(head_in, CFG4.NUM_RHYTHM)       # 6
        self.density_head     = nn.Linear(head_in, CFG4.NUM_DENSITY)      # 3
        self.dynamic_head     = nn.Linear(head_in, CFG4.NUM_DYNAMIC)      # 8
        self.texture_head     = nn.Linear(head_in, CFG4.NUM_TEXTURE)      # 6
        self.transition_head  = nn.Linear(head_in, CFG4.NUM_TRANSITION)   # 5
        self.arc_head         = nn.Linear(head_in, CFG4.NUM_ARC)          # 5

        # ── Multi-label head: instruments (BCELoss) ───────────────────
        self.instrument_head  = nn.Sequential(
            nn.Linear(head_in, 64), nn.ReLU(),
            nn.Linear(64, CFG4.NUM_INSTRUMENTS),
            nn.Sigmoid())

        # Xavier init for all linear layers
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self,
                scene_vector     : torch.Tensor,   # (B, 256)
                narrative_vector : torch.Tensor,   # (B, 256)
                emotion_probs    : torch.Tensor,   # (B, 7)
                ) -> dict:
        """
        Full forward pass through EmotionContextEncoder + trunk + 13 heads.
        Returns dict of all head outputs.
        """
        # Encode emotion distribution → 64d context
        emo_ctx = self.emo_encoder(emotion_probs)   # (B, 64)

        # Concatenate all three inputs: 256 + 256 + 64 = 576
        z = torch.cat([scene_vector, narrative_vector, emo_ctx],
                       dim=-1)                      # (B, 576)

        # Shared trunk: 576 → 512 → 256 → 128
        r = self.trunk(z)                           # (B, 128)

        return {
            # ── Regression ────────────────────────────────────────────
            'tempo_score'  : self.tempo_head(r).squeeze(-1),     # (B,)
            'intensity'    : self.intensity_head(r).squeeze(-1), # (B,)
            'valence'      : self.valence_head(r).squeeze(-1),   # (B,)
            # ── Classification ─────────────────────────────────────────
            'tonality'     : self.tonality_head(r),    # (B, 3)
            'harmonic_style': self.harmonic_head(r),   # (B, 8)
            'motion_type'  : self.motion_head(r),      # (B, 6)
            'rhythm_style' : self.rhythm_head(r),      # (B, 6)
            'rhythm_density': self.density_head(r),    # (B, 3)
            'dynamic_shape': self.dynamic_head(r),     # (B, 8)
            'texture'      : self.texture_head(r),     # (B, 6)
            'transition'   : self.transition_head(r),  # (B, 5)
            'arc_verify'   : self.arc_head(r),         # (B, 5)
            # ── Multi-label ────────────────────────────────────────────
            'instruments'  : self.instrument_head(r),  # (B, 15)
            # ── Trunk features (for downstream use) ───────────────────
            'trunk_features': r,                       # (B, 128)
        }

    # ── Utility ────────────────────────────────────────────────────────────

    @staticmethod
    def tempo_to_bpm(tempo_score: float) -> int:
        """Sigmoid output → BPM integer (design plan: 40 + score * 120)."""
        return int(CFG4.TEMPO_BASE + float(tempo_score) * CFG4.TEMPO_RANGE)

    def assemble_prompt(self,
                        outputs    : dict,
                        scene_id            = None,
                        arc_phase  : int    = 0,
                        tension    : float  = 0.0,
                        emotion    : str    = '',
                        scene_type : str    = '') -> str:
        """
        Converts raw head outputs (single scene, batch dim removed) into
        a structured music descriptor string.

        Format matches the design plan:
          "Scene {id} — {tempo} BPM, {tonality} key, {harmonic} harmony.
           Motion: {motion}. Dynamic shape: {dynamic}. Instruments: {list}."
        Extended with all other descriptors for completeness.

        Args:
            outputs    : dict from forward() with batch dim [0] already selected
            scene_id   : (season, episode, dialogue_id) tuple — for labelling
            arc_phase  : int 0-4 from Module 3
            tension    : float [0,1] from Module 3
            emotion    : string emotion name (optional, for context)
            scene_type : string scene type name (optional, for context)

        Returns:
            Structured multi-line prompt string.
        """
        def _scalar(key):
            v = outputs[key]
            return float(v.item() if isinstance(v, torch.Tensor) else v)

        def _argmax(key, names):
            v = outputs[key]
            if isinstance(v, torch.Tensor):
                idx = int(v.argmax().item())
            else:
                idx = int(np.argmax(np.array(v)))
            return names[min(idx, len(names) - 1)]

        tempo_bpm  = self.tempo_to_bpm(_scalar('tempo_score'))
        intensity  = _scalar('intensity')
        valence    = _scalar('valence')

        tonality   = _argmax('tonality',      TONALITY_NAMES)
        harmonic   = _argmax('harmonic_style',HARMONIC_NAMES)
        motion     = _argmax('motion_type',   MOTION_NAMES)
        rhythm     = _argmax('rhythm_style',  RHYTHM_NAMES)
        density    = _argmax('rhythm_density',DENSITY_NAMES)
        dynamic    = _argmax('dynamic_shape', DYNAMIC_NAMES)
        texture    = _argmax('texture',       TEXTURE_NAMES)
        transition = _argmax('transition',    TRANSITION_NAMES)
        arc_name   = ARC_PHASES[max(0, min(arc_phase, 4))]

        # Instruments: multi-label threshold = 0.5
        instr_vec  = outputs['instruments']
        if isinstance(instr_vec, torch.Tensor):
            flags = (instr_vec > 0.5).cpu().numpy().astype(bool)
        else:
            flags = (np.array(instr_vec) > 0.5)
        instruments = [INSTRUMENT_NAMES[i]
                       for i, f in enumerate(flags) if f]
        if not instruments:
            instruments = ['solo_piano']   # always at least one instrument

        # Intensity → notation dynamic
        if   intensity < 0.25: dyn_mark = 'pp'
        elif intensity < 0.45: dyn_mark = 'mp'
        elif intensity < 0.65: dyn_mark = 'mf'
        elif intensity < 0.85: dyn_mark = 'f'
        else:                   dyn_mark = 'ff'

        sid_str = (f"S{scene_id[0]}E{scene_id[1]}D{scene_id[2]}"
                   if scene_id else "scene")

        ctx_parts = []
        if emotion:    ctx_parts.append(f"emotion:{emotion}")
        if scene_type: ctx_parts.append(f"type:{scene_type}")
        ctx_str = f"  [{', '.join(ctx_parts)}]" if ctx_parts else ""

        # ── Core design-plan format + full descriptor block ───────────
        prompt = (
            f"Scene {sid_str} — "
            f"{tempo_bpm} BPM, {tonality} key, {harmonic} harmony. "
            f"Motion: {motion}. Dynamic shape: {dynamic}. "
            f"Instruments: {', '.join(instruments)}.\n"
            f"  arc:{arc_name}  tension:{tension:.2f}  "
            f"dynamics:{dyn_mark}  rhythm:{rhythm}({density})  "
            f"texture:{texture}  transition→next:{transition}"
            f"{ctx_str}"
        )
        return prompt


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Loss Function                                                   │
# │                                                                          │
# │  Multi-task weighted loss across all 13 heads.                          │
# │                                                                          │
# │  Weight rationale (from importance to the musical output):              │
# │    harmonic_style  ×1.0 — most consequential: defines the whole mood   │
# │    dynamic_shape   ×0.8 — shapes listener experience arc               │
# │    texture         ×0.7 — orchestration colour                         │
# │    motion_type     ×0.7 — rhythmic energy profile                      │
# │    transition      ×0.6 — affects inter-scene coherence                │
# │    tonality/rhythm ×0.6 each                                            │
# │    tempo/intensity ×0.5 each (regression, different scale)             │
# │    valence         ×0.4 (highly correlated with tonality)              │
# │    density/arc     ×0.3 each (auxiliary/verify)                        │
# │    instruments     ×0.5 (multi-label, many valid answers)              │
# └──────────────────────────────────────────────────────────────────────────┘

class MusicPlannerLoss(nn.Module):
    def __init__(self, class_weights: dict = None):
        super().__init__()
        cw = class_weights or {}

        # Cross-entropy with optional class weighting for imbalanced heads
        def _ce(key):
            w = cw.get(key)
            return nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)

        self.ce_harmonic   = _ce('harmonic_style')
        self.ce_dynamic    = _ce('dynamic_shape')
        self.ce_texture    = _ce('texture')
        self.ce_motion     = _ce('motion_type')
        self.ce_transition = _ce('transition')
        self.ce_tonality   = _ce('tonality')
        self.ce_rhythm     = _ce('rhythm_style')
        self.ce_density    = _ce('rhythm_density')
        self.ce_arc        = _ce('arc_verify')

        self.mse = nn.MSELoss()
        self.bce = nn.BCELoss()

    def forward(self, preds: dict, batch: dict) -> dict:
        D = DEVICE

        # ── Regression ────────────────────────────────────────────────
        l_tempo    = self.mse(preds['tempo_score'],
                              batch['tempo_norm'].to(D))
        l_intensity= self.mse(preds['intensity'],
                              batch['intensity'].to(D))
        l_valence  = self.mse(preds['valence'],
                              batch['valence'].to(D))

        # ── Classification ─────────────────────────────────────────────
        l_harmonic  = self.ce_harmonic(  preds['harmonic_style'],
                                          batch['harmonic_style'].to(D))
        l_dynamic   = self.ce_dynamic(   preds['dynamic_shape'],
                                          batch['dynamic_shape'].to(D))
        l_texture   = self.ce_texture(   preds['texture'],
                                          batch['texture'].to(D))
        l_motion    = self.ce_motion(    preds['motion_type'],
                                          batch['motion_type'].to(D))
        l_transition= self.ce_transition(preds['transition'],
                                          batch['transition'].to(D))
        l_tonality  = self.ce_tonality(  preds['tonality'],
                                          batch['tonality'].to(D))
        l_rhythm    = self.ce_rhythm(    preds['rhythm_style'],
                                          batch['rhythm_style'].to(D))
        l_density   = self.ce_density(   preds['rhythm_density'],
                                          batch['rhythm_density'].to(D))
        l_arc       = self.ce_arc(       preds['arc_verify'],
                                          batch['arc_verify'].to(D))

        # ── Multi-label ────────────────────────────────────────────────
        l_instr    = self.bce(preds['instruments'],
                              batch['instruments'].to(D))

        # ── Weighted total ─────────────────────────────────────────────
        total = (1.0 * l_harmonic    +
                 0.8 * l_dynamic     +
                 0.7 * l_texture     +
                 0.7 * l_motion      +
                 0.6 * l_transition  +
                 0.6 * l_tonality    +
                 0.6 * l_rhythm      +
                 0.5 * l_tempo       +
                 0.5 * l_intensity   +
                 0.5 * l_instr       +
                 0.4 * l_valence     +
                 0.3 * l_density     +
                 0.3 * l_arc)

        return {
            'total'      : total,
            # Individual losses for monitoring
            'harmonic'   : l_harmonic.item(),
            'dynamic'    : l_dynamic.item(),
            'texture'    : l_texture.item(),
            'motion'     : l_motion.item(),
            'tempo'      : l_tempo.item(),
            'instruments': l_instr.item(),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Train / Eval Loop                                               │
# │                                                                          │
# │  Primary metric: harmonic_style macro F1                                │
# │  (most consequential classification head)                               │
# └──────────────────────────────────────────────────────────────────────────┘

def run_epoch(model, loader, loss_fn,
              optimizer=None, phase='train',
              epoch=0, total=0) -> tuple:
    is_train = (phase == 'train')
    model.train() if is_train else model.eval()

    tot_loss       = 0.0
    lp             = defaultdict(float)
    all_ht, all_hp = [], []
    t0 = time.time()

    pbar = tqdm(
        loader,
        desc=(f"Ep {epoch+1:02d}/{total} "
              f"{'▶ Train' if is_train else '  Val  '}"),
        leave=True,
        bar_format="{l_bar}{bar:25}{r_bar}")

    for step, batch in enumerate(pbar):
        sv  = batch['scene_vector'].to(DEVICE)      # (B, 256)
        nv  = batch['narrative_vector'].to(DEVICE)  # (B, 256)
        ep  = batch['emotion_probs'].to(DEVICE)     # (B, 7)

        with torch.set_grad_enabled(is_train):
            preds  = model(sv, nv, ep)
            losses = loss_fn(preds, batch)

        if is_train:
            optimizer.zero_grad()
            losses['total'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        all_ht.extend(batch['harmonic_style'].numpy().tolist())
        all_hp.extend(
            preds['harmonic_style'].argmax(-1)
                                   .detach().cpu().numpy().tolist())

        tot_loss += losses['total'].item()
        for k, v in losses.items():
            if k != 'total':
                lp[k] += (v if isinstance(v, float) else v.item())

        if (step + 1) % 10 == 0:
            f1 = f1_score(all_ht, all_hp,
                          average='macro', zero_division=0) * 100
            pbar.set_postfix(
                loss=f"{tot_loss/(step+1):.3f}",
                harm_f1=f"{f1:.1f}%")

    n    = len(loader)
    avg  = tot_loss / n
    f1   = f1_score(all_ht, all_hp, average='macro', zero_division=0)
    acc  = accuracy_score(all_ht, all_hp)
    m, s = divmod(int(time.time() - t0), 60)

    print(f"    ↳ {phase.upper():5s}  loss={avg:.4f}  "
          f"[harm={lp['harmonic']/n:.3f}  "
          f"dyn={lp['dynamic']/n:.3f}  "
          f"tex={lp['texture']/n:.3f}  "
          f"tmp={lp['tempo']/n:.4f}]  "
          f"harm_f1={f1*100:.2f}%  ({m}m{s:02d}s)")

    return avg, {'macro_f1': f1, 'accuracy': acc, 'loss': avg}


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Plots                                                          │
# └──────────────────────────────────────────────────────────────────────────┘

DARK_BG  = '#0f0f1a'
PANEL_BG = '#1a1a2e'
ACCENT   = ['#2ecc71', '#e74c3c', '#f39c12', '#3498db',
            '#9b59b6', '#1abc9c', '#e67e22', '#ecf0f1']


def _style_ax(ax, title='', xlabel='', ylabel=''):
    ax.set_facecolor(PANEL_BG)
    ax.tick_params(colors='#cccccc', labelsize=8)
    ax.set_title(title, color='white', fontsize=9, fontweight='bold', pad=5)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=8)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=8)
    for sp in ax.spines.values():
        sp.set_edgecolor('#333355')


def plot_training_curves(history: dict):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 4 — Music Planner Training',
                 color='white', fontsize=11, fontweight='bold')

    ep_tr = [h['epoch'] for h in history['train']]
    ep_va = [h['epoch'] for h in history['val']]

    axes[0].plot(ep_tr, [h['loss'] for h in history['train']],
                 color='#3498db', lw=2, marker='o', ms=3, label='Train')
    axes[0].plot(ep_va, [h['loss'] for h in history['val']],
                 color='#e74c3c', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[0].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    _style_ax(axes[0], 'Total Loss', 'Epoch', 'Loss')

    axes[1].plot(ep_tr, [h['f1'] * 100 for h in history['train']],
                 color='#2ecc71', lw=2, marker='o', ms=3, label='Train')
    axes[1].plot(ep_va, [h['f1'] * 100 for h in history['val']],
                 color='#f39c12', lw=2, marker='s', ms=3,
                 linestyle='--', label='Val')
    axes[1].legend(fontsize=8, facecolor=PANEL_BG, labelcolor='white')
    axes[1].set_ylim(0, 100)
    _style_ax(axes[1], 'Harmonic Style Macro F1', 'Epoch', 'Macro F1 (%)')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m4_01_training_curves.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m4_01_training_curves.png")


def plot_target_distributions(samples: list):
    """4-panel: distributions of the 4 most important targets."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.patch.set_facecolor(DARK_BG)
    fig.suptitle('Module 4 — Training Target Distributions',
                 color='white', fontsize=11, fontweight='bold')

    panels = [
        ('harmonic_style', HARMONIC_NAMES,  axes[0][0]),
        ('dynamic_shape',  DYNAMIC_NAMES,   axes[0][1]),
        ('texture',        TEXTURE_NAMES,   axes[1][0]),
        ('transition',     TRANSITION_NAMES,axes[1][1]),
    ]
    for key, names, ax in panels:
        vals   = [s[key] for s in samples]
        counts = Counter(vals)
        cv     = [counts.get(i, 0) for i in range(len(names))]
        total  = max(sum(cv), 1)
        colors = ACCENT[:len(names)]
        bars   = ax.barh(names, cv, color=colors,
                         alpha=0.85, edgecolor='white', linewidth=0.4)
        for i, (bar, v) in enumerate(zip(bars, cv)):
            ax.text(v + max(cv) * 0.01, i,
                    f'{v/total*100:.0f}%',
                    va='center', color='white', fontsize=7)
        _style_ax(ax, key.replace('_', ' ').title(), 'Count', '')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m4_00_target_distributions.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m4_00_target_distributions.png")


def plot_instrument_heatmap(all_prompts: list):
    """Heatmap: instrument activation rate per dominant emotion."""
    mat = np.zeros((len(EMOTION_NAMES), CFG4.NUM_INSTRUMENTS))
    cnt = Counter()
    for p in all_prompts:
        e = p['dominant_emotion']
        cnt[e] += 1
        mat[e] += np.array(p['instruments_pred'])
    for e in range(len(EMOTION_NAMES)):
        if cnt[e] > 0:
            mat[e] /= cnt[e]

    fig, ax = plt.subplots(figsize=(16, 5))
    fig.patch.set_facecolor(DARK_BG)
    sns.heatmap(mat, annot=True, fmt='.2f',
                xticklabels=INSTRUMENT_NAMES,
                yticklabels=EMOTION_NAMES,
                cmap='Blues', ax=ax,
                linewidths=0.3, linecolor='#222244')
    ax.set_facecolor(PANEL_BG)
    ax.set_title('Instrument Activation Rate by Dominant Emotion',
                 color='white', fontsize=9, fontweight='bold')
    ax.set_xlabel('Instrument', color='#aaaaaa', fontsize=8)
    ax.set_ylabel('Emotion',    color='#aaaaaa', fontsize=8)
    ax.tick_params(colors='#cccccc', labelsize=7)
    plt.xticks(rotation=40, ha='right')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m4_02_instrument_heatmap.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m4_02_instrument_heatmap.png")


def plot_tempo_by_arc(all_prompts: list):
    """Violin: predicted BPM distribution per arc phase."""
    arc_tempos = defaultdict(list)
    for p in all_prompts:
        arc_tempos[p['arc_phase']].append(p['tempo_bpm'])

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor(DARK_BG)
    positions = list(range(len(ARC_PHASES)))
    data      = [arc_tempos.get(i, [80]) for i in positions]
    parts     = ax.violinplot(data, positions=positions,
                               showmeans=True, showmedians=False)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(ACCENT[i % len(ACCENT)])
        pc.set_alpha(0.75)
    parts['cmeans'].set_color('white')
    ax.set_xticks(positions)
    ax.set_xticklabels(ARC_PHASES, color='#cccccc', fontsize=9)
    ax.set_ylim(CFG4.TEMPO_BASE - 5,
                CFG4.TEMPO_BASE + CFG4.TEMPO_RANGE + 5)
    _style_ax(ax, 'Predicted BPM Distribution by Arc Phase',
              'Arc Phase', 'BPM')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m4_03_tempo_by_arc.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m4_03_tempo_by_arc.png")


def plot_harmonic_by_emotion(all_prompts: list):
    """Stacked bar: harmonic style breakdown per dominant emotion."""
    mat = np.zeros((len(EMOTION_NAMES), CFG4.NUM_HARMONIC))
    cnt = Counter()
    for p in all_prompts:
        e    = p['dominant_emotion']
        h    = HARMONIC_NAMES.index(p['harmonic_style'])
        mat[e][h] += 1
        cnt[e] += 1
    for e in range(len(EMOTION_NAMES)):
        if cnt[e] > 0:
            mat[e] /= cnt[e]

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor(DARK_BG)
    x     = np.arange(len(EMOTION_NAMES))
    bottom= np.zeros(len(EMOTION_NAMES))
    for h_i, h_name in enumerate(HARMONIC_NAMES):
        bars = ax.bar(x, mat[:, h_i], bottom=bottom,
                      label=h_name, color=ACCENT[h_i % len(ACCENT)],
                      alpha=0.85, edgecolor='white', linewidth=0.3)
        bottom += mat[:, h_i]
    ax.set_xticks(x)
    ax.set_xticklabels(EMOTION_NAMES, color='#cccccc', fontsize=9)
    ax.legend(fontsize=7, facecolor=PANEL_BG, labelcolor='white',
              loc='upper right', ncol=2)
    _style_ax(ax, 'Harmonic Style Breakdown by Dominant Emotion',
              'Emotion', 'Proportion')

    plt.tight_layout()
    p = f"{PLOT_DIR}/m4_04_harmonic_by_emotion.png"
    plt.savefig(p, dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.close()
    print(f"  ✓ m4_04_harmonic_by_emotion.png")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — Load + Join M2 and M3 Data                                     │
# │                                                                          │
# │  M2: scene_vectors.pt  → scene_metadata list, keyed by scene_id tuple  │
# │  M3: scene_narrative.pt → episode_narratives list                       │
# │                           each episode has per-scene narrative_vectors, │
# │                           tension_scores, arc_phases, reversal_scores   │
# │                                                                          │
# │  JOIN strategy:                                                          │
# │    Iterate M3 episodes → for each scene j at position j in episode ep  │
# │    look up the M2 entry by scene_id to get scene_vector + emotion_probs │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "=" * 62)
print("  LOADING MODULE 2 + MODULE 3 OUTPUTS")
print("=" * 62)

# ── Load M2 ──────────────────────────────────────────────────────────────
m2_data        = torch.load(CFG4.SCENE_VEC_PT, map_location='cpu', weights_only=False)
scene_metadata = m2_data['scene_metadata']

# Index M2 by scene_id (convert to hashable tuple of ints)
m2_by_id = {}
for meta in scene_metadata:
    sid = tuple(int(x) for x in meta['scene_id'])
    m2_by_id[sid] = meta

print(f"  M2 scenes loaded  : {len(m2_by_id):,}")

# ── Load M3 ──────────────────────────────────────────────────────────────
m3_data        = torch.load(CFG4.NARR_PT, map_location='cpu', weights_only=False)
episode_narrs  = m3_data['episode_narratives']

print(f"  M3 episodes loaded: {len(episode_narrs):,}")

# ── Join M2 + M3 per scene ───────────────────────────────────────────────
joined = []
unmatched = 0

for ep in episode_narrs:
    narr_vecs    = ep['narrative_vectors']     # (S, 256) tensor
    tension_seq  = ep['tension_scores']        # list[float]
    arc_seq      = ep['arc_phases']            # list[int]
    reversal_seq = ep['reversal_scores']       # list[float]
    scene_ids    = ep.get('scene_ids', [])
    S            = ep['num_scenes']

    for j in range(S):
        if j >= len(scene_ids):
            unmatched += 1
            continue
        sid = tuple(int(x) for x in scene_ids[j])
        if sid not in m2_by_id:
            unmatched += 1
            continue

        m2 = m2_by_id[sid]

        # ── Pull scene_vector and emotion_probs from M2 ───────────────
        sv = m2['scene_vector']
        if not isinstance(sv, torch.Tensor):
            sv = torch.tensor(sv, dtype=torch.float32)
        sv = sv.float()

        # emotion_probs: (7,) softmax from M2 — feeds EmotionContextEncoder
        ep_probs = m2.get('emotion_probs')
        if ep_probs is None:
            ep_probs = torch.zeros(CFG4.EMO_PROB_DIM)
            dom = int(m2.get('dominant_emotion', 0))
            ep_probs[dom] = 1.0
        if not isinstance(ep_probs, torch.Tensor):
            ep_probs = torch.tensor(ep_probs, dtype=torch.float32)
        ep_probs = ep_probs.float()

        # ── Pull narrative_vector from M3 ─────────────────────────────
        nv = narr_vecs[j]
        if not isinstance(nv, torch.Tensor):
            nv = torch.tensor(nv, dtype=torch.float32)
        nv = nv.float()

        # ── Derive all music targets ───────────────────────────────────
        dom_emotion = int(m2.get('dominant_emotion', 0))
        s_type      = int(m2.get('scene_type', 0))

        targets = derive_music_targets(
            dominant_emotion  = dom_emotion,
            scene_type        = s_type,
            arousal_score     = float(m2.get('arousal_score',
                                   m2.get('arousal_pred', 0.0))),
            conflict_score    = float(m2.get('conflict_score',
                                   m2.get('conflict_pred', 0.0))),
            trajectory_score  = float(m2.get('trajectory_score',
                                   m2.get('trajectory_pred', 0.0))),
            variance_score    = float(m2.get('variance_score',
                                   m2.get('variance_pred', 0.0))),
            pacing_score      = float(m2.get('pacing_score', 0.5)),
            emotion_dist      = (m2['emotion_dist']
                                 if isinstance(m2['emotion_dist'], np.ndarray)
                                 else np.array(m2['emotion_dist'],
                                               dtype=np.float32)),
            tension_score     = float(tension_seq[j]),
            arc_phase         = int(arc_seq[j]),
            reversal_score    = float(reversal_seq[j]),
        )

        joined.append({
            # Inputs
            'scene_vector'     : sv,          # (256,)
            'narrative_vector' : nv,          # (256,)
            'emotion_probs'    : ep_probs,    # (7,)
            # Metadata (for inference + plotting)
            'scene_id'         : sid,
            'episode_id'       : ep['episode_id'],
            'dominant_emotion' : dom_emotion,
            'scene_type'       : s_type,
            'arc_phase'        : int(arc_seq[j]),
            'tension_score'    : float(tension_seq[j]),
            'reversal_score'   : float(reversal_seq[j]),
            # All derived music targets
            **targets,
        })

print(f"  Joined scenes : {len(joined):,}")
print(f"  Unmatched     : {unmatched}")

# ── Sanity print ─────────────────────────────────────────────────────────
s0 = joined[0]
bpm0 = CFG4.TEMPO_BASE + int(float(s0['tempo_norm']) * CFG4.TEMPO_RANGE)
print(f"\n  Sample scene {s0['scene_id']}:")
print(f"    emotion  = {EMOTION_NAMES[s0['dominant_emotion']]}"
      f"  arc = {ARC_PHASES[s0['arc_phase']]}"
      f"  tension = {s0['tension_score']:.2f}")
print(f"    → tempo = {bpm0} BPM"
      f"   harmonic = {HARMONIC_NAMES[s0['harmonic_style']]}"
      f"   texture = {TEXTURE_NAMES[s0['texture']]}")
print(f"    → instruments: "
      f"{[INSTRUMENT_NAMES[i] for i,v in enumerate(s0['instruments']) if v>0.5]}")


# ── Build dataset + splits ────────────────────────────────────────────────
full_dataset = MusicDataset(joined)

all_idx  = list(range(len(full_dataset)))
strat    = [full_dataset.samples[i]['harmonic_style'] for i in all_idx]

tr_va, te = train_test_split(
    all_idx, test_size=CFG4.TEST_SPLIT,
    stratify=strat, random_state=SEED)
tr_va_s   = [strat[i] for i in tr_va]
tr, va    = train_test_split(
    tr_va,
    test_size=CFG4.VAL_SPLIT / (1.0 - CFG4.TEST_SPLIT),
    stratify=tr_va_s, random_state=SEED)

print(f"\n  Split: train={len(tr)}  val={len(va)}  test={len(te)}")

_kw = dict(collate_fn=collate_music, num_workers=2, pin_memory=True)
train_loader = DataLoader(Subset(full_dataset, tr),
                          CFG4.BATCH_SIZE, shuffle=True,  **_kw)
val_loader   = DataLoader(Subset(full_dataset, va),
                          CFG4.BATCH_SIZE, shuffle=False, **_kw)
test_loader  = DataLoader(Subset(full_dataset, te),
                          CFG4.BATCH_SIZE, shuffle=False, **_kw)

# ── Class weights for imbalanced heads ───────────────────────────────────
def _class_weights(key: str, n_cls: int, items: list) -> torch.Tensor:
    labs = [s[key] for s in items]
    cnt  = np.bincount(labs, minlength=n_cls).astype(float)
    cnt  = np.where(cnt == 0, 1, cnt)
    raw  = len(labs) / (n_cls * cnt)
    return torch.tensor(
        np.clip(raw, 0.1, CFG4.MAX_WEIGHT),
        dtype=torch.float32).to(DEVICE)

tr_samples = [full_dataset.samples[i] for i in tr]
cw = {
    'harmonic_style': _class_weights('harmonic_style', CFG4.NUM_HARMONIC, tr_samples),
    'dynamic_shape' : _class_weights('dynamic_shape',  CFG4.NUM_DYNAMIC,  tr_samples),
    'texture'       : _class_weights('texture',        CFG4.NUM_TEXTURE,  tr_samples),
    'motion_type'   : _class_weights('motion_type',    CFG4.NUM_MOTION,   tr_samples),
    'transition'    : _class_weights('transition',     CFG4.NUM_TRANSITION,tr_samples),
}

print(f"\n  Class weights (harmonic_style): "
      f"{[f'{w:.2f}' for w in cw['harmonic_style'].cpu().numpy()]}")

# Visualise target distributions
plot_target_distributions(joined)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — Build Model + Train                                            │
# └──────────────────────────────────────────────────────────────────────────┘

model4 = MusicPlannerV2(
    input_dim  = CFG4.INPUT_DIM,
    trunk_dims = CFG4.TRUNK_DIMS,
    dropout    = CFG4.DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model4.parameters())
train_params = sum(p.numel() for p in model4.parameters() if p.requires_grad)
print(f"\n  MusicPlannerV2 total params  : {total_params:,}")
print(f"  MusicPlannerV2 trainable     : {train_params:,}")
print(f"  EmotionContextEncoder params : "
      f"{sum(p.numel() for p in model4.emo_encoder.parameters()):,}")

loss_fn4   = MusicPlannerLoss(class_weights=cw)
optimizer4 = optim.Adam(model4.parameters(), lr=CFG4.LR,
                        weight_decay=1e-5)
scheduler4 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer4, mode='min', patience=3,
    factor=0.5, min_lr=1e-6)

print("\n" + "═" * 62)
print("  TRAINING — Music Planner V2")
print(f"  Input      : scene(256) + narrative(256) + emo_ctx(64) = 576d")
print(f"  Trunk      : 576 → 512 → 256 → 128  (3 bottleneck layers)")
print(f"  Heads      : 3 regression + 9 classification + 1 multi-label")
print(f"  Train/Val/Test: {len(tr)} / {len(va)} / {len(te)} scenes")
print(f"  Epochs {CFG4.EPOCHS}   Batch {CFG4.BATCH_SIZE}   LR {CFG4.LR}")
print("═" * 62)

history    = {'train': [], 'val': []}
best_f1    = 0.0
patience_c = 0

for epoch in range(CFG4.EPOCHS):
    tr_l, tr_m = run_epoch(model4, train_loader, loss_fn4,
                           optimizer4, 'train', epoch, CFG4.EPOCHS)
    va_l, va_m = run_epoch(model4, val_loader,   loss_fn4,
                           None, 'val', epoch, CFG4.EPOCHS)

    scheduler4.step(va_l)

    history['train'].append({'epoch': epoch+1,
                             'loss': tr_l, 'f1': tr_m['macro_f1']})
    history['val'].append(  {'epoch': epoch+1,
                             'loss': va_l, 'f1': va_m['macro_f1']})

    if va_m['macro_f1'] > best_f1:
        best_f1    = va_m['macro_f1']
        patience_c = 0
        torch.save({
            'model_state' : model4.state_dict(),
            'macro_f1'    : best_f1,
            'config' : {
                'input_dim'       : CFG4.INPUT_DIM,
                'trunk_dims'      : CFG4.TRUNK_DIMS,
                'dropout'         : CFG4.DROPOUT,
                'tempo_base'      : CFG4.TEMPO_BASE,
                'tempo_range'     : CFG4.TEMPO_RANGE,
                'emotion_names'   : EMOTION_NAMES,
                'arc_phases'      : ARC_PHASES,
                'harmonic_names'  : HARMONIC_NAMES,
                'dynamic_names'   : DYNAMIC_NAMES,
                'instrument_names': INSTRUMENT_NAMES,
                'scene_type_names': SCENE_TYPE_NAMES,
            },
        }, CFG4.BEST_CKPT)
        print(f"    ✓ Saved music_planner_best.pt  "
              f"(harm_f1={best_f1*100:.2f}%)")
    else:
        patience_c += 1
        if patience_c >= CFG4.EARLY_STOP:
            print(f"\n  ⏹ Early stop at epoch {epoch+1}")
            break

with open(f"{CFG4.CKPT_DIR}/m4_history.json", 'w') as f:
    json.dump(history, f, indent=2)

plot_training_curves(history)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Final Evaluation                                               │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 62)
print("  FINAL EVALUATION — TEST SET")
print("═" * 62)

ckpt = torch.load(CFG4.BEST_CKPT, map_location=DEVICE)
model4.load_state_dict(ckpt['model_state'])
model4.eval()

# Evaluate all classification heads on test split
HEAD_EVAL = [
    ('harmonic_style', HARMONIC_NAMES,  CFG4.NUM_HARMONIC),
    ('dynamic_shape',  DYNAMIC_NAMES,   CFG4.NUM_DYNAMIC),
    ('motion_type',    MOTION_NAMES,    CFG4.NUM_MOTION),
    ('texture',        TEXTURE_NAMES,   CFG4.NUM_TEXTURE),
    ('tonality',       TONALITY_NAMES,  CFG4.NUM_TONALITY),
    ('transition',     TRANSITION_NAMES,CFG4.NUM_TRANSITION),
    ('rhythm_style',   RHYTHM_NAMES,    CFG4.NUM_RHYTHM),
]

head_f1s = {}
print(f"\n  {'Head':<22} {'Macro F1':>10}  {'Accuracy':>10}")
print(f"  {'-'*44}")
for head_key, names, _ in HEAD_EVAL:
    all_t, all_p = [], []
    with torch.no_grad():
        for batch in test_loader:
            sv  = batch['scene_vector'].to(DEVICE)
            nv  = batch['narrative_vector'].to(DEVICE)
            ep  = batch['emotion_probs'].to(DEVICE)
            out = model4(sv, nv, ep)
            all_t.extend(batch[head_key].numpy().tolist())
            all_p.extend(
                out[head_key].argmax(-1).cpu().numpy().tolist())
    f1  = f1_score(all_t, all_p, average='macro', zero_division=0)
    acc = accuracy_score(all_t, all_p)
    head_f1s[head_key] = f1
    print(f"  {head_key:<22} {f1*100:>9.2f}%  {acc*100:>9.2f}%")

print(f"\n  Average macro F1 : "
      f"{np.mean(list(head_f1s.values()))*100:.2f}%")
print(f"  Best val harm F1 : {best_f1*100:.2f}%")

# Detailed report for primary head
print(f"\n  ── Harmonic Style — Full Classification Report ──")
all_t, all_p = [], []
with torch.no_grad():
    for batch in test_loader:
        sv  = batch['scene_vector'].to(DEVICE)
        nv  = batch['narrative_vector'].to(DEVICE)
        ep  = batch['emotion_probs'].to(DEVICE)
        out = model4(sv, nv, ep)
        all_t.extend(batch['harmonic_style'].numpy().tolist())
        all_p.extend(
            out['harmonic_style'].argmax(-1).cpu().numpy().tolist())
print(classification_report(
    all_t, all_p,
    target_names=HARMONIC_NAMES,
    labels=list(range(CFG4.NUM_HARMONIC)),
    zero_division=0))


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 14 — Generate Music Prompts (Pipeline Final Output)                 │
# │                                                                          │
# │  Runs the trained model over ALL scenes.                                │
# │  Assembles the final music descriptor string for each scene.           │
# │  This is the deliverable the pipeline produces.                        │
# │                                                                          │
# │  Saved to music_prompts.json:                                           │
# │    scene_id, episode_id, arc_phase, tension, dominant_emotion,         │
# │    scene_type, tempo_bpm, all 13 descriptors, instruments list,        │
# │    prompt string (design-plan format)                                   │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n" + "═" * 62)
print("  GENERATING MUSIC PROMPTS — ALL SCENES")
print("═" * 62)

full_loader = DataLoader(
    full_dataset, batch_size=64,
    collate_fn=collate_music,
    shuffle=False, num_workers=2)

all_prompts = []

with torch.no_grad():
    for i, batch in enumerate(tqdm(full_loader,
                                    desc="  Generating")):
        sv  = batch['scene_vector'].to(DEVICE)
        nv  = batch['narrative_vector'].to(DEVICE)
        ep  = batch['emotion_probs'].to(DEVICE)
        out = model4(sv, nv, ep)
        B   = sv.size(0)

        for b in range(B):
            g = i * 64 + b
            if g >= len(full_dataset):
                break
            s = full_dataset.samples[g]

            # Per-sample output dict (tensor, already on CPU after .item())
            out_b = {k: v[b].cpu()
                     for k, v in out.items()
                     if isinstance(v, torch.Tensor)}

            prompt_str = model4.assemble_prompt(
                out_b,
                scene_id   = s['scene_id'],
                arc_phase  = s['arc_phase'],
                tension    = s['tension_score'],
                emotion    = EMOTION_NAMES[s['dominant_emotion']],
                scene_type = SCENE_TYPE_NAMES[s['scene_type']])

            tempo_bpm = MusicPlannerV2.tempo_to_bpm(
                float(out_b['tempo_score'].item()))

            instr_pred = out_b['instruments'].numpy()

            all_prompts.append({
                # Scene identity
                'scene_id'      : s['scene_id'],
                'episode_id'    : s['episode_id'],
                # Narrative context
                'arc_phase'     : s['arc_phase'],
                'arc_phase_name': ARC_PHASES[s['arc_phase']],
                'tension_score' : s['tension_score'],
                'reversal_score': s['reversal_score'],
                # Emotional context
                'dominant_emotion' : s['dominant_emotion'],
                'emotion_name'     : EMOTION_NAMES[s['dominant_emotion']],
                'scene_type_name'  : SCENE_TYPE_NAMES[s['scene_type']],
                # ── 13 Music Descriptors ──────────────────────────────
                'tempo_bpm'        : tempo_bpm,
                'intensity'        : round(float(out_b['intensity'].item()), 3),
                'valence'          : round(float(out_b['valence'].item()), 3),
                'tonality'         : TONALITY_NAMES[out_b['tonality'].argmax().item()],
                'harmonic_style'   : HARMONIC_NAMES[out_b['harmonic_style'].argmax().item()],
                'motion_type'      : MOTION_NAMES[out_b['motion_type'].argmax().item()],
                'rhythm_style'     : RHYTHM_NAMES[out_b['rhythm_style'].argmax().item()],
                'rhythm_density'   : DENSITY_NAMES[out_b['rhythm_density'].argmax().item()],
                'dynamic_shape'    : DYNAMIC_NAMES[out_b['dynamic_shape'].argmax().item()],
                'texture'          : TEXTURE_NAMES[out_b['texture'].argmax().item()],
                'transition'       : TRANSITION_NAMES[out_b['transition'].argmax().item()],
                'instruments_pred' : instr_pred.tolist(),
                'instruments_on'   : [INSTRUMENT_NAMES[i]
                                      for i, v in enumerate(instr_pred)
                                      if v > 0.5],
                # ── Prompt (design plan format) ───────────────────────
                'prompt'           : prompt_str,
            })

with open(CFG4.PROMPTS_OUT, 'w') as f:
    json.dump(all_prompts, f, indent=2)

# ── Print sample prompts (the actual pipeline output) ──────────────────
print(f"\n  Generated {len(all_prompts):,} music prompts\n")
print(f"  ── SAMPLE PROMPTS ───────────────────────────────────────────")
for p in all_prompts[:4]:
    print()
    print(f"  {p['prompt']}")
    print(f"  " + "─" * 58)

# Summary statistics
bpms     = [p['tempo_bpm'] for p in all_prompts]
harm_cnt = Counter(p['harmonic_style'] for p in all_prompts)
instr_cnt= Counter(i for p in all_prompts
                   for i in p['instruments_on'])

print(f"\n  Tempo stats:  "
      f"min={min(bpms)}  max={max(bpms)}  "
      f"mean={np.mean(bpms):.0f} BPM")

print(f"\n  Harmonic style distribution:")
for name, cnt_v in sorted(harm_cnt.items(), key=lambda x: -x[1]):
    bar = '█' * int(cnt_v / len(all_prompts) * 30)
    print(f"    {name:<25}  {cnt_v:>5}  ({cnt_v/len(all_prompts)*100:.0f}%)  {bar}")

print(f"\n  Top instruments:")
for name, cnt_v in instr_cnt.most_common(8):
    bar = '█' * int(cnt_v / len(all_prompts) * 30)
    print(f"    {name:<22}  {cnt_v:>5}  ({cnt_v/len(all_prompts)*100:.0f}%)  {bar}")

# Generate remaining plots
plot_instrument_heatmap(all_prompts)
plot_tempo_by_arc(all_prompts)
plot_harmonic_by_emotion(all_prompts)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 15 — Zip All Outputs                                                │
# └──────────────────────────────────────────────────────────────────────────┘

import zipfile
from IPython.display import FileLink, display

ZIP_PATH = "/kaggle/working/module4_outputs.zip"
print("\n" + "═" * 62)
print("  ZIPPING MODULE 4 OUTPUTS")
print("═" * 62)

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for src_dir, arc_name in [
        ("/kaggle/working/m4_checkpoints", "m4_checkpoints"),
        ("/kaggle/working/plots",          "plots"),
    ]:
        for fpath in Path(src_dir).rglob('*'):
            if fpath.is_file():
                arcname = arc_name + '/' + str(
                    fpath.relative_to(src_dir))
                zf.write(str(fpath), arcname)
                print(f"  + {arcname}  "
                      f"({fpath.stat().st_size/1024:.1f} KB)")

print(f"\n  ✓ module4_outputs.zip  "
      f"({Path(ZIP_PATH).stat().st_size/1024/1024:.1f} MB)")

print(f"\n  {'═'*58}")
print(f"  PIPELINE COMPLETE")
print(f"  {'═'*58}")
print(f"  M1 : DTE utterance encoder  → 256d/utterance + emotion label")
print(f"  M2 : Scene encoder BiLSTM   → 256d/scene + emotion_curve")
print(f"  M3 : Narrative LSTM (causal)→ 256d/scene + tension + arc")
print(f"  M4 : Music Planner V2       → 13 descriptors + prompt")
print(f"  {'─'*58}")
print(f"  Input   : 576d  "
      f"(scene256 + narrative256 + emo_ctx64)")
print(f"  Trunk   : 576→512→256→128  (3 bottleneck layers)")
print(f"  Scenes  : {len(all_prompts):,} music prompts generated")
print(f"  Prompts : {CFG4.PROMPTS_OUT}")
print(f"  Harm F1 : {head_f1s.get('harmonic_style', 0)*100:.2f}%  "
      f"(best val {best_f1*100:.2f}%)")
print(f"  {'═'*58}")

display(FileLink(ZIP_PATH, result_html_prefix="⬇️  Download: "))

  Device : cuda
  GPU    : Tesla T4
  Config ready ✓
  INPUT_DIM = 576  (scene(256) + narrative(256) + emo_ctx(64))
  TRUNK     = 576 → 512 → 256 → 128
  Music vocabulary loaded:
    Harmonic styles : ['diatonic_warm', 'melodic_minor', 'dissonant_sparse', 'cluster_dissonance', 'chromatic_aggressive', 'modal_mystical', 'pedal_tone', 'atonal']
    Dynamic shapes  : ['slow_build', 'crescendo', 'sudden_hit_then_drive', 'drop_then_soft_build', 'sustained_peak', 'decrescendo', 'flat', 'pulse_and_release']
    Instruments     : 15 groups

  LOADING MODULE 2 + MODULE 3 OUTPUTS
  M2 scenes loaded  : 1,337
  M3 episodes loaded: 180
  Joined scenes : 1,333
  Unmatched     : 0

  Sample scene (1, 1, 19):
    emotion  = neutral  arc = setup  tension = 0.19
    → tempo = 83 BPM   harmonic = pedal_tone   texture = ambient_minimal
    → instruments: ['solo_piano', 'woodwinds']

  Split: train=933  val=200  test=200

  Class weights (harmonic_style): ['0.90', '3.53', '10.00', '10.00', '1.01', '10.00', 

Ep 01/35 ▶ Train:   0%|                         | 0/30 [00:00<?, ?it/s]

    ↳ TRAIN  loss=12.5369  [harm=2.590  dyn=3.042  tex=2.161  tmp=0.0530]  harm_f1=19.65%  (0m00s)


Ep 01/35   Val  :   0%|                         | 0/7 [00:00<?, ?it/s]

    ↳ VAL    loss=9.7144  [harm=1.975  dyn=2.473  tex=1.766  tmp=0.0208]  harm_f1=43.52%  (0m00s)
    ✓ Saved music_planner_best.pt  (harm_f1=43.52%)


...

Ep 22/35   Val  :   0%|                         | 0/7 [00:00<?, ?it/s]

    ↳ VAL    loss=7.6378  [harm=2.075  dyn=1.829  tex=1.339  tmp=0.0020]  harm_f1=63.79%  (0m00s)

  ⏹ Early stop at epoch 22
  ✓ m4_01_training_curves.png

══════════════════════════════════════════════════════════════
  FINAL EVALUATION — TEST SET
══════════════════════════════════════════════════════════════

  Head                     Macro F1    Accuracy
  --------------------------------------------
  harmonic_style             74.14%      78.50%
  dynamic_shape              69.03%      85.00%
  motion_type                74.78%      83.50%
  texture                    70.62%      75.50%
  tonality                   86.26%      86.50%
  transition                 69.82%      92.00%
  rhythm_style               45.65%      85.50%

  Average macro F1 : 70.04%
  Best val harm F1 : 66.72%

  ── Harmonic Style — Full Classification Report ──
                      precision    recall  f1-score   support

       diatonic_warm       0.49      0.81      0.61        27
       melodic_minor

  Generating:   0%|          | 0/21 [00:00<?, ?it/s]


  Generated 1,333 music prompts

  ── SAMPLE PROMPTS ───────────────────────────────────────────

  Scene S1E1D19 — 67 BPM, major key, diatonic_warm harmony. Motion: static_slow. Dynamic shape: slow_build. Instruments: solo_piano, woodwinds.
  arc:setup  tension:0.19  dynamics:pp  rhythm:driving_8th(minimal)  texture:ambient_minimal  transition→next:crossfade  [emotion:neutral, type:dialogue]
  ──────────────────────────────────────────────────────────

  Scene S1E1D152 — 53 BPM, minor key, melodic_minor harmony. Motion: slow_release. Dynamic shape: slow_build. Instruments: solo_piano, woodwinds.
  arc:setup  tension:0.00  dynamics:mp  rhythm:sustained(moderate)  texture:acoustic_intimate  transition→next:crossfade  [emotion:neutral, type:dialogue]
  ──────────────────────────────────────────────────────────

  Scene S1E1D164 — 76 BPM, minor key, atonal harmony. Motion: static_slow. Dynamic shape: pulse_and_release. Instruments: french_horn.
  arc:rising  tension:0.01  dynamics:pp  rh

/kaggle/working/module4_outputs.zip

In [37]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE INFERENCE  —  Screenplay → Music Prompt                       ║
# ╠══════════════════════════════════════════════════════════════════════════╣
# ║  Runs any screenplay through the full trained pipeline:                 ║
# ║    M1 (DTE) → M2 (SceneEncoderV3) → M3 (NarrativeLSTM) → M4 (Music)  ║
# ║                                                                         ║
# ║  All model classes below are copied VERBATIM from training code.        ║
# ║  All checkpoint keys match what each module's torch.save() wrote.      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 1 — Imports                                                         │
# └──────────────────────────────────────────────────────────────────────────┘

import re, json, math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device : {DEVICE}")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 2 — Paths  (edit these to match your Kaggle input paths)           │
# └──────────────────────────────────────────────────────────────────────────┘

M1_CKPT      = "/kaggle/input/datasets/suyashnpande/module1-phase2/checkpoints/module1_phase2_best.pt"
M2_CKPT      = "/kaggle/input/datasets/suyashnpande/module2-outputs/m2_checkpoints/scene_encoder_best.pt"
M3_CKPT      = "/kaggle/input/datasets/suyashnpande/module3-outputs/m3_checkpoints/narrative_lstm_best.pt"
M4_CKPT      = "/kaggle/input/datasets/suyashnpande/module4-outputs/m4_checkpoints/music_planner_best.pt"
PROMPTS_JSON = "/kaggle/input/datasets/suyashnpande/module4-outputs/m4_checkpoints/music_prompts.json"


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 3 — Shared label constants  (must match training exactly)           │
# └──────────────────────────────────────────────────────────────────────────┘

EMOTION_NAMES    = ['neutral', 'joyful', 'sad', 'mad',
                    'scared', 'peaceful', 'powerful']
IDX2LABEL        = {i: e for i, e in enumerate(EMOTION_NAMES)}
LABEL2IDX        = {e: i for i, e in enumerate(EMOTION_NAMES)}

SCENE_TYPE_NAMES = ['dialogue', 'action', 'investigation',
                    'suspense', 'dramatic_reveal', 'transition']

ARC_PHASES       = ['setup', 'rising', 'climax', 'falling', 'resolution']

# M4 vocabulary
TONALITY_NAMES   = ['major', 'minor', 'ambiguous']
HARMONIC_NAMES   = ['diatonic_warm', 'melodic_minor', 'dissonant_sparse',
                    'cluster_dissonance', 'chromatic_aggressive',
                    'modal_mystical', 'pedal_tone', 'atonal']
MOTION_NAMES     = ['static_slow', 'rising_tension', 'chaotic',
                    'slow_release', 'steady_drive', 'sudden_drop']
RHYTHM_NAMES     = ['sustained', 'ostinato', 'sparse_pulse',
                    'rubato', 'driving_8th', 'silence']
DENSITY_NAMES    = ['minimal', 'moderate', 'dense']
DYNAMIC_NAMES    = ['slow_build', 'crescendo', 'sudden_hit_then_drive',
                    'drop_then_soft_build', 'sustained_peak',
                    'decrescendo', 'flat', 'pulse_and_release']
TEXTURE_NAMES    = ['ambient_minimal', 'layered_ambient', 'hybrid_orchestral',
                    'emotional_orchestral', 'electronic_dark', 'acoustic_intimate']
TRANSITION_NAMES = ['attacca', 'crossfade', 'hard_cut',
                    'silence_gap', 'motif_bridge']
INSTRUMENT_NAMES = ['low_strings', 'high_strings', 'solo_violin', 'solo_cello',
                    'solo_piano', 'brass_stabs', 'french_horn', 'ambient_synth',
                    'sub_bass_drone', 'metallic_percussion', 'cinematic_drums',
                    'reverse_piano', 'distorted_synth_bass', 'choir', 'woodwinds']

print("  Constants loaded ✓")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 4 — Module 1 Architecture  (verbatim from module1_v6.py)           │
# │                                                                          │
# │  Checkpoint key  : 'utt_encoder'                                        │
# │  Vocab key       : 'vocab'  (dict word→int, inside same checkpoint)    │
# │  forward() args  : token_ids, speaker_ids, turn_positions,              │
# │                    utt_history  ← (B, NUM_PREV_UTT, MAX_LEN) tokens    │
# │  return_vector=True returns (logits, fused_256d, all_attn)             │
# └──────────────────────────────────────────────────────────────────────────┘

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.d_model = d_model; self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)
        for w in [self.W_q, self.W_k, self.W_v, self.W_o]:
            nn.init.xavier_uniform_(w.weight)

    def _split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, x, mask=None):
        B, T, _ = x.shape
        Q = self._split_heads(self.W_q(x))
        K = self._split_heads(self.W_k(x))
        V = self._split_heads(self.W_v(x))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_w = torch.nan_to_num(F.softmax(scores, dim=-1), nan=0.0)
        attn_w = self.drop(attn_w)
        ctx = torch.matmul(attn_w, V)
        ctx = ctx.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(ctx), attn_w


class FeedForward(nn.Module):
    def __init__(self, d_model=256, d_ff=512, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)

    def forward(self, x):
        return self.fc2(self.drop(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, num_heads=4, d_ff=512, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, dropout)
        self.ff   = FeedForward(d_model, d_ff, dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_w = self.attn(self.ln1(x), mask)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x, attn_w


class DialogueTurnEmbedding(nn.Module):
    def __init__(self, max_turns=60, embed_dim=32):
        super().__init__()
        self.embed = nn.Embedding(max_turns, embed_dim)
        nn.init.normal_(self.embed.weight, std=0.02)

    def forward(self, turn_positions):
        return self.embed(turn_positions)


class EmotionQueryToken(nn.Module):
    def __init__(self, d_model=256, num_emotions=7):
        super().__init__()
        self.prototypes = nn.Parameter(torch.randn(num_emotions, d_model) * 0.02)
        self.gate = nn.Linear(d_model, num_emotions)
        nn.init.xavier_uniform_(self.gate.weight)

    def get_query(self, x):
        mean_x  = x.mean(dim=1)
        weights = F.softmax(self.gate(mean_x), dim=-1)
        query   = torch.matmul(weights, self.prototypes)
        return query.unsqueeze(1)


class DialogueContextGRU(nn.Module):
    """Encodes utt_history: (B, NUM_PREV_UTT, MAX_LEN) token ids → (B, 128)."""
    def __init__(self, vocab_size, embed_dim=128, d_model=256,
                 context_dim=128, num_prev=5, max_len=50, dropout=0.2):
        super().__init__()
        self.num_prev = num_prev; self.max_len = max_len
        self.tok_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.utt_proj  = nn.Sequential(
            nn.Linear(embed_dim, d_model), nn.LayerNorm(d_model), nn.ReLU())
        self.gru  = nn.GRU(d_model, context_dim, num_layers=1, batch_first=True)
        self.proj = nn.Sequential(
            nn.Linear(context_dim, context_dim), nn.LayerNorm(context_dim), nn.Tanh())
        self.drop = nn.Dropout(dropout)

    def forward(self, utt_history):
        B, P, T = utt_history.shape
        flat    = utt_history.reshape(B * P, T).long()
        flat    = flat.clamp(0, self.tok_embed.num_embeddings - 1)
        emb     = self.tok_embed(flat)
        mask    = (flat != 0).float().unsqueeze(-1)
        pooled  = (emb * mask).sum(1) / (mask.sum(1) + 1e-6)
        pooled  = self.utt_proj(pooled).view(B, P, -1)
        _, h_n  = self.gru(pooled)
        return self.drop(self.proj(h_n.squeeze(0)))


class DialogueTransformerEncoder(nn.Module):
    """
    Module 1 model class — matches module1_v6.py exactly.
    Checkpoint key: 'utt_encoder'
    Call: model.forward(token_ids, speaker_ids, turn_positions,
                        utt_history, return_vector=True)
          → (logits, fused_256d, all_attn)
    """
    def __init__(self, vocab_size, embed_dim=128, d_model=256,
                 num_heads=4, num_layers=4, d_ff=512,
                 max_len=50, max_turns=60, speaker_dim=64, turn_dim=32,
                 num_speakers=2, num_emotions=7, num_prev=5, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.tok_embed  = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed  = nn.Embedding(max_len + 1, embed_dim)
        self.spk_embed  = nn.Embedding(num_speakers, speaker_dim)
        self.turn_embed = DialogueTurnEmbedding(max_turns, turn_dim)
        nn.init.normal_(self.tok_embed.weight, std=0.02)
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        nn.init.normal_(self.spk_embed.weight, std=0.02)

        input_concat_dim = embed_dim + embed_dim + speaker_dim + turn_dim  # 352
        self.input_proj = nn.Sequential(
            nn.Linear(input_concat_dim, d_model),
            nn.LayerNorm(d_model), nn.Dropout(dropout))

        self.emo_query = EmotionQueryToken(d_model, num_emotions)
        self.layers    = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)])
        self.final_ln  = nn.LayerNorm(d_model)

        self.ctx_gru = DialogueContextGRU(
            vocab_size=vocab_size, embed_dim=embed_dim, d_model=d_model,
            context_dim=128, num_prev=num_prev, max_len=max_len, dropout=dropout)

        self.fusion = nn.Sequential(
            nn.Linear(d_model + 128, d_model),
            nn.LayerNorm(d_model), nn.ReLU(), nn.Dropout(dropout))

        self.emo_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_emotions))

    def _build_pad_mask(self, token_ids, extra_token=True):
        B, T = token_ids.shape
        pad  = (token_ids != 0).float()
        if extra_token:
            pad = torch.cat([torch.ones(B, 1, device=token_ids.device), pad], dim=1)
        return pad.unsqueeze(1).unsqueeze(2)

    def encode(self, token_ids, speaker_ids, turn_positions):
        B, T = token_ids.shape
        pos  = torch.arange(T, device=token_ids.device).unsqueeze(0).expand(B, -1)
        tok_e = self.tok_embed(token_ids)
        pos_e = self.pos_embed(pos)
        spk_e = self.spk_embed(speaker_ids).unsqueeze(1).expand(-1, T, -1)
        trn_e = self.turn_embed(turn_positions).unsqueeze(1).expand(-1, T, -1)
        x     = self.input_proj(torch.cat([tok_e, pos_e, spk_e, trn_e], dim=-1))
        query = self.emo_query.get_query(x)
        x     = torch.cat([query, x], dim=1)
        mask  = self._build_pad_mask(token_ids)
        all_attn = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            all_attn.append(attn_w)
        x = self.final_ln(x)
        return x[:, 0, :], all_attn

    def forward(self, token_ids, speaker_ids, turn_positions,
                utt_history, return_vector=False):
        cls_out, all_attn = self.encode(token_ids, speaker_ids, turn_positions)
        ctx_out           = self.ctx_gru(utt_history)
        fused             = self.fusion(torch.cat([cls_out, ctx_out], dim=-1))
        logits            = self.emo_head(fused)
        if return_vector:
            return logits, fused, all_attn
        return logits


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 5 — Module 2 Architecture  (verbatim from module2_v3.py)           │
# │                                                                          │
# │  Checkpoint key: 'model_state'                                          │
# │  forward() args: emotion_matrix (B,L,256), mask (B,L bool),            │
# │                  scene_type_ids (B,) optional                           │
# └──────────────────────────────────────────────────────────────────────────┘

class SceneEncoderV3(nn.Module):
    def __init__(self, input_dim=256, scene_type_dim=32, hidden_dim=128,
                 scene_vec_dim=256, num_emotions=7, num_scene_types=6,
                 num_sentiments=4, dropout=0.20):
        super().__init__()
        bilstm_out_dim = hidden_dim * 2   # 256

        self.scene_type_emb = nn.Embedding(num_scene_types, scene_type_dim)
        nn.init.normal_(self.scene_type_emb.weight, std=0.02)

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim + scene_type_dim, 256),
            nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout))

        self.bilstm = nn.LSTM(
            input_size=256, hidden_size=hidden_dim, num_layers=2,
            batch_first=True, bidirectional=True, dropout=dropout)

        self.per_utt_emo_head = nn.Linear(bilstm_out_dim, num_emotions)
        self.per_utt_int_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())

        self.attn_fc   = nn.Linear(bilstm_out_dim, 1)
        self.scene_ln  = nn.LayerNorm(bilstm_out_dim)
        self.scene_drop= nn.Dropout(dropout)

        self.emotion_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_emotions))
        self.transition_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_emotions * num_emotions))
        self.scene_type_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 64), nn.ReLU(),
            nn.Linear(64, num_scene_types))
        self.sentiment_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 64), nn.ReLU(),
            nn.Linear(64, num_sentiments))
        self.variance_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.conflict_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())
        self.arousal_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())
        self.trajectory_head = nn.Sequential(
            nn.Linear(bilstm_out_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Tanh())

    def forward(self, emotion_matrix, mask, scene_type_ids=None):
        B, L, _ = emotion_matrix.shape
        if scene_type_ids is not None:
            st_e = self.scene_type_emb(scene_type_ids)
        else:
            st_e = torch.zeros(B, 32, device=emotion_matrix.device)
        st_e = st_e.unsqueeze(1).expand(-1, L, -1)

        x = self.input_proj(torch.cat([emotion_matrix, st_e], dim=-1))
        h, _ = self.bilstm(x)

        per_utt_emo = self.per_utt_emo_head(h)
        per_utt_int = self.per_utt_int_head(h).squeeze(-1)

        scores = self.attn_fc(h).squeeze(-1)
        scores = scores.masked_fill(~mask, float('-inf'))
        attn_w = torch.nan_to_num(F.softmax(scores, dim=1), nan=0.0)
        scene_v = torch.bmm(attn_w.unsqueeze(1), h).squeeze(1)
        scene_v = self.scene_drop(self.scene_ln(scene_v))

        trans_mat = self.transition_head(scene_v).view(B, 7, 7)
        trans_mat = torch.softmax(trans_mat, dim=-1)

        return {
            'scene_vector':         scene_v,
            'emotion_logits':       self.emotion_head(scene_v),
            'scene_type_logits':    self.scene_type_head(scene_v),
            'sentiment_logits':     self.sentiment_head(scene_v),
            'transition_matrix':    trans_mat,
            'variance_pred':        self.variance_head(scene_v).squeeze(-1),
            'conflict_pred':        self.conflict_head(scene_v).squeeze(-1),
            'arousal_pred':         self.arousal_head(scene_v).squeeze(-1),
            'trajectory_pred':      self.trajectory_head(scene_v).squeeze(-1),
            'emotion_curve_logits': per_utt_emo,
            'intensity_curve':      per_utt_int,
            'attn_weights':         attn_w,
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 6 — Module 3 Architecture  (verbatim from module3_v1.py)           │
# │                                                                          │
# │  Checkpoint key: 'model_state' and 'char2id'                           │
# │  forward() args: scene_vectors (B,S,256), char_ids (B,S,K)            │
# └──────────────────────────────────────────────────────────────────────────┘

class TensionMemoryCell(nn.Module):
    def __init__(self, scene_dim=256):
        super().__init__()
        self.decay_gate   = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
        self.arousal_gate = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())
        self.release_gate = nn.Sequential(
            nn.Linear(scene_dim, 64), nn.ReLU(), nn.Linear(64, 1), nn.Sigmoid())

    def forward(self, scene_vector, prev_tension):
        gamma   = self.decay_gate(scene_vector)
        arousal = self.arousal_gate(scene_vector)
        release = self.release_gate(scene_vector)
        return torch.clamp(gamma * prev_tension + arousal - release, 0.0, 1.0)


class CharacterMemoryBank(nn.Module):
    def __init__(self, scene_dim=256, char_dim=64, max_chars=20):
        super().__init__()
        self.char_dim  = char_dim
        self.max_chars = max_chars
        self.gru_cell  = nn.GRUCell(scene_dim, char_dim)
        self.proj      = nn.Linear(char_dim, char_dim)
        nn.init.xavier_uniform_(self.proj.weight)

    def forward(self, scene_vector, char_ids, char_memory):
        B, K    = char_ids.shape
        updated = char_memory.clone()
        for k in range(K):
            ids   = char_ids[:, k]
            valid = (ids >= 0)
            if not valid.any(): continue
            safe_ids = ids.clamp(0, self.max_chars - 1)
            curr_mem = char_memory[
                torch.arange(B, device=scene_vector.device), safe_ids]
            new_mem  = self.gru_cell(scene_vector, curr_mem)
            valid_b  = valid.nonzero(as_tuple=True)[0]
            for b in valid_b:
                updated[b, safe_ids[b]] = new_mem[b].detach()
        active_mask = (char_ids >= 0).float()
        safe_all    = char_ids.clamp(0, self.max_chars - 1)
        active_mems = updated[
            torch.arange(B, device=scene_vector.device)
                 .unsqueeze(1).expand(B, K), safe_all]
        denom    = active_mask.sum(dim=1, keepdim=True) + 1e-6
        char_ctx = (active_mems * active_mask.unsqueeze(-1)).sum(1) / denom
        return updated, self.proj(char_ctx)


class NarrativeLSTM(nn.Module):
    def __init__(self, scene_dim=256, hidden_dim=128, narr_dim=256,
                 char_dim=64, max_chars=20, num_arc=5,
                 num_emotions=7, dropout=0.20):
        super().__init__()
        self.char_dim   = char_dim
        self.max_chars  = max_chars

        self.lstm = nn.LSTM(
            input_size=scene_dim, hidden_size=hidden_dim,
            num_layers=2, batch_first=True,
            bidirectional=False, dropout=dropout)

        self.tension_cell = TensionMemoryCell(scene_dim)
        self.char_bank    = CharacterMemoryBank(scene_dim, char_dim, max_chars)

        fuse_in = hidden_dim + 1 + char_dim   # 128+1+64 = 193
        self.fusion = nn.Sequential(
            nn.Linear(fuse_in, narr_dim),
            nn.LayerNorm(narr_dim), nn.ReLU(), nn.Dropout(dropout))

        self.arc_head = nn.Sequential(
            nn.Linear(narr_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_arc))
        self.reversal_head = nn.Sequential(
            nn.Linear(narr_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.tension_head = nn.Sequential(
            nn.Linear(narr_dim, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid())
        self.emotion_head = nn.Sequential(
            nn.Linear(narr_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_emotions))

    def forward(self, scene_vectors, char_ids):
        B, S, _ = scene_vectors.shape
        lstm_out, _ = self.lstm(scene_vectors)
        tension  = torch.zeros(B, 1, device=scene_vectors.device)
        char_mem = torch.zeros(B, self.max_chars, self.char_dim,
                               device=scene_vectors.device)
        narr_vecs, tension_preds = [], []
        arc_list, reversal_list, emotion_list = [], [], []

        for t in range(S):
            sv   = scene_vectors[:, t, :]
            lh   = lstm_out[:, t, :]
            cids = char_ids[:, t, :]
            tension  = self.tension_cell(sv, tension)
            char_mem, char_ctx = self.char_bank(sv, cids, char_mem)
            fused = self.fusion(torch.cat([lh, tension, char_ctx], dim=-1))
            narr_vecs.append(fused)
            tension_preds.append(tension)
            arc_list.append(self.arc_head(fused))
            reversal_list.append(self.reversal_head(fused))
            emotion_list.append(self.emotion_head(fused))

        return {
            'narrative_vectors': torch.stack(narr_vecs,     dim=1),
            'tension_scores':    torch.stack(tension_preds, dim=1),
            'arc_logits':        torch.stack(arc_list,      dim=1),
            'reversal_scores':   torch.stack(reversal_list, dim=1),
            'emotion_logits':    torch.stack(emotion_list,  dim=1),
        }


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 7 — Module 4 Architecture  (verbatim from module4_v2.py)           │
# │                                                                          │
# │  Checkpoint key: 'model_state'                                          │
# │  forward() args: scene_vector (B,256), narrative_vector (B,256),       │
# │                  emotion_probs (B,7)                                    │
# └──────────────────────────────────────────────────────────────────────────┘

class EmotionContextEncoder(nn.Module):
    def __init__(self, in_dim=7, out_dim=64, dropout=0.20):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32), nn.LayerNorm(32), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, out_dim), nn.LayerNorm(out_dim), nn.ReLU())

    def forward(self, ep): return self.net(ep)


class MusicPlannerV2(nn.Module):
    def __init__(self, input_dim=576, trunk_dims=(512, 256, 128), dropout=0.20):
        super().__init__()
        self.emo_encoder = EmotionContextEncoder(7, 64, dropout)
        layers, in_d = [], input_dim
        for out_d in trunk_dims:
            layers += [nn.Linear(in_d, out_d), nn.LayerNorm(out_d),
                       nn.ReLU(), nn.Dropout(dropout)]
            in_d = out_d
        self.trunk = nn.Sequential(*layers)
        H = trunk_dims[-1]  # 128

        self.tempo_head     = nn.Sequential(nn.Linear(H,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
        self.intensity_head = nn.Sequential(nn.Linear(H,32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
        self.valence_head   = nn.Sequential(nn.Linear(H,32),nn.ReLU(),nn.Linear(32,1),nn.Tanh())
        self.tonality_head    = nn.Linear(H, 3)
        self.harmonic_head    = nn.Linear(H, 8)
        self.motion_head      = nn.Linear(H, 6)
        self.rhythm_head      = nn.Linear(H, 6)
        self.density_head     = nn.Linear(H, 3)
        self.dynamic_head     = nn.Linear(H, 8)
        self.texture_head     = nn.Linear(H, 6)
        self.transition_head  = nn.Linear(H, 5)
        self.arc_head         = nn.Linear(H, 5)
        self.instrument_head  = nn.Sequential(
            nn.Linear(H,64), nn.ReLU(), nn.Linear(64,15), nn.Sigmoid())

    def forward(self, sv, nv, ep):
        ec = self.emo_encoder(ep)
        z  = torch.cat([sv, nv, ec], dim=-1)
        r  = self.trunk(z)
        return {
            'tempo_score'    : self.tempo_head(r).squeeze(-1),
            'intensity'      : self.intensity_head(r).squeeze(-1),
            'valence'        : self.valence_head(r).squeeze(-1),
            'tonality'       : self.tonality_head(r),
            'harmonic_style' : self.harmonic_head(r),
            'motion_type'    : self.motion_head(r),
            'rhythm_style'   : self.rhythm_head(r),
            'rhythm_density' : self.density_head(r),
            'dynamic_shape'  : self.dynamic_head(r),
            'texture'        : self.texture_head(r),
            'transition'     : self.transition_head(r),
            'arc_verify'     : self.arc_head(r),
            'instruments'    : self.instrument_head(r),
        }

    @staticmethod
    def to_bpm(ts): return int(40 + float(ts) * 120)


print("  All model classes defined ✓")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 8 — Load All Checkpoints                                            │
# └──────────────────────────────────────────────────────────────────────────┘

def load_models():
    print("\n  Loading checkpoints...")

    # ── M1 ────────────────────────────────────────────────────────────────
    ck1   = torch.load(M1_CKPT, map_location=DEVICE)
    cfg1  = ck1['config']
    vocab = ck1['vocab']                  # dict  word → int
    m1 = DialogueTransformerEncoder(
        vocab_size   = cfg1['vocab_size'],
        embed_dim    = cfg1['embed_dim'],
        d_model      = cfg1['d_model'],
        num_heads    = cfg1['num_heads'],
        num_layers   = cfg1['num_layers'],
        d_ff         = cfg1['d_ff'],
        speaker_dim  = cfg1['speaker_dim'],
        turn_dim     = cfg1['turn_dim'],
        max_turns    = cfg1['max_turns'],
        num_speakers = cfg1['num_speakers'],
        num_emotions = cfg1['num_emotions'],
        num_prev     = cfg1['num_prev'],
        dropout      = cfg1['dropout'],
    ).to(DEVICE)
    m1.load_state_dict(ck1['utt_encoder'])   # key is 'utt_encoder'
    m1.eval()
    print(f"    M1 ✓  vocab={len(vocab):,}  f1={ck1['macro_f1']*100:.1f}%")

    # ── M2 ────────────────────────────────────────────────────────────────
    ck2 = torch.load(M2_CKPT, map_location=DEVICE)
    m2  = SceneEncoderV3(
        input_dim       = ck2['config']['input_dim'],
        hidden_dim      = ck2['config']['hidden_dim'],
        scene_vec_dim   = ck2['config']['scene_vec_dim'],
        num_emotions    = ck2['config']['num_emotions'],
        num_scene_types = ck2['config']['num_scene_types'],
        num_sentiments  = ck2['config']['num_sentiments'],
        dropout         = ck2['config']['dropout'],
    ).to(DEVICE)
    m2.load_state_dict(ck2['model_state'])   # key is 'model_state'
    m2.eval()
    print(f"    M2 ✓  f1={ck2['macro_f1']*100:.1f}%")

    # ── M3 ────────────────────────────────────────────────────────────────
    ck3     = torch.load(M3_CKPT, map_location=DEVICE)
    char2id = ck3['char2id']              # dict speaker_name → int
    cfg3    = ck3['config']
    m3 = NarrativeLSTM(
        scene_dim    = cfg3['scene_dim'],
        hidden_dim   = cfg3['hidden_dim'],
        narr_dim     = cfg3['narr_dim'],
        char_dim     = cfg3['char_dim'],
        max_chars    = cfg3['max_chars'],
        num_arc      = cfg3['num_arc'],
        num_emotions = cfg3['num_emotions'],
        dropout      = cfg3['dropout'],
    ).to(DEVICE)
    m3.load_state_dict(ck3['model_state'])
    m3.eval()
    print(f"    M3 ✓  chars={len(char2id)}  f1={ck3['macro_f1']*100:.1f}%")

    # ── M4 ────────────────────────────────────────────────────────────────
    ck4 = torch.load(M4_CKPT, map_location=DEVICE)
    m4  = MusicPlannerV2().to(DEVICE)
    m4.load_state_dict(ck4['model_state'])
    m4.eval()
    print(f"    M4 ✓  harm_f1={ck4['macro_f1']*100:.1f}%")

    return m1, vocab, m2, m3, char2id, m4

m1, vocab, m2, m3, char2id, m4 = load_models()


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 9 — Inference Helpers                                               │
# └──────────────────────────────────────────────────────────────────────────┘

def tokenize(text: str, max_len: int = 50) -> list:
    """Whitespace tokenize, lowercase, keep alphanumeric+apostrophe."""
    tokens = re.sub(r"[^a-zA-Z0-9' ]", ' ', text.lower()).split()
    UNK = vocab.get('<UNK>', 1)
    ids = [vocab.get(t, UNK) for t in tokens[:max_len]]
    ids += [0] * (max_len - len(ids))    # pad to max_len
    return ids


def infer_scene_type(labels: list, speakers: list) -> int:
    """Heuristic — copied verbatim from module2_v3.py infer_scene_type()."""
    n      = len(labels)
    counts = Counter(IDX2LABEL[l] for l in labels)
    if counts.get('scared',   0) / n >= 0.60: return SCENE_TYPE_NAMES.index('suspense')
    if counts.get('mad',      0) / n >= 0.50: return SCENE_TYPE_NAMES.index('action')
    if counts.get('powerful', 0) / n >= 0.50: return SCENE_TYPE_NAMES.index('dramatic_reveal')
    if counts.get('neutral',  0) / n >= 0.70: return SCENE_TYPE_NAMES.index('transition')
    if len(set(speakers)) == 1 and n >= 4:    return SCENE_TYPE_NAMES.index('investigation')
    return SCENE_TYPE_NAMES.index('dialogue')


CHARS_PER_SCENE = 5   # K — matches EpisodeDataset in module3_v1.py

def encode_char_ids(speakers: list, char2id: dict) -> list:
    """Encode up to CHARS_PER_SCENE unique speakers per scene. -1 = padding."""
    unique = list(dict.fromkeys(str(s) for s in speakers))[:CHARS_PER_SCENE]
    ids    = [char2id.get(s, 0) for s in unique]    # 0 = UNKNOWN
    ids   += [-1] * (CHARS_PER_SCENE - len(ids))    # -1 = pad
    return ids


def format_prompt(out: dict, scene_id, arc_phase: int,
                  tension: float, emotion: str, scene_type: str) -> str:
    """Assembles music descriptor string (matches module4_v2.py assemble_prompt)."""
    def _s(k): return float(out[k].item() if isinstance(out[k], torch.Tensor) else out[k])
    def _c(k, names):
        v = out[k]
        idx = int(v.argmax().item() if isinstance(v, torch.Tensor) else np.argmax(v))
        return names[min(idx, len(names)-1)]

    bpm        = MusicPlannerV2.to_bpm(_s('tempo_score'))
    intensity  = _s('intensity')
    tonality   = _c('tonality',      TONALITY_NAMES)
    harmonic   = _c('harmonic_style',HARMONIC_NAMES)
    motion     = _c('motion_type',   MOTION_NAMES)
    rhythm     = _c('rhythm_style',  RHYTHM_NAMES)
    density    = _c('rhythm_density',DENSITY_NAMES)
    dynamic    = _c('dynamic_shape', DYNAMIC_NAMES)
    texture    = _c('texture',       TEXTURE_NAMES)
    transition = _c('transition',    TRANSITION_NAMES)
    arc_name   = ARC_PHASES[max(0, min(arc_phase, 4))]

    iv    = out['instruments']
    flags = (iv > 0.5).cpu().numpy() if isinstance(iv, torch.Tensor) else (np.array(iv) > 0.5)
    instr = [INSTRUMENT_NAMES[i] for i, f in enumerate(flags) if f] or ['solo_piano']

    dmark = ('pp' if intensity<0.25 else 'mp' if intensity<0.45
             else 'mf' if intensity<0.65 else 'f' if intensity<0.85 else 'ff')
    sid   = f"S{scene_id[0]}E{scene_id[1]}D{scene_id[2]}" if scene_id else "scene"

    return (f"Scene {sid} — {bpm} BPM, {tonality} key, {harmonic} harmony. "
            f"Motion: {motion}. Dynamic shape: {dynamic}. "
            f"Instruments: {', '.join(instr)}.\n"
            f"  arc:{arc_name}  tension:{tension:.2f}  dynamics:{dmark}  "
            f"rhythm:{rhythm}({density})  texture:{texture}  "
            f"transition→next:{transition}  [emotion:{emotion}, type:{scene_type}]")


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 10 — Full Pipeline  run_screenplay()                                │
# │                                                                          │
# │  Input:                                                                  │
# │    screenplay = [                                                        │
# │        {                                                                 │
# │            'scene_id'  : (season, episode, dialogue_id),                │
# │            'utterances': [                                               │
# │                {'speaker': 'ROSS',   'text': "We were on a break!"},   │
# │                {'speaker': 'RACHEL', 'text': "Oh, I don't think so."}, │
# │            ]                                                             │
# │        },                                                                │
# │        ...   # more scenes in the same episode                          │
# │    ]                                                                     │
# │                                                                          │
# │  Important: pass ALL scenes of the episode together so M3's causal     │
# │  LSTM has proper narrative context for each scene.                      │
# └──────────────────────────────────────────────────────────────────────────┘

@torch.no_grad()
def run_screenplay(screenplay: list, verbose: bool = True) -> list:
    """
    Runs the full M1→M2→M3→M4 pipeline on an episode (list of scenes).
    Returns list of result dicts, one per scene, with 'prompt' key.
    """
    NUM_PREV = 5    # CFG1.NUM_PREV_UTT
    MAX_LEN  = 50   # CFG1.MAX_LEN

    # ── PASS 1: M1+M2 — encode every scene independently ─────────────────
    scene_vectors  = []   # list of (256,) tensors, one per scene
    scene_m2_outs  = []   # list of M2 output dicts
    scene_labels   = []   # list of dominant emotion int per scene
    scene_speakers = []   # list of unique speaker lists per scene
    scene_utt_labels= []  # list of per-utt label lists

    for sc_idx, scene in enumerate(screenplay):
        sid  = scene['scene_id']
        utts = scene['utterances']
        U    = len(utts)

        if verbose:
            print(f"\n  ── Scene {sid}  ({U} utterances) ──────────────────────")

        # Build M1 inputs for all utterances in the scene
        # utt_history: (U, NUM_PREV, MAX_LEN) — rolling token history
        all_tok  = []
        all_spk  = []
        all_turn = []
        all_hist = []

        # Sliding window of previous token sequences
        history_window = [([0] * MAX_LEN)] * NUM_PREV   # list of MAX_LEN lists

        for t_idx, utt in enumerate(utts):
            tok_ids = tokenize(utt['text'], MAX_LEN)
            spk_id  = 0   # M1 was trained with MAX_SPEAKERS=2; 0 is valid for all
            turn_pos= min(t_idx, 59)   # max_turns=60

            all_tok.append(tok_ids)
            all_spk.append(spk_id)
            all_turn.append(turn_pos)
            all_hist.append([list(h) for h in history_window])

            # Slide window: drop oldest, append current
            history_window = history_window[1:] + [tok_ids]

        # Stack into tensors: batch = all utterances of this scene
        tok_t  = torch.tensor(all_tok,  dtype=torch.long,  device=DEVICE)  # (U, MAX_LEN)
        spk_t  = torch.tensor(all_spk,  dtype=torch.long,  device=DEVICE)  # (U,)
        turn_t = torch.tensor(all_turn, dtype=torch.long,  device=DEVICE)  # (U,)
        hist_t = torch.tensor(all_hist, dtype=torch.long,  device=DEVICE)  # (U, NUM_PREV, MAX_LEN)

        # M1: encode all utterances in one batched forward pass
        logits, utt_vecs, _ = m1(tok_t, spk_t, turn_t, hist_t,
                                  return_vector=True)
        # logits: (U, 7), utt_vecs: (U, 256)
        utt_labels_sc = logits.argmax(-1).cpu().tolist()   # per-utterance emotion ints
        utt_conf      = torch.softmax(logits, -1).max(-1).values

        if verbose:
            for i, utt in enumerate(utts):
                print(f"    [{i}] {utt['speaker']:10s} | "
                      f"{utt['text'][:52]:<52} → "
                      f"{EMOTION_NAMES[utt_labels_sc[i]]:8s} "
                      f"({utt_conf[i]*100:.0f}%)")

        # M2: scene vector
        # emotion_matrix: (1, U, 256), mask: (1, U) all True
        emo_mat    = utt_vecs.unsqueeze(0)                             # (1, U, 256)
        mask_sc    = torch.ones(1, U, dtype=torch.bool, device=DEVICE)

        spk_names  = [u['speaker'] for u in utts]
        st_int     = infer_scene_type(utt_labels_sc, spk_names)
        st_t_scene = torch.tensor([st_int], dtype=torch.long, device=DEVICE)

        m2_out  = m2(emo_mat, mask_sc, st_t_scene)
        sv      = m2_out['scene_vector'][0]                            # (256,)
        dom_emo = int(m2_out['emotion_logits'][0].argmax().item())

        if verbose:
            ar = float(m2_out['arousal_pred'][0].item())
            cf = float(m2_out['conflict_pred'][0].item())
            print(f"    → M2: emotion={EMOTION_NAMES[dom_emo]:8s}  "
                  f"arousal={ar:+.2f}  conflict={cf:+.2f}  "
                  f"scene_type={SCENE_TYPE_NAMES[st_int]}")

        scene_vectors.append(sv)
        scene_m2_outs.append(m2_out)
        scene_labels.append(dom_emo)
        scene_speakers.append(spk_names)
        scene_utt_labels.append(utt_labels_sc)

    # ── PASS 2: M3 — process whole episode at once (causal) ───────────────
    S = len(scene_vectors)

    # scene_vectors: (1, S, 256)
    sv_seq = torch.stack(scene_vectors).unsqueeze(0).to(DEVICE)   # (1, S, 256)

    # char_ids: (1, S, K)  — speaker IDs per scene
    char_ids_list = []
    for spk_names in scene_speakers:
        char_ids_list.append(encode_char_ids(spk_names, char2id))
    char_t = torch.tensor(char_ids_list, dtype=torch.long,
                           device=DEVICE).unsqueeze(0)              # (1, S, K)

    m3_out = m3(sv_seq, char_t)
    # narrative_vectors: (1, S, 256)
    # tension_scores:    (1, S, 1)
    # arc_logits:        (1, S, 5)

    narr_vecs  = m3_out['narrative_vectors'][0]     # (S, 256)
    tensions   = m3_out['tension_scores'][0, :, 0]  # (S,)
    arc_preds  = m3_out['arc_logits'][0].argmax(-1) # (S,)

    if verbose:
        print(f"\n  → M3 narrative summary:")
        for t in range(S):
            print(f"    scene {t+1}: arc={ARC_PHASES[arc_preds[t].item()]:10s}  "
                  f"tension={tensions[t].item():.2f}")

    # ── PASS 3: M4 — music descriptors per scene ──────────────────────────
    results = []
    for sc_idx in range(S):
        sid    = screenplay[sc_idx]['scene_id']
        m2o    = scene_m2_outs[sc_idx]
        dom    = scene_labels[sc_idx]
        st_int = infer_scene_type(scene_utt_labels[sc_idx],
                                   scene_speakers[sc_idx])

        sv_t  = scene_vectors[sc_idx].unsqueeze(0)           # (1, 256)
        nv_t  = narr_vecs[sc_idx].unsqueeze(0)               # (1, 256)
        ep_t  = torch.softmax(
            m2o['emotion_logits'], dim=-1)                    # (1, 7)

        m4_out = m4(sv_t, nv_t, ep_t)

        arc_ph  = int(arc_preds[sc_idx].item())
        tension = float(tensions[sc_idx].item())
        prompt  = format_prompt(
            {k: v[0] for k, v in m4_out.items()},
            scene_id   = sid,
            arc_phase  = arc_ph,
            tension    = tension,
            emotion    = EMOTION_NAMES[dom],
            scene_type = SCENE_TYPE_NAMES[st_int])

        if verbose:
            print(f"\n  ╔══ MUSIC PROMPT  Scene {sid} {'═'*30}╗")
            for line in prompt.split('\n'):
                print(f"  ║  {line}")
            print(f"  ╚{'═'*55}╝")

        bpm = MusicPlannerV2.to_bpm(float(m4_out['tempo_score'][0].item()))
        iv  = m4_out['instruments'][0].cpu().numpy()

        results.append({
            'scene_id'        : sid,
            'speakers'        : list(set(scene_speakers[sc_idx])),
            'utt_emotions'    : [EMOTION_NAMES[l] for l in scene_utt_labels[sc_idx]],
            'dominant_emotion': EMOTION_NAMES[dom],
            'scene_type'      : SCENE_TYPE_NAMES[st_int],
            'arc_phase'       : ARC_PHASES[arc_ph],
            'tension'         : round(tension, 3),
            'tempo_bpm'       : bpm,
            'harmonic_style'  : HARMONIC_NAMES[m4_out['harmonic_style'][0].argmax().item()],
            'dynamic_shape'   : DYNAMIC_NAMES[m4_out['dynamic_shape'][0].argmax().item()],
            'texture'         : TEXTURE_NAMES[m4_out['texture'][0].argmax().item()],
            'instruments_on'  : [INSTRUMENT_NAMES[i]
                                  for i, v in enumerate(iv) if v > 0.5],
            'prompt'          : prompt,
        })

    return results


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 11 — MODE A: Instant Lookup (no model required)                     │
# │                                                                          │
# │  Looks up any scene from music_prompts.json by (season, episode, dlg). │
# └──────────────────────────────────────────────────────────────────────────┘

def lookup(scene_id: tuple):
    """Print the pre-generated prompt for a known MELD scene."""
    with open(PROMPTS_JSON) as f:
        all_prompts = json.load(f)
    target = list(scene_id)
    for p in all_prompts:
        if list(p['scene_id']) == target:
            print(f"\n  {'═'*58}")
            print(f"  LOOKUP  Scene {p['scene_id']}")
            print(f"  {'═'*58}")
            print(f"  {p['prompt']}")
            print(f"  instruments : {', '.join(p['instruments_on'])}")
            print(f"  {'═'*58}")
            return
    print(f"  ✗ Scene {scene_id} not found in music_prompts.json")

# Sample lookups — the four scenes printed in the M4 output log
for sid in [(1,1,19), (1,1,152), (1,1,164), (1,1,467)]:
    lookup(sid)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 12 — MODE B: Full Inference on New Screenplay                       │
# │                                                                          │
# │  "The Break" — 3 scenes from a Friends-style episode.                  │
# │  All three scenes passed together so M3 has full causal context.       │
# └──────────────────────────────────────────────────────────────────────────┘

SCREENPLAY = [
    {
        'scene_id'   : (99, 1, 1),
        'utterances' : [
            {'speaker': 'CHANDLER', 'text': "Could this apartment BE any more normal right now?"},
            {'speaker': 'JOEY',     'text': "Yeah I don't know what you're talking about."},
            {'speaker': 'CHANDLER', 'text': "Everything is fine. Ross and Rachel are totally fine."},
            {'speaker': 'JOEY',     'text': "Dude. They are not fine."},
            {'speaker': 'CHANDLER', 'text': "I know. I just really want someone to be fine."},
        ]
    },
    {
        'scene_id'   : (99, 1, 2),
        'utterances' : [
            {'speaker': 'ROSS',    'text': "We were on a break! That means something, Rachel!"},
            {'speaker': 'RACHEL',  'text': "A break from us. Not a break from decency!"},
            {'speaker': 'ROSS',    'text': "I was so scared I was losing you. You have no idea."},
            {'speaker': 'RACHEL',  'text': "So that makes it okay? That makes everything okay?"},
            {'speaker': 'ROSS',    'text': "No. Nothing about this is okay. I know that."},
            {'speaker': 'RACHEL',  'text': "I trusted you. I trusted you completely."},
        ]
    },
    {
        'scene_id'   : (99, 1, 3),
        'utterances' : [
            {'speaker': 'RACHEL',  'text': "I just need to go."},
            {'speaker': 'ROSS',    'text': "Rachel please. Don't do this."},
            {'speaker': 'RACHEL',  'text': "I am sorry Ross. I am really sorry."},
            {'speaker': 'ROSS',    'text': "Okay."},
        ]
    },
]

print("\n\n" + "═"*62)
print("  MODE B — Full Inference on New Screenplay")
print("  'The Break' — 3-scene Friends episode")
print("═"*62)

results = run_screenplay(SCREENPLAY, verbose=True)


# ┌──────────────────────────────────────────────────────────────────────────┐
# │ CELL 13 — Summary Table + Save                                           │
# └──────────────────────────────────────────────────────────────────────────┘

print("\n\n  ══ OUTPUT SUMMARY ══════════════════════════════════════════")
print(f"  {'Scene':<14} {'Emotion':<10} {'Arc':<11} {'BPM':>4}  "
      f"{'Harmonic':<22} {'Dynamic':<24} {'Instruments'}")
print(f"  {'─'*100}")
for r in results:
    print(f"  {str(r['scene_id']):<14} "
          f"{r['dominant_emotion']:<10} "
          f"{r['arc_phase']:<11} "
          f"{r['tempo_bpm']:>4}  "
          f"{r['harmonic_style']:<22} "
          f"{r['dynamic_shape']:<24} "
          f"{', '.join(r['instruments_on'][:3])}")

with open("/kaggle/working/inference_results.json", 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"\n  Saved → /kaggle/working/inference_results.json")

  Device : cuda
  Constants loaded ✓
  All model classes defined ✓

  Loading checkpoints...
    M1 ✓  vocab=20,000  f1=23.4%
    M2 ✓  f1=55.6%
    M3 ✓  chars=20  f1=48.3%
    M4 ✓  harm_f1=66.7%

  ══════════════════════════════════════════════════════════
  LOOKUP  Scene [1, 1, 19]
  ══════════════════════════════════════════════════════════
  Scene S1E1D19 — 67 BPM, major key, diatonic_warm harmony. Motion: static_slow. Dynamic shape: slow_build. Instruments: solo_piano, woodwinds.
  arc:setup  tension:0.19  dynamics:pp  rhythm:driving_8th(minimal)  texture:ambient_minimal  transition→next:crossfade  [emotion:neutral, type:dialogue]
  instruments : solo_piano, woodwinds
  ══════════════════════════════════════════════════════════

  ══════════════════════════════════════════════════════════
  LOOKUP  Scene [1, 1, 152]
  ══════════════════════════════════════════════════════════
  Scene S1E1D152 — 53 BPM, minor key, melodic_minor harmony. Motion: slow_release. Dynamic shape: slow_b